# Pipeline experimental completo (Optuna y V1–V5)

Este notebook aplica un **pipeline común** para evaluar cuatro arquitecturas (MLP, LSTM, TCN y Transformer) sobre un mismo problema de clasificación multiclase: **predecir la dirección del penalti (3 clases)** a partir de embeddings de marcha extraídos antes del golpeo del balón. El conjunto de datos contiene **432 vídeos** y está **desbalanceado** (derecha: 209, centro: 66, izquierda: 157), por lo que el criterio principal de selección de modelos es el **F1-score macro** (además de accuracy, pérdidas y F1 por clase).

### Datos de entrada: embeddings por `video_ID` y `step`
Los experimentos parten de varios ficheros de embeddings (CSV) generados sobre los mismos vídeos. Cada fila representa un `video_ID` y un `step`, e incluye:
- el vector numérico `feat_*` (representación del jugador),
- metadatos de contexto (p. ej., *pitch side* y *kicker foot*),
- y la etiqueta objetivo `shoot_zone`.

Un detalle importante es que los “steps” **no se interpretan como tiempo**, sino como una **partición espacial del cuerpo (bandas horizontales)** derivada de módulos tipo HPP/HPM. Por eso, el eje `step` es comparable dentro de un mismo embedding, pero puede variar en número total de steps entre extractores. Esto condiciona especialmente la combinación de representaciones y el uso de modelos secuenciales.

Además, la extracción de embeddings se limita a fotogramas **anteriores al golpeo**, reduciendo el riesgo de fuga de información hacia la etiqueta.

### Preparación general (común a todas las fases)
Antes de entrenar, se valida el formato (columnas, `video_ID`, steps disponibles y dimensionalidad). Es relevante porque no todos los embeddings tienen exactamente la misma dimensionalidad (por ejemplo, puede variar el número de `feat_*`), lo que obliga a tratar con cuidado la reutilización directa de configuraciones o la combinación entre representaciones.

En el entrenamiento, el particionado se hace **por `video_ID`** (no por filas) para evitar que steps del mismo vídeo caigan simultáneamente en train y test, lo que inflaría artificialmente el rendimiento. Este split por vídeo es la base del protocolo común de evaluación.

---

### Fase Optuna (búsqueda inicial de hiperparámetros)
La fase Optuna realiza una **búsqueda sistemática de hiperparámetros** para cada arquitectura y cada tipo de embedding, con el objetivo de establecer una base de configuraciones razonables y comparables. Para acelerar la exploración se usa **poda temprana** (p. ej., Hyperband): los trials reportan métricas intermedias y se detienen automáticamente si son poco prometedores, concentrando cómputo en configuraciones con mejor evolución.

El resultado de Optuna se almacena como:
- **CSVs de trials** por embedding (hiperparámetros + métricas),
- y un **CSV resumen** con los mejores parámetros por embedding, que servirá como punto de partida en V1.

---

### V1: resultados base con las mejores configuraciones (por embedding)
En V1 se toman las **mejores configuraciones** obtenidas en Optuna y se reentrena cada arquitectura para generar un conjunto de **resultados base**. Esta fase sirve para:
- ver qué embeddings y qué familias de modelos rinden mejor bajo un protocolo homogéneo,
- y tener una referencia clara para comparar V2–V5.

---

### V2: exploración guiada (presets a partir de lo observado)
Tras V1, la fase V2 amplía la exploración con **configuraciones candidatas (presets)** basadas en lo aprendido: variaciones de arquitectura (tamaños, número de capas/bloques), estrategias de agregación/pooling y parámetros de optimización. El objetivo es identificar no solo máximos puntuales, sino **rangos de configuración estables** que generalicen mejor.

---

### V3: reentrenamiento robusto (entrenar mejor, no solo explorar más)
V3 se centra en la **estabilidad del entrenamiento** y en mitigar el efecto del desbalance. Se seleccionan configuraciones destacadas (de V1/V2) y se reentrena incorporando medidas como:
- **class weights** (compensación de clase minoritaria),
- **label smoothing**,
- **gradient clipping**,
además de mantener un criterio de checkpointing consistente (mejor F1 macro).

---

### V4: fusión de embeddings (mean / concat y versiones “reduced”)
En V4 se evalúa si combinar representaciones aporta información complementaria. Se prueban fusiones por:
- **media** (mean),
- **concatenación** (concat),
manteniendo el protocolo de entrenamiento para poder atribuir cambios de rendimiento a la representación.

Cuando las representaciones no son compatibles directamente (por ejemplo, distinto número de steps), se aplica **alineamiento/reducción proporcional** para homogeneizar la estructura antes de fusionar, asegurando que los steps combinados representen regiones comparables.

---

### V5: selección de steps (Top-k%) para reducir ruido
En V5 se estudia explícitamente qué steps aportan más información. A partir de un análisis de relevancia por step, se generan embeddings recortados conservando solo el **Top-30%, Top-50% y Top-70%** de steps mejor valorados y se reentrenan modelos usando configuraciones destacadas. El objetivo es comprobar si eliminar steps menos informativos **mantiene**, **mejora** o **degrada** el rendimiento.

---

### Cómo se guardan resultados y modelos (trazabilidad)
El guardado se implementa de forma unificada:
- **Modelos**: cuando una ejecución alcanza su mejor estado (según el criterio del experimento), se guarda un checkpoint `.pth` con `state_dict` y la `config` completa (y, si aplica, artefactos de preprocesado como PCA/escalado).
- **Resultados**: cada entrenamiento añade una fila a un **CSV de resultados** (por arquitectura y fase) con un núcleo común: `id`, `target_embedding`, métricas finales (train/test loss, accuracy, f1_macro, f1_per_class), métricas del mejor estado (best_f1, best_epoch, etc.) y metadatos (epochs_trained, duración, seed). Esto permite reconstruir tablas y comparar arquitecturas sin ambigüedades.


## Interpretación de Métricas de Evaluación para Modelos de Clasificación

### Matriz de Confusión

La matriz de confusión es una tabla que resume el rendimiento de un modelo de clasificación. Es una herramienta fundamental para entender los aciertos y errores del modelo.

* **Verdaderos Positivos (VP):** El modelo predijo la clase positiva y la etiqueta real era positiva. ¡Acierto!
* **Verdaderos Negativos (VN):** El modelo predijo la clase negativa y la etiqueta real era negativa. ¡Acierto!
* **Falsos Positivos (FP):** El modelo predijo la clase positiva, pero la etiqueta real era negativa. Es un **error de Tipo I** o "falsa alarma".
* **Falsos Negativos (FN):** El modelo predijo la clase negativa, pero la etiqueta real era positiva. Es un **error de Tipo II** o "falso negativo".

Una matriz ideal tiene valores altos en la **diagonal principal** (VP y VN) y valores cercanos a cero fuera de ella. Los valores fuera de la diagonal nos dicen qué errores comete el modelo. Por ejemplo, en un modelo de diagnóstico médico, un FN (no detectar una enfermedad) suele ser mucho más grave que un FP.

### Curva ROC (Receiver Operating Characteristic) y AUC
AUC-ROC: Mide la capacidad del modelo para distinguir entre clases, sin importar el umbral de decisión. Es una métrica robusta, pero puede ser engañosa en conjuntos de datos desequilibrados.

La curva ROC es una gráfica que visualiza el rendimiento de un clasificador en todos los umbrales de decisión posibles. Compara la tasa de aciertos con la tasa de falsas alarmas.

* **Eje Y: Tasa de Verdaderos Positivos (TPR)** o **Sensibilidad**. Proporción de positivos reales que se identificaron correctamente.
* **Eje X: Tasa de Falsos Positivos (FPR)**. Proporción de negativos reales que se identificaron incorrectamente como positivos.


* Un modelo perfecto tendría una curva que pasa por la esquina superior izquierda, logrando un 100% de TPR con 0% de FPR.
* La línea diagonal punteada representa un clasificador aleatorio. Si tu curva está por debajo de esta línea, significa que tu modelo es peor que adivinar al azar.


El **AUC (Área Bajo la Curva ROC)** es un valor numérico que resume el rendimiento de la curva ROC. Es la probabilidad de que el modelo clasifique a un positivo aleatorio por encima de un negativo aleatorio.

* **AUC = 1.0:** Modelo perfecto.
* **0.5 < AUC < 1.0:** El modelo rinde mejor que el azar. Cuanto más cerca de 1.0, mejor.
* **AUC = 0.5:** El modelo es tan bueno como la clasificación aleatoria.



### Curva PR (Precision-Recall) y AP
AUC-PR (AP): Mide el rendimiento del modelo en la identificación de la clase minoritaria. Es una métrica más adecuada para conjuntos de datos desequilibrados, ya que se centra en la precisión de las predicciones positivas.

La curva PR es especialmente útil para evaluar modelos en **conjuntos de datos desequilibrados**, donde la clase de interés (la minoritaria) es rara.

* **Eje Y: Precisión**. Proporción de predicciones positivas que fueron correctas. $\text{Precisión} = \frac{\text{VP}}{\text{VP} + \text{FP}}$
* **Eje X: Recall (Sensibilidad)**. Proporción de positivos reales que se identificaron correctamente. $\text{Recall} = \frac{\text{VP}}{\text{VP} + \text{FN}}$

* Un modelo ideal tiene una curva que se acerca a la esquina superior derecha, logrando una alta precisión y un alto recall simultáneamente.
* La curva PR es más informativa que la ROC en casos de desequilibrio de clases, ya que se centra en la capacidad del modelo para identificar correctamente la clase minoritaria sin ser engañada por la gran cantidad de negativos.

El **AP (Average Precision)** es el área bajo la curva PR y resume su rendimiento en un solo número. Un AP más alto indica que el modelo es mejor en la tarea de identificar la clase minoritaria.

* **Un AP alto** significa que el modelo es capaz de encontrar un gran número de positivos (alto recall) sin generar una gran cantidad de falsas alarmas (alta precisión).


---
---
# Modelo MLP

## Preparación de datos, split y bucle de entrenamiento (MLP)

La función `prepare_mlp_data` transforma cada secuencia asociada a un `video_ID` en un **único vector fijo** aplicando *pooling* temporal (`mean` o `max`) sobre las columnas `feat_*`. A partir de esos vectores por vídeo, realiza un **split estratificado por vídeo** (train/test), evitando fugas de información entre fotogramas del mismo vídeo. La normalización (`minmax` o `L2`) se ajusta **solo con el train** y luego se aplica a train y test; opcionalmente, se aplica **PCA** (fit en train, transform en ambos) para reducir dimensionalidad manteniendo una fracción de varianza especificada. Finalmente devuelve los `TensorDataset` y los IDs de vídeos usados en train/test (y, si se solicita, también el `scaler` y el `pca`).

La función `run_training_mlp` implementa el bucle de entrenamiento/evaluación para un modelo MLP ya instanciado. Permite elegir optimizador (`Adam`, `AdamW`, `SGD` o `RMSprop`), usar **pesos de clase** (calculados desde el `DataLoader` o proporcionados manualmente), y aplicar **label smoothing** y **clipping de gradiente**. En cada época calcula pérdidas y métricas en train/test (accuracy, F1 macro y F1 por clase), y guarda un *checkpoint* del **mejor estado según F1** (condicionado por un umbral de pérdida). Además, incorpora un **early stopping híbrido** que detiene el entrenamiento si la pérdida es muy baja de forma sostenida o si tanto la pérdida como el F1 se estancan durante un número de épocas.


## Optimización de hiperparámetros con Optuna (MLP)

En esta sección se emplea **Optuna** para seleccionar hiperparámetros de la arquitectura **MLP** de forma **independiente para cada embedding** disponible en `Gait_Embeddings_good`. Para cada embedding se realiza primero una partición estratificada por `video_ID` en **train (90%)** y **test (10%)**, de modo que la optimización solo utiliza el conjunto de *train* y el *test* queda reservado para evaluación posterior.

La función objetivo aplica **validación cruzada estratificada (5-fold)** sobre el conjunto de entrenamiento. En cada *trial*, Optuna muestrea una combinación de hiperparámetros que incluye: estrategia de *pooling* temporal (p. ej., `mean` o `max`), normalización de entrada (`minmax` o `L2`), uso opcional de PCA y fracción de reducción, estructura de la MLP (número de capas y tamaño base con distintos patrones), función de activación, configuración de **LayerNorm** y **Dropout** por capa, y parámetros de optimización (optimizador, `learning rate`, `weight_decay` y momentum cuando aplica). El criterio a maximizar es el **F1-score macro promedio** entre folds. Además, se emplean **pruning** y **early stopping** para reducir el coste de cómputo descartando trials poco prometedores.

Como salida, para cada embedding se guardan los *trials* en un **CSV ya “limpio”** (con listas reconstruidas como `hidden_layers`, `norm_layers`, `dropout_layers`, etc.) y se genera un resumen global `best_params_MLP.csv` con la mejor configuración por embedding.

In [ ]:
from src.mlp.optuna import optimize_embeddings

best_df = optimize_embeddings()
best_df.head()

## V1: Entrenamiento con los mejores configuraciones de Optuna (MLP)

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id  
from src.mlp.prepare_data import prepare_mlp_data
from src.mlp.model import FlexibleMLP
from src.mlp.train import run_training_mlp
from src.utils.save_methods import save_train_test_ids, save_model



DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "Gait_Embeddings_good"

RESULTS_PATH = "results/MLP/mlp_results_v1_Optuna.csv"

EPOCHS = 5000
TEST_SIZE = 0.1
SEED = 42
F1_THRESHOLD = 0.6
SAVE_IDS = True


# Cargar BEST PARAMS obtenidos de Optuna
best_params_df = pd.read_csv("results/MLP/Optuna/best_params_MLP.csv")
os.makedirs("saved_models/MLP/v1", exist_ok=True)
embedding_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]


# Obtener ID inicial para esta ejecución
csvs_for_ids = RESULTS_PATH

id = get_max_id(csvs_for_ids) 
id += 1                        

is_first_config = not os.path.exists(RESULTS_PATH)
print(f"[IDs] Starting id = {id}")
print(f"[CSV] First write? {is_first_config}")

# Loop: entrenar cada configuración óptima de Optuna
for _, param_row in best_params_df.iterrows():

    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    results = []

    source_embedding = param_row["embedding"]
    print(f"\n==============================")
    print(f" Entrenando configuración óptima → {source_embedding}")
    print(f"==============================")

    # Construir configuración del modelo
    config = {
        "hidden_layers": eval(param_row["hidden_layers"]),
        "pca_frac": param_row["pca_frac"] if not pd.isna(param_row["pca_frac"]) else None,
        "norm_layers": eval(param_row["norm_layers"]),
        "dropout_layers": eval(param_row["dropout_layers"]),
        "dropout_rate": float(param_row["dropout"]),
        "pooling": param_row["pooling"],
        "norm": param_row["norm"],
        "batch_size": int(param_row["batch_size"]),
        "num_layers": int(param_row["num_layers"]),
        "base_dim": int(param_row["base_dim"]),
        "activation": param_row["activation"],
        "optimizer": param_row["optimizer"],
        "momentum_sgd": (
            float(param_row["momentum_sgd"])
            if "momentum_sgd" in param_row.index and not pd.isna(param_row["momentum_sgd"])
            else None
        ),
        "learning_rate": float(param_row["lr"]),
        "weight_decay": float(param_row["weight_decay"]),
        "use_pca": bool(param_row["use_pca"]),
    }


    # Entrenar MLP para cada embedding
    for fname in embedding_files:
        print(f"\n--- Entrenando MLP con {fname} ---")

        df = pd.read_csv(os.path.join(DATA_DIR, fname))

        # Preparar datos (se respeta PCA si venía en el CSV)
        tr_ds, te_ds, train_ids, test_ids = prepare_mlp_data(
            df,
            pooling=config["pooling"],
            norm=config["norm"],
            test_size=TEST_SIZE,
            use_pca=config["use_pca"],
            pca_frac=config["pca_frac"],
            seed=SEED,
        )

        # Guardar splits
        if SAVE_IDS:
            save_train_test_ids(
                fname, train_ids, test_ids, SEED,
                output_dir=f"results/MLP/splits/seed{SEED}"
            )

        # Añadir input_dim real
        input_dim = tr_ds[0][0].shape[0]

        # DataLoaders
        tr_loader = DataLoader(tr_ds, batch_size=config["batch_size"], shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=config["batch_size"], shuffle=False)


        # Crear modelo en base a la config
        model = FlexibleMLP(
            input_dim=input_dim,
            hidden_layers=config["hidden_layers"],
            activation=config["activation"],
            dropout=config["dropout_rate"],
            normalization_layers=config["norm_layers"],
            dropout_layers=config["dropout_layers"]
        ).to(DEVICE)

        # Entrenamiento
        run_id = id
        id += 1

        history, best_state = run_training_mlp(
            model,
            tr_loader,
            te_loader,
            epochs=EPOCHS,
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"],
            optimizer_name=config["optimizer"],
            momentum_sgd=config["momentum_sgd"],
        )

        # Guardar resultados
        results.append({
            "id": run_id,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": source_embedding,
            "model": "MLP",
            "seed": SEED,
            "input_dim": input_dim,
            **config,
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "epochs_trained": history["epochs_trained"],
            "best_epoch": history["best_epoch"],
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "training": "STOPPED" if history["early_stopped"] == True else "COMPLETED",
        })

        # Si supera el threshold, guardar modelo
        if history["best_f1"] >= F1_THRESHOLD:
            print(f"Guardando modelo (ep {history['best_epoch']}), F1={history['best_f1']:.4f}")
            model.load_state_dict(best_state)
            model_path = f"saved_models/MLP/v1/mlp_{run_id}_v1_{fname.replace('.csv','')}.pth"
            save_model(model, model_path, config, run_id)


    # Guardar resultados globales
    df_res = pd.DataFrame(results)

    if is_first_config:
        df_res.to_csv(RESULTS_PATH, index=False)
        is_first_config = False
    else:
        with open(RESULTS_PATH, "a") as f:
            f.write("\n")
        df_res.to_csv(RESULTS_PATH, mode="a", header=False, index=False)

print("\n=== ENTRENAMIENTO MLP V1 COMPLETADO ===")


## V2: Entrenamiento a partir de configuraciones personalizadas (MLP)

La fase V2 sustituye “una mejor config por embedding” por un conjunto de presets globales que suelen venir de filtrar/deduplicar configuraciones buenas (p. ej., por best_f1). Para cada preset, se entrena el mismo MLP contra todos los embeddings, manteniendo el mismo protocolo de preparación (prepare_mlp_data) y el mismo bucle de entrenamiento (run_training_mlp). Los IDs se continúan desde históricos con get_max_id() para que no haya colisiones entre V1/V2, y los modelos que superan el umbral se guardan en SAVE_DIR.

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.mlp.prepare_data import prepare_mlp_data
from src.mlp.model import FlexibleMLP
from src.mlp.train import run_training_mlp
from src.utils.save_methods import save_model, save_train_test_ids


DATA_DIR    = "Gait_Embeddings_good"
PRESETS_CSV = "presets_configs/MLP/presets_mlp_v2.csv"
RESULTS_CSV = "results/MLP/mlp_results_v2_DEFINITIVO.csv"
SAVE_DIR    = "saved_models/MLP/v2"

TEST_SIZE   = 0.1
SEED        = 42
EPOCHS      = 5000
THRESHOLD   = 0.60
SAVE_IDS    = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)

# Obtener ID inicial para esta ejecución
hist_files = ["results/MLP/mlp_results_v1_Optuna.csv", RESULTS_CSV]
id_counter = get_max_id(hist_files)
print(f"[IDs] Starting id = {id_counter + 1}")

presets = pd.read_csv(PRESETS_CSV)
emb_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])

# Leer y entrenar cada preset
for preset_idx, row in presets.iterrows():

    print(f"\nPreset {preset_idx+1}/{len(presets)}")
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    all_results = []

    pooling      = row["pooling"]
    norm         = row["norm"]
    use_pca      = bool(row["use_pca"])
    pca_frac     = None if pd.isna(row.get("pca_frac", None)) else row["pca_frac"]

    num_layers   = int(row["num_layers"])
    base_dim     = int(row["base_dim"])
    hidden_layers = eval(row["hidden_layers"])

    activation   = row["activation"]
    dropout_rate = float(row["dropout_rate"])

    norm_layers    = eval(row["norm_layers"])
    dropout_layers = eval(row["dropout_layers"])

    batch_size     = int(row["batch_size"])
    optimizer_name = row["optimizer"]
    lr             = float(row["learning_rate"])
    weight_decay   = float(row["weight_decay"])
    momentum_sgd   = float(row["momentum_sgd"]) if optimizer_name.upper() == "SGD" else None

    config = {
        "pooling": pooling,
        "norm": norm,
        "use_pca": use_pca,
        "pca_frac": pca_frac,
        "num_layers": num_layers,
        "base_dim": base_dim,
        "hidden_layers": hidden_layers,
        "activation": activation,
        "dropout_rate": dropout_rate,
        "norm_layers": norm_layers,
        "dropout_layers": dropout_layers,
        "batch_size": batch_size,
        "optimizer": optimizer_name,
        "momentum_sgd": momentum_sgd,
        "learning_rate": lr,
        "weight_decay": weight_decay,
    }

    # Entrenar MLP para cada embedding
    for fname in emb_files:

        print(f"  Entrenando con {fname}")
        df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)

        tr_ds, te_ds, train_ids, test_ids = prepare_mlp_data(
            df,
            pooling=pooling,
            norm=norm,
            test_size=TEST_SIZE,
            use_pca=use_pca,
            pca_frac=pca_frac,
            seed=SEED,
        )

        if SAVE_IDS:
            save_train_test_ids(
                fname, train_ids, test_ids, SEED,
                output_dir=f"results/MLP/splits/seed{SEED}"
            )

        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

        input_dim = tr_ds[0][0].shape[0]

        # Crear modelo en base a la config
        model = FlexibleMLP(
            input_dim=input_dim,
            hidden_layers=hidden_layers,
            activation=activation,
            dropout=dropout_rate,
            normalization_layers=norm_layers,
            dropout_layers=dropout_layers
        ).to(DEVICE)

        run_id = id_counter + 1
        id_counter = run_id

        # Entrenamiento
        history, best_state = run_training_mlp(
            model,
            tr_loader,
            te_loader,
            epochs=EPOCHS,
            lr=lr,
            weight_decay=weight_decay,
            optimizer_name=optimizer_name,
            momentum_sgd=momentum_sgd,
        )

        # Guardar resultados
        all_results.append({
            "id": run_id,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": f"Preset_{preset_idx+1}",
            "model": "MLP",
            "seed": SEED,
            "input_dim": input_dim,
            **config,
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
            "epochs_trained": history["epochs_trained"],
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": history["best_epoch"],
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
        })

        if history["best_f1"] >= THRESHOLD:
            model.load_state_dict(best_state)
            model_path = os.path.join(SAVE_DIR, f"mlp{run_id}_v2_{fname.replace('.csv','')}.pth")
            save_model(model, model_path, config, run_id)
    
    if all_results:
        df_results = pd.DataFrame(all_results)
        if os.path.exists(RESULTS_CSV):
            with open(RESULTS_CSV, "a") as f:
                f.write("\n")
            df_results.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
        else:
            df_results.to_csv(RESULTS_CSV, index=False)

print("\nENTRENAMIENTOS MLP V2 COMPLETADOS")


## V3: Re-entrenamiento robusto (MLP)

En la fase **V3** se re-entrenan únicamente las configuraciones que ya demostraron buen rendimiento en fases previas (V1/V2), pero añadiendo **técnicas de estabilización y generalización**: `class_weights`, `label_smoothing` y `clip_grad_norm`. El objetivo es reducir la sensibilidad al desbalance (especialmente la clase minoritaria) y obtener entrenamientos más estables, con menor variación entre embeddings y mejores pérdidas en test.

El flujo es:
1. **Cargar históricos (V1/V2)** y construir una firma estable de cada configuración (hiperparámetros relevantes) para poder hacer *matching* con el `id` original.
2. Usar `find_best_configs(...)` para generar un CSV de **presets** filtrando por `best_f1 >= THRESHOLD` y eliminando duplicados por configuración.
3. Para cada preset, re-entrenar el modelo en **todos los embeddings**, reutilizando el `id` histórico cuando exista.
4. Guardar resultados en un CSV (append por bloques) con métricas finales y del mejor estado (`best_f1`, `best_epoch`, pérdidas, etc.).
5. Guardar checkpoints `.pth` solo si el modelo supera el umbral definido.

Estas tres técnicas aportan:
- **Class weights**: penaliza más los errores en clases minoritarias, mejorando macro-F1.
- **Label smoothing**: reduce la sobreconfianza y actúa como regularizador.
- **Gradient clipping**: evita actualizaciones explosivas y estabiliza el aprendizaje.


In [ ]:
import os
import json
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.find_best import find_best_configs
from src.mlp.prepare_data import prepare_mlp_data
from src.mlp.model import FlexibleMLP
from src.mlp.train import run_training_mlp
from src.utils.save_methods import save_model


DATA_DIR = "Gait_Embeddings_good"
RESULTS_CSV = "results/MLP/mlp_results_v3.csv"
SAVE_DIR = "saved_models/MLP/v3"

HISTORICAL = [
    "results/MLP/mlp_results_v1_Optuna.csv",
    "results/MLP/mlp_results_v2.csv",
]

TEST_SIZE = 0.1
SEED = 45
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000
THRESHOLD = 0.6

LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0
USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)

MATCH_COLS = [
    "hidden_layers", "pca_frac", "norm_layers", "dropout_layers", "dropout_rate", "pooling",
    "norm", "batch_size", "num_layers", "base_dim", "activation", "optimizer",
    "momentum_sgd", "learning_rate", "weight_decay", "use_pca"
]

# Normalización de valores para comparación
def normalize_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, str):
        s = v.strip()
        if s == "" or s.lower() in ["none", "nan"]:
            return None
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        if s.lower() in ["true", "false"]:
            return s.lower() == "true"
        return s.lower()
    return v


def normalize_config_row(row):
    return {col: normalize_value(row.get(col)) for col in MATCH_COLS}


def config_signature(row):
    clean = normalize_config_row(row)
    return json.dumps(clean, sort_keys=True)


def parse_list(v):
    if pd.isna(v):
        return []
    if isinstance(v, list):
        return v
    if isinstance(v, str):
        s = v.strip()
        if s == "" or s.lower() in ["none", "nan"]:
            return []
        try:
            out = ast.literal_eval(s)
            return out if isinstance(out, list) else []
        except Exception:
            return []
    return []

# Cargar datos históricos y crear lookup de (embedding, config) → id
hist_files = [p for p in HISTORICAL if os.path.exists(p)]
if not hist_files:
    raise FileNotFoundError("No se encontraron CSVs históricos (V1/V2) para MLP.")

df_hist = pd.concat([pd.read_csv(f, low_memory=False) for f in hist_files], ignore_index=True)
df_hist["__sig__"] = df_hist.apply(config_signature, axis=1)

LOOKUP = {
    (row["target_embedding"], row["__sig__"]): int(row["id"])
    for _, row in df_hist.iterrows()
}

print(f"[LOOKUP] {len(LOOKUP)} combinaciones (embedding, config) con id histórico.")

# Encontrar mejores configuraciones
best_configs_path = "presets_configs/MLP/best_configs_mlp_v3.csv"
if not os.path.exists(best_configs_path):
    presets_df = find_best_configs(
        csv_files=hist_files,
        output_file=best_configs_path,
        model_name="MLP",
        f1_threshold=THRESHOLD
    )
else:
    presets_df = pd.read_csv(best_configs_path)

if presets_df is None or presets_df.empty:
    raise ValueError("No hay presets válidos para V3 con el umbral indicado.")

print(f"[PRESETS] {len(presets_df)} configuraciones únicas seleccionadas para V3.")


emb_list = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

# Leer y entrenar cada preset
for preset_idx, prow in presets_df.iterrows():
    print(f"\nPreset {preset_idx + 1}/{len(presets_df)}")
    all_results = []

    p_sig = config_signature(prow)

    for fname in emb_list:
        run_id = LOOKUP.get((fname, p_sig))
        if run_id is None:
            print(f"  - Sin id previo: {fname} (se omite)")
            continue

        df_emb = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)

        config = {
            "hidden_layers": parse_list(prow.get("hidden_layers")),
            "pca_frac": normalize_value(prow.get("pca_frac")),
            "norm_layers": parse_list(prow.get("norm_layers")),
            "dropout_layers": parse_list(prow.get("dropout_layers")),
            "dropout_rate": float(prow.get("dropout_rate", 0.0)) if pd.notna(prow.get("dropout_rate", 0.0)) else 0.0,
            "pooling": str(prow.get("pooling", "mean")).strip().lower(),
            "norm": "" if pd.isna(prow.get("norm")) else str(prow.get("norm")).strip(),
            "batch_size": int(float(prow.get("batch_size", 64))),
            "num_layers": int(float(prow.get("num_layers", 1))),
            "base_dim": int(float(prow.get("base_dim", 256))) if pd.notna(prow.get("base_dim")) else 256,
            "activation": str(prow.get("activation", "relu")).strip().lower(),
            "optimizer": str(prow.get("optimizer", "AdamW")).strip(),
            "momentum_sgd": None,
            "learning_rate": float(prow.get("learning_rate", 1e-4)),
            "weight_decay": float(prow.get("weight_decay", 0.0)) if pd.notna(prow.get("weight_decay")) else 0.0,
            "use_pca": bool(normalize_value(prow.get("use_pca", False))),
        }

        if config["optimizer"].upper() == "SGD":
            m = prow.get("momentum_sgd", 0.0)
            config["momentum_sgd"] = 0.0 if pd.isna(m) else float(m)

        if not config["hidden_layers"]:
            config["hidden_layers"] = [max(64, config["base_dim"] // (2 ** i)) for i in range(config["num_layers"])]

        tr_ds, te_ds, _, _ = prepare_mlp_data(
            df_emb,
            pooling=config["pooling"],
            norm=config["norm"],
            test_size=TEST_SIZE,
            use_pca=config["use_pca"],
            pca_frac=config["pca_frac"],
            seed=SEED
        )

        tr_loader = DataLoader(tr_ds, batch_size=config["batch_size"], shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=config["batch_size"], shuffle=False)

        input_dim = tr_ds[0][0].shape[0]
        config["input_dim"] = input_dim

        # Modelo y entrenamiento
        model = FlexibleMLP(
            input_dim=input_dim,
            hidden_layers=config["hidden_layers"],
            activation=config["activation"],
            dropout=float(config["dropout_rate"]),
            normalization_layers=config["norm_layers"],
            dropout_layers=config["dropout_layers"]
        ).to(DEVICE)

        history, best_state = run_training_mlp(
            model,
            tr_loader,
            te_loader,
            epochs=EPOCHS,
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"],
            optimizer_name=config["optimizer"],
            momentum_sgd=config["momentum_sgd"],
            use_class_weights=USE_CLASS_WEIGHTS,
            class_weights=CLASS_WEIGHTS,
            label_smoothing=LABEL_SMOOTH,
            clip_grad_norm=CLIP_NORM
        )

        class_w_saved = history.get("class_weights", CLASS_WEIGHTS)

        # Guardar resultados
        all_results.append({
            "id": run_id,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": "best_configs_v3",
            "model": "MLP",
            "seed": SEED,
            **config,
            "label_smoothing": LABEL_SMOOTH,
            "clip_grad_norm": CLIP_NORM,
            "class_weights": str(class_w_saved),
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": history["best_epoch"],
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "epochs_trained": history["epochs_trained"],
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        })

        # Si supera el threshold, guardar modelo
        if history["best_f1"] >= THRESHOLD:
            model.load_state_dict(best_state)
            ckpt_path = os.path.join(SAVE_DIR, f"mlp{run_id}_v3_{fname.replace('.csv', '')}.pth")
            save_model(model, ckpt_path, config, run_id)

        print(f"  + {fname} | id={run_id} | best_f1={history['best_f1']:.4f}")

    # Guardar resultados globales
    if all_results:
        df_out = pd.DataFrame(all_results)
        if os.path.exists(RESULTS_CSV):
            with open(RESULTS_CSV, "a", newline="") as f:
                f.write("\n")
            df_out.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
        else:
            df_out.to_csv(RESULTS_CSV, index=False)

print("\nEntrenamiento MLP V3 finalizado.")


## Comparación de resultados de V1, V2, y V3 (MLP) 


*   **`mean_f1` (y `mean_diff`)**: Esta es la métrica más importante para evaluar la mejora general. `mean_f1_v3` es el F1-score *promedio* de todos los entrenamientos de la `v3` para un embedding, mientras que `mean_f1_v1_v2` es el promedio de los históricos. La columna **`mean_diff`** te dice si, en conjunto, tus nuevos entrenamientos son mejores. **Un valor positivo y alto aquí es tu principal objetivo**, ya que indica que el rendimiento promedio ha mejorado de forma consistente.

*   **`max_f1` (y `max_diff`)**: Esta métrica se centra en el rendimiento pico. Compara el mejor F1-score individual que se consiguió en `v3` (`max_f1_v3`) con el mejor récord histórico (`max_f1_v1_v2`). La columna **`max_diff`** te muestra si has batido un nuevo "récord" de rendimiento. Es útil para ver el potencial máximo de una configuración, aunque una mejora aquí podría ser un golpe de suerte si la media no ha subido.

*   **`std_f1` (Desviación Estándar)**: Esta métrica mide la **estabilidad y consistencia** de tus resultados. Un valor bajo es bueno, ya que significa que la mayoría de los entrenamientos para ese embedding obtuvieron un F1-score muy similar. Si **`std_f1_v3` es menor que `std_f1_v1_v2`**, es una excelente noticia, porque indica que tus nuevos entrenamientos no solo son mejores, sino también más fiables y predecibles.

In [ ]:
from src.utils.comparison_results import compare_model_results

MODEL_TYPE_MLP = "MLP"
HISTORICAL_FILES_MLP = [
    "results/MLP/mlp_results_v1_Optuna.csv",
    "results/MLP/mlp_results_v2.csv"
]
FINAL_FILE_MLP = "results/MLP/mlp_results_v3.csv"
OUTPUT_DIR_MLP = "results/Comparisons_V1_V2_V3"

# 2. Llama a la función universal
compare_model_results(
    model_type=MODEL_TYPE_MLP,
    historical_files=HISTORICAL_FILES_MLP,
    final_file_path=FINAL_FILE_MLP,
    output_dir=OUTPUT_DIR_MLP
)

## V4: Entrenamientos con embeddings combinados (MLP)

En la fase **V4** se evalúa el comportamiento de la arquitectura **MLP** cuando la entrada no proviene de un único extractor, sino de **embeddings combinados**. En concreto, se trabaja con dos estrategias habituales: **concat** (concatenación de vectores) y **mean** (promedio entre embeddings), generando así dos subconjuntos de datos independientes dentro de `Gait_Embeddings_Combined/`.

A nivel metodológico, V4 mantiene la idea central de las fases anteriores: reutilizar **presets** de configuraciones ya seleccionadas y entrenarlas de forma sistemática sobre todos los embeddings disponibles. Sin embargo, en embeddings combinados se observó que el entrenamiento puede volverse más sensible a la escala de los gradientes y a la dominancia de la clase mayoritaria, por lo que se presta especial atención a técnicas de estabilidad y desbalance.

En particular, el uso de **class weights** permite compensar el desbalance entre clases para que la pérdida no esté dominada por la clase mayoritaria. El **gradient clipping** actúa como mecanismo de estabilidad evitando actualizaciones excesivas cuando la norma del gradiente crece (algo más probable al aumentar dimensionalidad o al combinar representaciones). Finalmente, el **label smoothing** puede aportar regularización adicional reduciendo la sobreconfianza del modelo, aunque su beneficio real depende del coste de convergencia y del comportamiento observado en cada arquitectura.

En consecuencia, V4 se plantea como una fase de verificación: comprobar si las configuraciones que funcionan bien en embeddings individuales siguen siendo competitivas en embeddings combinados, y registrar de forma separada los resultados de **concat** y **mean** para comparar su impacto en pérdida, estabilidad y macro-F1.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.mlp.prepare_data import prepare_mlp_data
from src.mlp.model import FlexibleMLP
from src.mlp.train import run_training_mlp
from src.utils.save_methods import save_model


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
TEST_SIZE = 0.1
EPOCHS = 5000
THRESHOLD = 0.60

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None

RESULTS_DIR = "results/MLP"
os.makedirs(RESULTS_DIR, exist_ok=True)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")


EXPERIMENTS = [
    {
        "name": "v4_concat",
        "data_dir": "Gait_Embeddings_Combined",
        "subfolder": "concat",
        "presets_csv": "presets_configs/MLP/best_configs_mlp.csv", # Best configs seleccionados, y otros presets posteriores
        "results_csv": os.path.join(RESULTS_DIR, "mlp_results_v4_concat.csv"),
        "save_dir": "saved_models/MLP/v4_concat",
        "label_smooth": 0.0,
        "clip_norm": None,
    },
    {
        "name": "v4_mean",
        "data_dir": "Gait_Embeddings_Combined",
        "subfolder": "mean",
        "presets_csv": "presets_configs/MLP/best_configs_mlp.csv", # Best configs seleccionados, y otros presets posteriores
        "results_csv": os.path.join(RESULTS_DIR, "mlp_results_v4_mean.csv"),
        "save_dir": "saved_models/MLP/v4_mean",
        "label_smooth": 0.0,
        "clip_norm": None,
    },
    {
        "name": "v4_reduced",
        "data_dir": "Gait_Embeddings_Combined",
        "subfolder": "reduced",
        "presets_csv": "presets_configs/MLP/best_configs_mlp.csv", # Best configs seleccionados, y otros presets posteriores
        "results_csv": os.path.join(RESULTS_DIR, "mlp_results_v4_reduced.csv"),
        "save_dir": "saved_models/MLP/v4_reduced",
        "label_smooth": 0.02,
        "clip_norm": 1.0,
    },
]

RUN_EXPERIMENTS = ["v4_reduced"]


def _as_bool(x):
    return str(x).strip().lower() in ["true", "1", "yes"]


def _safe_list(v, default="[]"):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ast.literal_eval(default)
    if isinstance(v, str):
        return ast.literal_eval(v)
    return v


for exp in EXPERIMENTS:
    if exp["name"] not in RUN_EXPERIMENTS:
        continue

    data_dir = exp["data_dir"]
    subfolder = exp["subfolder"]
    presets_csv = exp["presets_csv"]
    results_csv = exp["results_csv"]
    save_dir = exp["save_dir"]
    label_smooth = exp["label_smooth"]
    clip_norm = exp["clip_norm"]

    os.makedirs(save_dir, exist_ok=True)

    if not os.path.exists(presets_csv):
        raise FileNotFoundError(f"No existe presets_csv: {presets_csv}")

    presets = pd.read_csv(presets_csv)

    if subfolder is not None:
        emb_dir = os.path.join(data_dir, subfolder)
    else:
        emb_dir = data_dir

    if not os.path.isdir(emb_dir):
        print(f"No existe emb_dir: {emb_dir} (se salta {exp['name']})")
        continue

    emb_files = sorted([f for f in os.listdir(emb_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f"No hay CSVs en: {emb_dir} (se salta {exp['name']})")
        continue

    id_counter = get_max_id([results_csv])

    print(f"\n[{exp['name']}] embeddings={len(emb_files)} | start_id={id_counter + 1}")
    print(f"[{exp['name']}] results_csv={results_csv}")
    print(f"[{exp['name']}] save_dir={save_dir}")

    for preset_idx, row in presets.iterrows():
        print(f"\n[{exp['name']}] Preset {preset_idx + 1}/{len(presets)}")
        all_results = []

        pooling = row["pooling"]
        norm = row["norm"]

        use_pca = _as_bool(row["use_pca"])
        pca_frac = None if pd.isna(row.get("pca_frac", None)) else float(row["pca_frac"])

        num_layers = int(row["num_layers"])
        base_dim = int(row["base_dim"])

        hidden_layers = _safe_list(row["hidden_layers"])
        activation = row["activation"]
        dropout_rate = float(row["dropout_rate"])

        norm_layers = _safe_list(row["norm_layers"] if "norm_layers" in row else row.get("norm_layer", "[]"))
        dropout_layers = _safe_list(row["dropout_layers"] if "dropout_layers" in row else row.get("do_layer", "[]"))

        batch_size = int(row["batch_size"])
        optimizer_name = row["optimizer"]
        lr = float(row["learning_rate"]) if "learning_rate" in row else float(row["lr"])
        weight_decay = float(row["weight_decay"])
        momentum_sgd = float(row["momentum_sgd"]) if str(optimizer_name).upper() == "SGD" else None

        for fname in emb_files:
            df = pd.read_csv(os.path.join(emb_dir, fname), low_memory=False)

            tr_ds, te_ds, train_ids, test_ids = prepare_mlp_data(
                df,
                pooling=pooling,
                norm=norm,
                test_size=TEST_SIZE,
                use_pca=use_pca,
                pca_frac=pca_frac,
                seed=SEED,
            )

            tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
            te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

            input_dim = tr_ds[0][0].shape[0]

            model = FlexibleMLP(
                input_dim=input_dim,
                hidden_layers=hidden_layers,
                activation=activation,
                dropout=dropout_rate,
                normalization_layers=norm_layers,
                dropout_layers=dropout_layers
            ).to(DEVICE)

            id_counter += 1
            run_id = id_counter

            history, best_state = run_training_mlp(
                model,
                tr_loader,
                te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
                label_smoothing=label_smooth,
                clip_grad_norm=clip_norm
            )

            config = {
                "input_dim": input_dim,
                "hidden_layers": hidden_layers,
                "pooling": pooling,
                "norm": norm,
                "batch_size": batch_size,
                "num_layers": num_layers,
                "base_dim": base_dim,
                "activation": activation,
                "dropout_rate": dropout_rate,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
                "use_pca": use_pca,
                "pca_frac": pca_frac,
                "norm_layers": norm_layers,
                "dropout_layers": dropout_layers,
            }

            all_results.append({
                "id": run_id,
                "timestamp": timestamp,
                "target_embedding": fname,
                "source_params": f"Preset_{preset_idx + 1}",
                "model": "MLP",
                "seed": SEED,
                **config,
                "class_weights": history.get("class_weights", None),
                "label_smoothing": label_smooth,
                "clip_grad_norm": clip_norm,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                extra = f"_{subfolder}" if subfolder is not None else ""
                model_path = os.path.join(
                    save_dir,
                    f"mlp{run_id}_v4{extra}_{fname.replace('.csv', '')}.pth"
                )
                save_model(model, model_path, config, run_id)

        df_results = pd.DataFrame(all_results)

        if os.path.exists(results_csv):
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)
        else:
            df_results.to_csv(results_csv, index=False)

print("\nEntrenamientos MLP V4 completados.")


## V5: Entrenamiento con embeddings Top-K Steps (MLP)

En la fase **V5** se evalúa el rendimiento de la arquitectura **MLP** cuando los embeddings han sido generados conservando únicamente un porcentaje **Top-K** de pasos (por ejemplo **Top-30%**, **Top-50%** y **Top-70%**). El objetivo es comprobar si, al filtrar la secuencia y mantener solo los pasos más relevantes, el vector final resulta más discriminativo para la clasificación de la dirección del penalti.

El flujo de ejecución mantiene el mismo protocolo de entrenamiento que en fases anteriores, cambiando únicamente la **fuente de datos**:

- Los embeddings se cargan desde subcarpetas específicas dentro de `Gait_Embeddings_TopSteps/` (una por variante Top-K).  
- Para cada variante (30/50/70), se recorren todos los CSV disponibles y se entrenan modelos con un conjunto de **presets** (`best_configs_mlp.csv`) previamente seleccionados.  
- La función `prepare_mlp_data` agrega los frames por `video_ID` aplicando **pooling temporal** (`mean` o `max`), realiza un split estratificado **train/test (90/10)** y aplica la **normalización** (MinMax o L2). Si el preset lo indica, también aplica **PCA** usando únicamente el train para el ajuste.

El modelo se construye mediante `FlexibleMLP` usando exactamente los hiperparámetros del preset (capas ocultas, activación, dropout y selección de capas con normalización/dropout). El entrenamiento se ejecuta con `run_training_mlp` aplicando la “receta” definida para V5:

- **class weights** para compensar el desbalance entre clases,  
- **gradient clipping** para estabilizar el entrenamiento,  
- **label smoothing** como regularizador para reducir la sobreconfianza.

Los resultados se guardan en un CSV independiente por variante Top-K (Top-30/50/70), registrando: hiperparámetros, métricas finales y métricas del mejor estado (`best_f1`, `best_epoch`, `best_f1_per_class`, pérdidas y accuracy). Además, se incluye `top_variant` para mantener trazabilidad de la variante usada. Finalmente, si el modelo supera el umbral `THRESHOLD` en `best_f1`, el checkpoint se guarda en la carpeta correspondiente a esa variante.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.mlp.prepare_data import prepare_mlp_data
from src.mlp.model import FlexibleMLP
from src.mlp.train import run_training_mlp
from src.utils.save_methods import save_model


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "Gait_Embeddings_TopSteps"
PRESETS_CSV = "presets_configs/MLP/best_configs_mlp.csv"

RESULTS_DIR = "results/MLP"
os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_CSV_TOP30 = os.path.join(RESULTS_DIR, "mlp_results_v5_top30%.csv")
RESULTS_CSV_TOP50 = os.path.join(RESULTS_DIR, "mlp_results_v5_top50%.csv")
RESULTS_CSV_TOP70 = os.path.join(RESULTS_DIR, "mlp_results_v5_top70%.csv")

SAVE_DIR_TOP30 = "saved_models/MLP/v5_top30%"
SAVE_DIR_TOP50 = "saved_models/MLP/v5_top50%"
SAVE_DIR_TOP70 = "saved_models/MLP/v5_top70%"

os.makedirs(SAVE_DIR_TOP30, exist_ok=True)
os.makedirs(SAVE_DIR_TOP50, exist_ok=True)
os.makedirs(SAVE_DIR_TOP70, exist_ok=True)

TEST_SIZE = 0.1
SEED = 42
EPOCHS = 5000
THRESHOLD = 0.60

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
presets = pd.read_csv(PRESETS_CSV)


def _as_bool(x):
    return str(x).strip().lower() in ["true", "1", "yes"]


def _safe_list(v, default="[]"):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ast.literal_eval(default)
    if isinstance(v, str):
        return ast.literal_eval(v)
    return v


SUBFOLDERS = [
    ("30%", RESULTS_CSV_TOP30, SAVE_DIR_TOP30),
    ("50%", RESULTS_CSV_TOP50, SAVE_DIR_TOP50),
    ("70%", RESULTS_CSV_TOP70, SAVE_DIR_TOP70),
]


for sub_name, results_csv, save_dir in SUBFOLDERS:
    sub_dir = os.path.join(DATA_DIR, sub_name)
    if not os.path.isdir(sub_dir):
        print(f" Subcarpeta no encontrada: {sub_dir} (se salta)")
        continue

    emb_files = sorted([f for f in os.listdir(sub_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f" No hay CSVs en: {sub_dir} (se salta)")
        continue

    id_counter = get_max_id([results_csv])

    print(f"\n[{sub_name}] embeddings={len(emb_files)} | start_id={id_counter + 1}")
    print(f"[{sub_name}] results_csv={results_csv}")
    print(f"[{sub_name}] save_dir={save_dir}")

    for preset_idx, row in presets.iterrows():
        print(f"\n[{sub_name}] Preset {preset_idx + 1}/{len(presets)}")
        all_results = []

        pooling = row["pooling"]
        norm = row["norm"]

        use_pca = _as_bool(row["use_pca"])
        pca_frac = None if pd.isna(row.get("pca_frac", None)) else float(row["pca_frac"])

        num_layers = int(row["num_layers"])
        base_dim = int(row["base_dim"])

        hidden_layers = _safe_list(row["hidden_layers"])
        activation = str(row["activation"]).strip()
        dropout_rate = float(row["dropout_rate"])

        norm_layers = _safe_list(row["norm_layers"] if "norm_layers" in row else row.get("norm_layer", "[]"))
        dropout_layers = _safe_list(row["dropout_layers"] if "dropout_layers" in row else row.get("do_layer", "[]"))

        batch_size = int(row["batch_size"])
        optimizer_name = str(row["optimizer"]).strip()
        lr = float(row["learning_rate"]) if "learning_rate" in row else float(row["lr"])
        weight_decay = float(row.get("weight_decay", 0.0))
        momentum_sgd = float(row.get("momentum_sgd", 0.0)) if optimizer_name.upper() == "SGD" else None

        for fname in emb_files:
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            tr_ds, te_ds, train_ids, test_ids = prepare_mlp_data(
                df,
                pooling=pooling,
                norm=norm,
                test_size=TEST_SIZE,
                use_pca=use_pca,
                pca_frac=pca_frac,
                seed=SEED,
            )

            tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
            te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

            input_dim = tr_ds[0][0].shape[0]

            model = FlexibleMLP(
                input_dim=input_dim,
                hidden_layers=hidden_layers,
                activation=activation,
                dropout=dropout_rate,
                normalization_layers=norm_layers,
                dropout_layers=dropout_layers
            ).to(DEVICE)

            id_counter += 1
            run_id = id_counter

            history, best_state = run_training_mlp(
                model,
                tr_loader,
                te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM
            )

            config = {
                "input_dim": input_dim,
                "hidden_layers": hidden_layers,
                "pooling": pooling,
                "norm": norm,
                "batch_size": batch_size,
                "num_layers": num_layers,
                "base_dim": base_dim,
                "activation": activation,
                "dropout_rate": dropout_rate,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
                "use_pca": use_pca,
                "pca_frac": pca_frac,
                "norm_layers": norm_layers,
                "dropout_layers": dropout_layers,
            }

            cw_to_store = str(CLASS_WEIGHTS) if (USE_CLASS_WEIGHTS and CLASS_WEIGHTS is not None) else history.get("class_weights", None)

            all_results.append({
                "id": run_id,
                "timestamp": timestamp,
                "target_embedding": fname,
                "top_variant": sub_name,
                "source_params": f"Preset_{preset_idx + 1}",
                "model": "MLP",
                "seed": SEED,
                **config,
                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                model_path = os.path.join(
                    save_dir,
                    f"mlp{run_id}_v5_top{sub_name.replace('%','')}_{fname.replace('.csv','')}.pth"
                )
                save_model(model, model_path, config, run_id)

        df_results = pd.DataFrame(all_results)
        if os.path.exists(results_csv):
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)
        else:
            df_results.to_csv(results_csv, index=False)

print("\nEntrenamientos MLP V5 completados (Top-30/50/70).")


## V5: Prueba rápida con embeddings Low-K Steps (MLP)

Esta celda ejecuta una **prueba comparativa** entrenando la MLP sobre embeddings generados con **Low-K steps** (peores pasos). El objetivo es únicamente verificar que, de forma general, estos embeddings tienden a rendir **algo peor** que los embeddings completos o los **Top-K**, sin necesidad de guardar checkpoints `.pth` (solo se registran métricas en CSV).


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.mlp.prepare_data import prepare_mlp_data
from src.mlp.model import FlexibleMLP
from src.mlp.train import run_training_mlp
from src.utils.find_best import topK_from_each_topx_csv


topK_from_each_topx_csv(
    input_dir="results/MLP/",
    output_dir="presets_configs/MLP/preset_top_K",
    model_name="MLP",
    top_k=5,
    dedup=True
)


DATA_DIR = "Gait_Embeddings_LowSteps"

VARIANTS = [
    ("30%", "presets_configs/MLP/preset_top_K/mlp_results_v5_top30%_top5.csv", "results/MLP/mlp_results_v5_low30%.csv"),
    ("50%", "presets_configs/MLP/preset_top_K/mlp_results_v5_top50%_top5.csv", "results/MLP/mlp_results_v5_low50%.csv"),
    ("70%", "presets_configs/MLP/preset_top_K/mlp_results_v5_top70%_top5.csv", "results/MLP/mlp_results_v5_low70%.csv"),
]

os.makedirs("results/MLP", exist_ok=True)

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

for sub_name, presets_csv, results_csv in VARIANTS:
    sub_dir = os.path.join(DATA_DIR, sub_name)
    if not os.path.isdir(sub_dir):
        print(f"No existe: {sub_dir} (skip)")
        continue
    if not os.path.exists(presets_csv):
        print(f"No existe: {presets_csv} (skip)")
        continue

    presets = pd.read_csv(presets_csv, low_memory=False)
    emb_files = sorted([f for f in os.listdir(sub_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f"No hay embeddings en: {sub_dir} (skip)")
        continue

    id_counter = get_max_id([results_csv]) + 1
    first_write = not os.path.exists(results_csv)

    print(f"\n=== LOW {sub_name} | presets={len(presets)} | embeddings={len(emb_files)} ===")
    print(f"-> results: {results_csv}")
    print(f"[IDs] starting id = {id_counter}")

    for preset_idx, row in presets.iterrows():
        all_results = []
        best_f1_ref = row["best_f1"] if "best_f1" in row else None
        src = str(row["source_params"]).strip() if "source_params" in row else f"Preset_{preset_idx+1}"
        print(f"--- {sub_name} preset {preset_idx+1}/{len(presets)} | ref_best_f1={best_f1_ref} ---")

        hidden_layers = ast.literal_eval(str(row["hidden_layers"]))
        norm_layers = ast.literal_eval(str(row["norm_layers"]))
        dropout_layers = ast.literal_eval(str(row["dropout_layers"]))

        dropout_rate = float(row["dropout_rate"])
        activation = str(row["activation"]).strip()
        base_dim = int(row["base_dim"])
        batch_size = int(row["batch_size"])
        num_layers = int(row["num_layers"])
        pooling = str(row["pooling"]).strip()
        norm = str(row["norm"]).strip()

        optimizer_name = str(row["optimizer"]).strip()
        momentum_sgd = None if pd.isna(row.get("momentum_sgd", None)) else float(row["momentum_sgd"])
        lr = float(row["learning_rate"])
        weight_decay = float(row["weight_decay"])

        use_pca = bool(row["use_pca"])
        pca_frac = None if pd.isna(row["pca_frac"]) else float(row["pca_frac"])

        for fname in emb_files:
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            tr_ds, te_ds, _, _ = prepare_mlp_data(
                df,
                pooling=pooling,
                norm=norm,
                test_size=TEST_SIZE,
                use_pca=use_pca,
                pca_frac=pca_frac,
                seed=SEED,
            )

            tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
            te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

            input_dim = tr_ds[0][0].shape[0]

            model = FlexibleMLP(
                input_dim=input_dim,
                hidden_layers=hidden_layers,
                activation=activation,
                dropout=dropout_rate,
                normalization_layers=norm_layers,
                dropout_layers=dropout_layers,
            ).to(DEVICE)

            run_id = id_counter
            id_counter += 1

            history, _ = run_training_mlp(
                model,
                tr_loader,
                te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM,
            )

            config = {
                "input_dim": input_dim,
                "hidden_layers": hidden_layers,
                "norm_layers": norm_layers,
                "dropout_layers": dropout_layers,
                "dropout_rate": dropout_rate,
                "activation": activation,
                "base_dim": base_dim,
                "batch_size": batch_size,
                "num_layers": num_layers,
                "pooling": pooling,
                "norm": norm,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
                "use_pca": use_pca,
                "pca_frac": pca_frac,
            }

            all_results.append({
                "id": run_id,
                "timestamp": timestamp,
                "target_embedding": fname,
                "top_variant": sub_name,
                "source_params": src,
                "model": "MLP",
                "seed": SEED,
                **config,
                "class_weights": history.get("class_weights", None),
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

        df_results = pd.DataFrame(all_results)
        if first_write:
            df_results.to_csv(results_csv, index=False)
            first_write = False
        else:
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)

print("\n=== DONE: MLP V5 Low-K Steps (solo CSV, sin guardar .pth) ===")


## Chequeo de configuración de MLP

In [ ]:
from src.utils.check_model import check_model_config

model_path = "saved_models/MLP/v4_mean/mlp1_v4_baseline_31steps_mean.pth"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
check_model_config(model_path, device)

## Matriz de Confusión - Curva ROC - Curva PR (Precision Recall)

En el siguiente código se carga un modelo MLP previamente entrenado junto con su configuración guardada, y lo evalúa con un conjunto de prueba. Primero recupera el modelo y su configuración desde el archivo guardado, luego carga los datos de embeddings correspondientes y los preprocesa exactamente como se hizo durante el entrenamiento (usando el mismo pooling, normalización y PCA). Finalmente, evalúa el rendimiento del modelo mostrando la matriz de confusión (que indica cuántas predicciones fueron correctas e incorrectas para cada clase) y las curvas ROC, presentando métricas como la precisión total y desglosando los aciertos y errores.

In [ ]:
import os
import re
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.mlp.model import FlexibleMLP
from src.mlp.prepare_data import prepare_mlp_data
from src.utils.evaluate_model import evaluate_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "Gait_Embeddings_Combined/mean"
MODEL_PATH = "saved_models/MLP/v4_mean/mlp1_v4_baseline_31steps_mean.pth"

TEST_SIZE = 0.1
SEED = 42

def _parse_list(x, default=None):
    if default is None:
        default = []
    if x is None:
        return default
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else default
        except Exception:
            return default
    return default

def _get(cfg, *keys, default=None):
    for k in keys:
        if k in cfg and cfg[k] is not None:
            return cfg[k]
    return default

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
config = checkpoint.get("config", {})
state_dict = checkpoint.get("model_state", checkpoint.get("model_state_dict", {}))

if not state_dict:
    raise ValueError("El checkpoint no contiene 'model_state' ni 'model_state_dict'.")

try:
    embedding_name_part = "_".join(os.path.basename(MODEL_PATH).split("_")[2:])
    EMBEDDING_FILE = embedding_name_part.replace(".pth", ".csv")
except Exception as e:
    raise ValueError(f"No se pudo inferir el embedding desde el nombre del modelo: {e}")

emb_path = os.path.join(DATA_DIR, EMBEDDING_FILE)
if not os.path.exists(emb_path):
    raise FileNotFoundError(f"No existe el embedding CSV inferido: {emb_path}")

hidden_layers = _parse_list(_get(config, "hidden_layers", default=[]))
norm_layers = _parse_list(_get(config, "norm_layers", "norm_layer", default=[]))
dropout_layers = _parse_list(_get(config, "dropout_layers", "do_layer", default=[]))

activation = str(_get(config, "activation", default="relu")).strip()
dropout_rate = float(_get(config, "dropout_rate", "dropout", default=0.0))

pooling = str(_get(config, "pooling", default="mean")).strip()
norm = str(_get(config, "norm", default="minmax")).strip()

use_pca = bool(_get(config, "use_pca", default=False))
pca_frac = _get(config, "pca_frac", default=None)
pca_frac = None if pca_frac is None else float(pca_frac)

batch_size = int(_get(config, "batch_size", default=64))

df = pd.read_csv(emb_path, low_memory=False)
_, te_ds, _, _ = prepare_mlp_data(
    df,
    pooling=pooling,
    norm=norm,
    test_size=TEST_SIZE,
    use_pca=use_pca,
    pca_frac=pca_frac,
    seed=SEED,
)

te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

input_dim = int(_get(config, "input_dim", default=te_ds[0][0].shape[0]))

model = FlexibleMLP(
    input_dim=input_dim,
    hidden_layers=hidden_layers,
    activation=activation,
    dropout=dropout_rate,
    normalization_layers=norm_layers,
    dropout_layers=dropout_layers,
).to(DEVICE)

model.load_state_dict(state_dict, strict=True)
model.eval()

class_names = ["derecha", "centro", "izquierda"]
evaluate_model(model, te_loader, DEVICE, class_names, model_type="mlp")


-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
# Modelo LSTM y BiLSTM

## Preparación de datos, split y bucle de entrenamiento (LSTM)

En la LSTM, a diferencia de la MLP, **no se aplica pooling temporal**: para cada `video_ID` se conserva la secuencia completa de frames `(T, D)` y su etiqueta `shoot_zone`. El método `prepare_lstm_data()` construye estas secuencias, realiza un **split estratificado por vídeo** (train/test), y aplica la **normalización ajustada solo con train**: en modo *MinMax* se ajusta un `MinMaxScaler` con todos los frames de entrenamiento apilados y luego se transforma cada secuencia; en modo *L2* se normaliza cada frame individualmente. Finalmente, devuelve listas de pares `(secuencia, etiqueta)` junto con los IDs de train/test para asegurar trazabilidad.

Como las secuencias tienen longitudes distintas, `collate_sequences()` se encarga de preparar cada batch: aplica **padding** al máximo `T` del batch y devuelve también el vector de **longitudes reales**, necesario para que el modelo procese correctamente el enmascarado/packing temporal.

El entrenamiento se implementa en `run_training_lstm()`, compartiendo la misma filosofía general del resto de arquitecturas: selección de optimizador (Adam/AdamW/SGD/RMSprop), cálculo opcional de **class weights** para mitigar desbalance, soporte de **label smoothing** y **gradient clipping** como regularización/estabilización, y ejecución con *mixed precision* cuando hay GPU. En cada época se registran `train_loss`, `test_loss`, `accuracy` y `macro-F1`, además del F1 por clase. El modelo guarda un checkpoint del mejor estado cuando la pérdida de train está por debajo de un umbral y mejora el F1, y aplica un **early stopping híbrido** combinando estancamiento de pérdida y F1 (más una parada adicional si la pérdida cae a valores extremadamente bajos), para evitar entrenamientos innecesariamente largos.


## Optimización de hiperparámetros con Optuna (LSTM)

Esta celda ejecuta una búsqueda automática de hiperparámetros con **Optuna** para la arquitectura LSTM, lanzando un estudio independiente por cada archivo de embeddings en `Gait_Embeddings_good`. Para cada embedding se realiza primero un **split estratificado por `video_ID`** (90% train / 10% test) y, usando únicamente el conjunto de train, se optimiza el **macro-F1 medio** mediante validación cruzada estratificada **5-fold**.

En cada *trial*, Optuna genera una configuración **siempre válida** para la LSTM (número de capas, patrón de tamaños por capa y compatibilidad con el modo `native_mode`, incluyendo reglas coherentes para `dropout`, `dropout_layers` y `norm_layers`). Además, se muestrean hiperparámetros de entrenamiento y “head” del modelo: bidireccionalidad, tipo de *pooling* temporal, capa fully-connected opcional (`fc_hidden`), normalización posterior a la LSTM, normalización de entrada (MinMax/L2), optimizador (Adam/AdamW/SGD/RMSprop), `lr`, `weight_decay`, `batch_size` y `momentum` en SGD.

La optimización se acelera con **HyperbandPruner** (para descartar trials con bajo rendimiento) y *early stopping* por fold, y al finalizar se guarda (i) un CSV por embedding con los trials en formato “limpio” y reproducible, y (ii) un resumen global `best_params_LSTM.csv` con la mejor configuración por embedding.


In [ ]:
from src.lstm.optuna import optimize_embeddings

best_df = optimize_embeddings()
best_df.head()

## V1: Entrenamiento con las mejores configuraciones de Optuna (LSTM)

Esta celda replica la misma lógica de la fase **V1** aplicada en MLP, pero adaptada a un modelo secuencial (**LSTM**) y a su espacio de hiperparámetros.

A partir del CSV `best_params_LSTM.csv` (generado en la fase Optuna), se recorre cada *configuración ganadora* aprendida sobre un embedding “source”. Para cada una de esas configuraciones, se vuelve a entrenar el modelo sobre **todos los embeddings disponibles** en `Gait_Embeddings_good`, de forma que se evalúa la **transferencia** de esa arquitectura/optimización a distintas representaciones.

Para cada embedding:
- Se cargan las secuencias por `video_ID` y se realiza un **split estratificado train/test** (90/10) manteniendo la etiqueta del vídeo.
- Se aplica la **misma normalización** definida en la configuración (MinMax o L2), ajustando cualquier estadístico **solo con train**.
- Se construyen `DataLoader` con `collate_sequences`, que hace padding dinámico y devuelve también `lengths`, necesario para que la LSTM ignore el padding.

Después:
- Se instancia `FlexibleLSTM` con los hiperparámetros de Optuna (número de capas, tamaños por capa, bidireccionalidad, pooling temporal, capa fully-connected opcional, dropout/norm por capa o en modo nativo, etc.).
- Se entrena con `run_training_lstm`, aplicando el mismo criterio que en el resto del pipeline: registro de métricas por época, selección del mejor estado y early stopping.

Finalmente, cada ejecución deja trazabilidad completa:
- Se guarda una fila por entrenamiento en `lstm_results_v1_Optuna.csv` con métricas finales y mejores (`best_f1`, `best_epoch`, pérdidas, accuracy, F1 por clase, duración, etc.), junto con toda la configuración usada.
- Opcionalmente se guardan los `train_ids/test_ids` para poder reproducir el split.
- Si el modelo supera un umbral de **macro-F1**, se almacena su checkpoint `.pth` con pesos y configuración para evaluación posterior.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.lstm.prepare_data import prepare_lstm_data, collate_sequences
from src.lstm.model import FlexibleLSTM
from src.lstm.train import run_training_lstm
from src.utils.save_methods import save_train_test_ids, save_model


# Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] Using device: {DEVICE}")

DATA_DIR = "Gait_Embeddings_good"
BEST_PARAMS_CSV = "results/LSTM/Optuna/best_params_LSTM.csv"
RESULTS_CSV = "results/LSTM/lstm_results_v1_Optuna.csv"

SAVE_MODEL_DIR = "saved_models/LSTM/v1"
SPLITS_DIR = "results/LSTM/splits"

SAVE_IDS = True
EPOCHS = 5000
F1_THRESHOLD = 0.55
TEST_SIZE = 0.1
SEED = 42
USE_NATIVE = True

os.makedirs(SAVE_MODEL_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)


# Cargar best params de Optuna
best_params_df = pd.read_csv(BEST_PARAMS_CSV, low_memory=False)
embedding_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])


# IDs y modo escritura CSV
current_id = get_max_id([RESULTS_CSV]) + 1
is_first_block = not os.path.exists(RESULTS_CSV)
print(f"[IDs] Starting id = {current_id}")
print(f"[CSV] First write? {is_first_block}")


def parse_list(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if s in ["", "None", "nan"]:
        return []
    try:
        out = ast.literal_eval(s)
        return out if isinstance(out, list) else []
    except Exception:
        return []


# Loop: cada fila = “mejor config” aprendida en un embedding source
for _, param_row in best_params_df.iterrows():
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    results = []

    source_embedding = param_row["embedding"]
    print(f"\n=== Training using Optuna best config from {source_embedding} ===")

    layer_sizes = parse_list(param_row.get("layer_sizes", pd.NA))
    dropout_layers = parse_list(param_row.get("dropout_layers", pd.NA))
    norm_layers = parse_list(param_row.get("norm_layers", pd.NA))

    num_layers = int(param_row["num_layers"])
    dropout = float(param_row["dropout"])

    native_opt = bool(param_row.get("native_mode", False))
    native_mode = native_opt if USE_NATIVE else False

    fc_hidden = param_row.get("fc_hidden", None)
    fc_hidden = None if pd.isna(fc_hidden) else int(fc_hidden)

    activation = param_row.get("activation", None)
    if isinstance(activation, float) and pd.isna(activation):
        activation = None
    if fc_hidden is None:
        activation = None

    config_base = {
        "num_layers": num_layers,
        "layer_sizes": layer_sizes,
        "bidirectional": bool(param_row["bidirectional"]),
        "norm": str(param_row["norm"]).strip(),
        "dropout": dropout,
        "dropout_layers": dropout_layers,
        "norm_layers": norm_layers,
        "pooling": str(param_row["pooling"]).strip(),
        "fc_hidden": fc_hidden,
        "activation": activation,
        "norm_post_lstm": bool(param_row["norm_post_lstm"]),
        "batch_size": int(param_row["batch_size"]),
        "optimizer": str(param_row["optimizer"]).strip(),
        "learning_rate": float(param_row["lr"]),
        "weight_decay": float(param_row["weight_decay"]),
        "momentum_sgd": None,
        "native_mode": native_mode,
    }

    if config_base["optimizer"].upper() == "SGD":
        mom = param_row.get("momentum_sgd", 0.0)
        config_base["momentum_sgd"] = 0.0 if pd.isna(mom) else float(mom)

    # Aplicar la config a todos los embeddings target
    for fname in embedding_files:
        print(f"\n--- Training LSTM for embedding: {fname} ---")

        df_emb = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)

        train_list, test_list, train_ids, test_ids = prepare_lstm_data(
            df_emb,
            norm=config_base["norm"],
            test_size=TEST_SIZE,
            seed=SEED,
        )

        if SAVE_IDS:
            save_train_test_ids(
                fname,
                train_ids,
                test_ids,
                SEED,
                output_dir=os.path.join(SPLITS_DIR, f"seed{SEED}")
            )

        if len(train_list) == 0:
            print(f" Train vacío para {fname} (skip)")
            continue

        input_dim = int(train_list[0][0].shape[1])

        tr_loader = DataLoader(
            train_list,
            batch_size=config_base["batch_size"],
            shuffle=True,
            collate_fn=collate_sequences
        )
        te_loader = DataLoader(
            test_list,
            batch_size=config_base["batch_size"],
            shuffle=False,
            collate_fn=collate_sequences
        )

        model = FlexibleLSTM(
            input_dim=input_dim,
            hidden_dim=None,
            num_layers=config_base["num_layers"],
            bidirectional=config_base["bidirectional"],
            dropout=config_base["dropout"],
            dropout_layers=config_base["dropout_layers"],
            norm_layers=config_base["norm_layers"],
            fc_hidden=config_base["fc_hidden"],
            num_classes=3,
            activation=config_base["activation"],
            pooling=config_base["pooling"],
            norm_post_lstm=config_base["norm_post_lstm"],
            layer_sizes=config_base["layer_sizes"],
            native_mode=config_base["native_mode"],
        ).to(DEVICE)

        run_id = current_id
        current_id += 1

        history, best_state = run_training_lstm(
            model,
            tr_loader,
            te_loader,
            epochs=EPOCHS,
            lr=config_base["learning_rate"],
            weight_decay=config_base["weight_decay"],
            optimizer_name=config_base["optimizer"],
            momentum_sgd=config_base["momentum_sgd"],
        )

        cfg = dict(config_base)
        cfg["input_dim"] = input_dim

        results.append({
            "id": run_id,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": source_embedding,
            "model": "LSTM",
            "seed": SEED,
            **cfg,
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": int(history["best_epoch"]),
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "epochs_trained": int(history["epochs_trained"]),
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        })

        if history["best_f1"] >= F1_THRESHOLD:
            print(f"  -> Saving checkpoint (best F1={history['best_f1']:.4f})")
            model.load_state_dict(best_state)
            path = os.path.join(SAVE_MODEL_DIR, f"lstm{run_id}_v1_{fname.replace('.csv','')}.pth")
            save_model(model, path, cfg, run_id)

    df_out = pd.DataFrame(results)

    if is_first_block:
        df_out.to_csv(RESULTS_CSV, index=False)
        is_first_block = False
    else:
        with open(RESULTS_CSV, "a", newline="") as f:
            f.write("\n")
        df_out.to_csv(RESULTS_CSV, mode="a", header=False, index=False)

    print(f"\n Block for {source_embedding} saved to {RESULTS_CSV}")

print("\nAll LSTM v1 trainings completed successfully.\n")


## V2: Entrenamiento a partir de configuraciones personalizadas (LSTM)

En esta fase se cargan configuraciones personalizadas ya definidas en un CSV y se reutilizan para entrenar la LSTM sobre **todos los embeddings** disponibles. Para cada preset se reconstruye la arquitectura (capas y tamaños, modo nativo/modular, bidireccionalidad y pooling) y los hiperparámetros de optimización (optimizador, LR, weight decay, batch size), se prepara el split train/test por `video_ID`, y se ejecuta el bucle de entrenamiento guardando los resultados en un CSV. Opcionalmente, si el mejor macro-F1 supera un umbral, se guarda también el checkpoint del modelo.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.lstm.prepare_data import prepare_lstm_data, collate_sequences
from src.lstm.model import FlexibleLSTM
from src.lstm.train import run_training_lstm
from src.utils.save_methods import save_model


DATA_DIR = "Gait_Embeddings_good"
PRESETS_CSV = "presets_configs/LSTM/best_configs_lstm.csv"
RESULTS_CSV = "results/LSTM/lstm_results_v2.csv"
SAVE_DIR = "saved_models/LSTM/v2"

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000
THRESHOLD = 0.55

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)

id_counter = get_max_id([RESULTS_CSV]) + 1

presets = pd.read_csv(PRESETS_CSV, low_memory=False)
emb_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")


def parse_list(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if s in ["", "None", "nan"]:
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []


def parse_bool(val):
    return str(val).strip().lower() in ["true", "1", "yes"]


for preset_idx, row in presets.iterrows():
    print(f"\n=== Ejecutando PRESET {preset_idx + 1}/{len(presets)} ===")
    all_results = []

    layer_sizes = parse_list(row.get("layer_sizes"))
    dropout_layers = parse_list(row.get("dropout_layers"))
    norm_layers = parse_list(row.get("norm_layers"))

    num_layers = int(row["num_layers"])
    dropout = float(row["dropout"])
    pooling = str(row["pooling"]).strip()
    norm_mode = str(row["norm"]).strip()

    optimizer_name = str(row["optimizer"]).strip()
    batch_size = int(row["batch_size"])
    lr = float(row["learning_rate"])
    weight_decay = float(row["weight_decay"])

    bidirectional = parse_bool(row.get("bidirectional", False))
    norm_post_lstm = parse_bool(row.get("norm_post_lstm", False))
    native_mode = parse_bool(row.get("native_mode", False))

    fc_hidden = row.get("fc_hidden", None)
    if pd.isna(fc_hidden):
        fc_hidden = None
    else:
        fc_hidden = int(fc_hidden)

    activation = row.get("activation", None)
    if pd.isna(activation):
        activation = None
    else:
        activation = str(activation).strip()

    momentum_sgd = None
    if optimizer_name.upper() == "SGD":
        mom = row.get("momentum_sgd", 0.0)
        momentum_sgd = float(mom) if not pd.isna(mom) else 0.0

    for fname in emb_files:
        print(f"  -> Entrenando LSTM v2 con embedding: {fname}")

        df_emb = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)

        tr_list, te_list, train_ids, test_ids = prepare_lstm_data(
            df_emb,
            norm=norm_mode,
            test_size=TEST_SIZE,
            seed=SEED,
        )

        tr_loader = DataLoader(
            tr_list,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_sequences,
        )
        te_loader = DataLoader(
            te_list,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_sequences,
        )

        input_dim = df_emb.filter(like="feat_").shape[1]

        config = {
            "input_dim": input_dim,
            "num_layers": num_layers,
            "layer_sizes": layer_sizes,
            "bidirectional": bidirectional,
            "norm": norm_mode,
            "dropout": dropout,
            "dropout_layers": dropout_layers,
            "norm_layers": norm_layers,
            "pooling": pooling,
            "fc_hidden": fc_hidden,
            "activation": activation,
            "norm_post_lstm": norm_post_lstm,
            "batch_size": batch_size,
            "optimizer": optimizer_name,
            "momentum_sgd": momentum_sgd,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "native_mode": native_mode,
        }

        model = FlexibleLSTM(
            input_dim=input_dim,
            hidden_dim=None,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout,
            dropout_layers=dropout_layers,
            norm_layers=norm_layers,
            fc_hidden=fc_hidden,
            activation=(activation if fc_hidden is not None else "relu"),
            pooling=pooling,
            norm_post_lstm=norm_post_lstm,
            num_classes=3,
            layer_sizes=layer_sizes,
            native_mode=native_mode,
        ).to(DEVICE)

        run_id = id_counter
        id_counter += 1

        history, best_state = run_training_lstm(
            model,
            tr_loader,
            te_loader,
            epochs=EPOCHS,
            lr=lr,
            weight_decay=weight_decay,
            optimizer_name=optimizer_name,
            momentum_sgd=momentum_sgd,
        )

        all_results.append({
            "id": run_id,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": f"Preset_{preset_idx + 1}",
            "model": "LSTM",
            "seed": SEED,
            **config,
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
            "epochs_trained": int(history["epochs_trained"]),
            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": int(history["best_epoch"]),
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        })

        if history["best_f1"] >= THRESHOLD:
            model.load_state_dict(best_state)
            model_path = os.path.join(SAVE_DIR, f"lstm{run_id}_v2_{fname.replace('.csv','')}.pth")
            save_model(model, model_path, config, run_id)

    df_block = pd.DataFrame(all_results)
    if os.path.exists(RESULTS_CSV):
        with open(RESULTS_CSV, "a", newline="") as f:
            f.write("\n")
        df_block.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_block.to_csv(RESULTS_CSV, index=False)

print("\n==== ENTRENAMIENTOS LSTM v2 COMPLETADOS ====\n")


## V3: Re-entrenamiento robusto (LSTM)

Esta fase vuelve a entrenar **las mejores configuraciones ya encontradas en V1–V2**, pero aplicando una “receta” más robusta (ponderación de clases, recorte de gradiente y suavizado de etiquetas). Para mantener trazabilidad, se reconstruye un **lookup** que asigna a cada pareja *(embedding objetivo, configuración)* el **mismo `id`** que tenía en V1–V2 (firma estable por hiperparámetros). Con ello, cada configuración se re-ejecuta sobre todos los embeddings, se guardan las métricas finales y las mejores (`best_f1`, `best_epoch`, etc.) en un CSV V3, y opcionalmente se guarda el checkpoint si supera el umbral de rendimiento.


In [ ]:
import os
import json
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.find_best import find_best_configs
from src.utils.save_methods import save_model
from src.lstm.prepare_data import prepare_lstm_data, collate_sequences
from src.lstm.model import FlexibleLSTM
from src.lstm.train import run_training_lstm


DATA_DIR = "Gait_Embeddings_good"
RESULTS_CSV = "results/LSTM/lstm_results_v3.csv"
SAVE_DIR = "saved_models/LSTM/v3"

HISTORICAL = [
    "results/LSTM/lstm_results_v1_Optuna.csv",
    "results/LSTM/lstm_results_v2.csv",
]

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000
THRESHOLD = 0.60

LABEL_SMOOTH = 0.03
CLIP_NORM = 1.0
CLASS_WEIGHTS = [1.0, 3.1864, 1.3333]

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)


MATCH_COLS = [
    "num_layers", "layer_sizes", "bidirectional", "norm", "dropout",
    "dropout_layers", "norm_layers", "pooling", "fc_hidden", "activation",
    "norm_post_lstm", "batch_size", "optimizer", "momentum_sgd",
    "learning_rate", "weight_decay", "native_mode",
]


def normalize_value(v):
    if pd.isna(v):
        return None

    if isinstance(v, str):
        s = v.strip()
        if s == "" or s.lower() in ["none", "nan"]:
            return None

        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        if s.lower() in ["true", "false"]:
            return s.lower() == "true"

        return s.lower()

    return v


def normalize_config_row(row):
    clean = {}
    for col in MATCH_COLS:
        if col == "learning_rate":
            v = row.get("learning_rate", None)
            if pd.isna(v) or v is None:
                v = row.get("lr", None)
            clean[col] = normalize_value(v)
        else:
            clean[col] = normalize_value(row.get(col))
    return clean


def config_signature(row):
    return json.dumps(normalize_config_row(row), sort_keys=True)


def parse_list(v):
    if pd.isna(v):
        return []
    if isinstance(v, str):
        s = v.strip()
        if s == "" or s.lower() in ["none", "nan"]:
            return []
        try:
            out = ast.literal_eval(s)
            return out if isinstance(out, list) else []
        except Exception:
            return []
    return v if isinstance(v, list) else []


hist_files = [p for p in HISTORICAL if os.path.exists(p)]
if not hist_files:
    raise FileNotFoundError("No se encontraron archivos históricos LSTM (V1/V2).")

df_hist = pd.concat([pd.read_csv(f, low_memory=False) for f in hist_files], ignore_index=True)
df_hist["__sig__"] = df_hist.apply(config_signature, axis=1)

LOOKUP = {
    (row["target_embedding"], row["__sig__"]): int(row["id"])
    for _, row in df_hist.iterrows()
}

print(f"[LOOKUP] Configuraciones indexadas: {len(LOOKUP)}")


best_configs_path = "results/LSTM/best_configs_lstm.csv"
if not os.path.exists(best_configs_path):
    presets_df = find_best_configs(
        csv_files=hist_files,
        output_file=best_configs_path,
        model_name="LSTM",
        f1_threshold=THRESHOLD,
    )
else:
    presets_df = pd.read_csv(best_configs_path, low_memory=False)

if presets_df is None or presets_df.empty:
    raise ValueError("No se encontraron configuraciones buenas para V3.")

print(f"[PRESETS] Seleccionadas: {len(presets_df)}")


emb_list = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

for idx, prow in presets_df.iterrows():
    print(f"\n=== PRESET {idx + 1}/{len(presets_df)} ===")

    p_sig = config_signature(prow)
    all_results = []

    for fname in emb_list:
        run_id = LOOKUP.get((fname, p_sig))
        if run_id is None:
            print(f"  -> {fname}: sin ID previo para esta config (skip)")
            continue

        layer_sizes = parse_list(prow.get("layer_sizes"))
        dropout_layers = parse_list(prow.get("dropout_layers"))
        norm_layers = parse_list(prow.get("norm_layers"))

        num_layers = len(layer_sizes) if layer_sizes else int(prow.get("num_layers", 1))
        bidirectional = bool(normalize_value(prow.get("bidirectional", False)))
        norm_post_lstm = bool(normalize_value(prow.get("norm_post_lstm", False)))
        native_mode = bool(normalize_value(prow.get("native_mode", False)))

        pooling = prow.get("pooling", "last")
        pooling = "last" if pd.isna(pooling) else str(pooling).strip().lower()

        norm = prow.get("norm", "")
        norm = "" if pd.isna(norm) else str(norm).strip()

        dropout = float(prow.get("dropout", 0.0)) if not pd.isna(prow.get("dropout", 0.0)) else 0.0
        batch_size = int(float(prow.get("batch_size", 64)))

        optimizer_name = str(prow.get("optimizer", "AdamW")).strip()
        lr = prow.get("learning_rate", prow.get("lr", 1e-4))
        lr = float(lr)

        weight_decay = float(prow.get("weight_decay", 0.0)) if not pd.isna(prow.get("weight_decay", 0.0)) else 0.0

        momentum_sgd = None
        if optimizer_name.upper() == "SGD":
            m = prow.get("momentum_sgd", 0.0)
            momentum_sgd = 0.0 if pd.isna(m) else float(m)

        fc_hidden = prow.get("fc_hidden", None)
        if pd.isna(fc_hidden):
            fc_hidden = None
        else:
            fc_hidden = int(float(fc_hidden))

        activation = prow.get("activation", None)
        if fc_hidden is None:
            activation = None
        else:
            activation = "relu" if pd.isna(activation) else str(activation).strip().lower()

        config = {
            "num_layers": num_layers,
            "layer_sizes": layer_sizes,
            "bidirectional": bidirectional,
            "norm": norm,
            "dropout": dropout,
            "dropout_layers": dropout_layers,
            "norm_layers": norm_layers,
            "pooling": pooling,
            "fc_hidden": fc_hidden,
            "activation": activation,
            "norm_post_lstm": norm_post_lstm,
            "batch_size": batch_size,
            "optimizer": optimizer_name,
            "momentum_sgd": momentum_sgd,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "native_mode": native_mode,
        }

        df_emb = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
        train_list, test_list, _, _ = prepare_lstm_data(
            df_emb,
            norm=norm,
            test_size=TEST_SIZE,
            seed=SEED,
        )

        tr_loader = DataLoader(train_list, batch_size=batch_size, shuffle=True, collate_fn=collate_sequences)
        te_loader = DataLoader(test_list, batch_size=batch_size, shuffle=False, collate_fn=collate_sequences)

        input_dim = df_emb.filter(like="feat_").shape[1]
        config["input_dim"] = input_dim

        model = FlexibleLSTM(
            input_dim=input_dim,
            hidden_dim=None,
            num_layers=num_layers,
            layer_sizes=(layer_sizes if layer_sizes else None),
            bidirectional=bidirectional,
            dropout=dropout,
            dropout_layers=dropout_layers,
            norm_layers=norm_layers,
            fc_hidden=fc_hidden,
            activation=(activation if fc_hidden is not None else "relu"),
            pooling=pooling,
            norm_post_lstm=norm_post_lstm,
            native_mode=native_mode,
            num_classes=3,
        ).to(DEVICE)

        history, best_state = run_training_lstm(
            model,
            tr_loader,
            te_loader,
            epochs=EPOCHS,
            lr=lr,
            weight_decay=weight_decay,
            optimizer_name=optimizer_name,
            momentum_sgd=momentum_sgd,
            use_class_weights=True,
            class_weights=CLASS_WEIGHTS,
            label_smoothing=LABEL_SMOOTH,
            clip_grad_norm=CLIP_NORM,
        )

        res = {
            "id": run_id,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": "best_configs_v3",
            "model": "LSTM",
            "seed": SEED,
            **config,
            "label_smoothing": LABEL_SMOOTH,
            "clip_grad_norm": CLIP_NORM,
            "class_weights": str(CLASS_WEIGHTS),
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": int(history["best_epoch"]),
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "epochs_trained": int(history["epochs_trained"]),
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        }

        all_results.append(res)

        if history["best_f1"] >= THRESHOLD:
            model.load_state_dict(best_pred_state := best_state)
            model_path = os.path.join(SAVE_DIR, f"lstm{run_id}_v3_{fname.replace('.csv', '')}.pth")
            save_model(model, model_path, config, run_id)

    if all_results:
        df_out = pd.DataFrame(all_results).sort_values(by="id")
        if os.path.exists(RESULTS_CSV):
            with open(RESULTS_CSV, "a", newline="") as f:
                f.write("\n")
            df_out.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
        else:
            df_out.to_csv(RESULTS_CSV, index=False)
        print("  -> Resultados añadidos al CSV V3")
    else:
        print("  -> Sin matches para este preset (nada que guardar)")

print("\n=== ENTRENAMIENTO LSTM V3 FINALIZADO ===")


## Comparación de resultados entre diferentes entrenamientos


*   **`mean_f1` (y `mean_diff`)**: Esta es la métrica más importante para evaluar la mejora general. `mean_f1_v3` es el F1-score *promedio* de todos los entrenamientos de la `v3` para un embedding, mientras que `mean_f1_v1_v2` es el promedio de los históricos. La columna **`mean_diff`** te dice si, en conjunto, tus nuevos entrenamientos son mejores. **Un valor positivo y alto aquí es tu principal objetivo**, ya que indica que el rendimiento promedio ha mejorado de forma consistente.

*   **`max_f1` (y `max_diff`)**: Esta métrica se centra en el rendimiento pico. Compara el mejor F1-score individual que se consiguió en `v3` (`max_f1_v3`) con el mejor récord histórico (`max_f1_v1_v2`). La columna **`max_diff`** te muestra si has batido un nuevo "récord" de rendimiento. Es útil para ver el potencial máximo de una configuración, aunque una mejora aquí podría ser un golpe de suerte si la media no ha subido.

*   **`std_f1` (Desviación Estándar)**: Esta métrica mide la **estabilidad y consistencia** de tus resultados. Un valor bajo es bueno, ya que significa que la mayoría de los entrenamientos para ese embedding obtuvieron un F1-score muy similar. Si **`std_f1_v3` es menor que `std_f1_v1_v2`**, es una excelente noticia, porque indica que tus nuevos entrenamientos no solo son mejores, sino también más fiables y predecibles.

In [ ]:
from src.utils.comparison_results import compare_model_results


MODEL_TYPE_LSTM = "LSTM"
HISTORICAL_FILES_LSTM = [
    "results/LSTM/lstm_results_v1_Optuna.csv",
    "results/LSTM/lstm_results_v2.csv"
]
FINAL_FILE_LSTM = "results/LSTM/lstm_results_v3.csv"
OUTPUT_DIR_LSTM = "results/Comparisons_V1_V2_V3"

compare_model_results(
    model_type=MODEL_TYPE_LSTM,
    historical_files=HISTORICAL_FILES_LSTM,
    final_file_path=FINAL_FILE_LSTM,
    output_dir=OUTPUT_DIR_LSTM
)

## V4: Entrenamientos con embeddings combinados (LSTM)

Esta fase repite el mismo bucle de entrenamiento que en V4, pero separando explícitamente los **presets por variante** (`concat`, `mean`, `reduced`). Para cada variante se carga su CSV de presets correspondiente, se entrenan esos presets contra todos los embeddings de esa carpeta y se guardan los resultados en un CSV independiente por variante (y, opcionalmente, los checkpoints que superen el umbral de `best_f1`).


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.utils.save_methods import save_model
from src.lstm.prepare_data import prepare_lstm_data, collate_sequences
from src.lstm.model import FlexibleLSTM
from src.lstm.train import run_training_lstm


DATA_ROOT = "Gait_Embeddings_Combined"

RESULTS_DIR = "results/LSTM"
SAVE_ROOT = "saved_models/LSTM"

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000
THRESHOLD = 0.60

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0

VARIANTS = [
    ("concat",  os.path.join(DATA_ROOT, "concat"),  "presets_configs/LSTM/presets_lstm_concat.csv"),
    ("mean",    os.path.join(DATA_ROOT, "mean"),    "presets_configs/LSTM/presets_lstm_mean.csv"),
    ("reduced", os.path.join(DATA_ROOT, "reduced"), "presets_configs/LSTM/presets_lstm_reduced.csv"),
]

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SAVE_ROOT, exist_ok=True)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")


def parse_bool(x):
    return str(x).strip().lower() in ["true", "1", "yes"]


def parse_list(x):
    if pd.isna(x):
        return []
    try:
        out = ast.literal_eval(str(x))
        return out if isinstance(out, list) else []
    except Exception:
        return []


def safe_int(x, default=None):
    if x is None or pd.isna(x):
        return default
    try:
        return int(float(x))
    except Exception:
        return default


def safe_float(x, default=None):
    if x is None or pd.isna(x):
        return default
    try:
        return float(x)
    except Exception:
        return default


for variant_name, variant_dir, presets_csv in VARIANTS:
    if not os.path.isdir(variant_dir):
        print(f" No existe: {variant_dir} (skip)")
        continue
    if not os.path.exists(presets_csv):
        print(f" No existe: {presets_csv} (skip)")
        continue

    emb_files = sorted([f for f in os.listdir(variant_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f" No hay CSVs en: {variant_dir} (skip)")
        continue

    presets = pd.read_csv(presets_csv, low_memory=False)

    results_csv = os.path.join(RESULTS_DIR, f"lstm_results_v4_{variant_name}.csv")
    save_dir = os.path.join(SAVE_ROOT, f"v4_{variant_name}")
    os.makedirs(save_dir, exist_ok=True)

    id_counter = get_max_id([results_csv])

    print(f"\n==============================")
    print(f"📂 Variante: {variant_name} | embeddings={len(emb_files)} | presets={len(presets)}")
    print(f"🧾 Presets -> {presets_csv}")
    print(f"📝 Resultados -> {results_csv}")
    print(f"💾 Modelos -> {save_dir}")
    print(f"🆔 ID inicial -> {id_counter + 1}")
    print(f"==============================\n")

    for preset_idx, row in presets.iterrows():
        print(f"\n=== [{variant_name}] PRESET {preset_idx + 1}/{len(presets)} ===")
        all_results = []

        layer_sizes = parse_list(row.get("layer_sizes"))
        dropout_layers = parse_list(row.get("dropout_layers"))
        norm_layers = parse_list(row.get("norm_layers"))

        num_layers = safe_int(row.get("num_layers"), default=(len(layer_sizes) if layer_sizes else 1))
        if layer_sizes and num_layers != len(layer_sizes):
            num_layers = len(layer_sizes)

        bidirectional = parse_bool(row.get("bidirectional", False))
        norm_mode = str(row.get("norm", "L2")).strip()
        dropout = safe_float(row.get("dropout", 0.0), default=0.0)

        pooling = str(row.get("pooling", "last")).strip().lower()
        fc_hidden = safe_int(row.get("fc_hidden"), default=None)

        activation = row.get("activation", None)
        if fc_hidden is None:
            activation = None
        else:
            activation = "relu" if (activation is None or pd.isna(activation)) else str(activation).strip().lower()

        norm_post_lstm = parse_bool(row.get("norm_post_lstm", False))
        native_mode = parse_bool(row.get("native_mode", False))

        batch_size = safe_int(row.get("batch_size"), default=64)
        optimizer_name = str(row.get("optimizer", "AdamW")).strip()
        lr = safe_float(row.get("learning_rate", row.get("lr", 1e-4)), default=1e-4)
        weight_decay = safe_float(row.get("weight_decay", 0.0), default=0.0)
        momentum_sgd = safe_float(row.get("momentum_sgd", 0.0), default=0.0) if optimizer_name.upper() == "SGD" else None

        preset_source = row.get("source_params", f"Preset_{preset_idx + 1}")
        preset_source = str(preset_source).strip()

        for fname in emb_files:
            print(f"  -> [{variant_name}] embedding: {fname}")

            df = pd.read_csv(os.path.join(variant_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_lstm_data(
                df,
                norm=norm_mode,
                test_size=TEST_SIZE,
                seed=SEED,
            )

            tr_loader = DataLoader(
                train_list,
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_sequences,
            )
            te_loader = DataLoader(
                test_list,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_sequences,
            )

            input_dim = int(df.filter(like="feat_").shape[1])

            model = FlexibleLSTM(
                input_dim=input_dim,
                hidden_dim=None,
                num_layers=num_layers,
                layer_sizes=(layer_sizes if layer_sizes else None),
                bidirectional=bidirectional,
                dropout=dropout,
                dropout_layers=dropout_layers,
                norm_layers=norm_layers,
                fc_hidden=fc_hidden,
                activation=(activation if fc_hidden is not None else "relu"),
                pooling=pooling,
                norm_post_lstm=norm_post_lstm,
                native_mode=native_mode,
                num_classes=3,
            ).to(DEVICE)

            id_counter += 1
            run_id = id_counter

            history, best_state = run_training_lstm(
                model,
                tr_loader,
                te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM,
            )

            config = {
                "input_dim": input_dim,
                "num_layers": num_layers,
                "layer_sizes": layer_sizes,
                "bidirectional": bidirectional,
                "norm": norm_mode,
                "dropout": dropout,
                "dropout_layers": dropout_layers,
                "norm_layers": norm_layers,
                "pooling": pooling,
                "fc_hidden": fc_hidden,
                "activation": activation,
                "norm_post_lstm": norm_post_lstm,
                "native_mode": native_mode,
                "batch_size": batch_size,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
            }

            cw_to_store = str(CLASS_WEIGHTS) if (USE_CLASS_WEIGHTS and CLASS_WEIGHTS is not None) else history.get("class_weights", None)

            all_results.append({
                "id": run_id,
                "timestamp": timestamp,
                "target_embedding": fname,
                "variant": variant_name,
                "source_params": preset_source,
                "model": "LSTM",
                "seed": SEED,
                **config,
                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                model_path = os.path.join(save_dir, f"lstm{run_id}_v4_{variant_name}_{fname.replace('.csv','')}.pth")
                save_model(model, model_path, config, run_id)

        df_results = pd.DataFrame(all_results)
        if os.path.exists(results_csv):
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)
        else:
            df_results.to_csv(results_csv, index=False)

print("\n=== ENTRENAMIENTOS LSTM V4 COMPLETADOS (presets separados por variante) ===")


## V5: Entrenamiento con embeddings Top-K Steps

En esta fase se repite el pipeline de entrenamiento de V4, pero usando embeddings donde solo se conservan los **steps más relevantes** (Top-K). Para cada variante (Top30/Top50/Top70) se recorren todos sus CSVs, se aplica cada preset de LSTM y se guardan las métricas (loss, accuracy, macro-F1 y métricas del mejor checkpoint) en un CSV independiente por variante, manteniendo el campo `top_variant` para trazabilidad del recorte aplicado. Los modelos que superan el umbral de `best_f1` se guardan opcionalmente como checkpoint.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.utils.save_methods import save_model
from src.lstm.prepare_data import prepare_lstm_data, collate_sequences
from src.lstm.model import FlexibleLSTM
from src.lstm.train import run_training_lstm


DATA_DIR = "Gait_Embeddings_TopSteps"
PRESETS_CSV = "presets_configs/LSTM/best_configs_lstm.csv"

RESULTS_DIR = "results/LSTM"
SAVE_ROOT = "saved_models/LSTM"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SAVE_ROOT, exist_ok=True)

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000
THRESHOLD = 0.60

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
presets = pd.read_csv(PRESETS_CSV, low_memory=False)

SUBFOLDERS = [
    ("30%", os.path.join(RESULTS_DIR, "lstm_results_v5_top30%.csv"), os.path.join(SAVE_ROOT, "v5_top30%")),
    ("50%", os.path.join(RESULTS_DIR, "lstm_results_v5_top50%.csv"), os.path.join(SAVE_ROOT, "v5_top50%")),
    ("70%", os.path.join(RESULTS_DIR, "lstm_results_v5_top70%.csv"), os.path.join(SAVE_ROOT, "v5_top70%")),
]


def parse_bool(x):
    return str(x).strip().lower() in ["true", "1", "yes"]


def parse_list(x):
    if pd.isna(x):
        return []
    try:
        out = ast.literal_eval(str(x))
        return out if isinstance(out, list) else []
    except Exception:
        return []


def safe_int(x, default=None):
    if x is None or pd.isna(x):
        return default
    try:
        return int(float(x))
    except Exception:
        return default


def safe_float(x, default=None):
    if x is None or pd.isna(x):
        return default
    try:
        return float(x)
    except Exception:
        return default


def get_embedding_files(folder_path: str):
    if not os.path.isdir(folder_path):
        return []
    return sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])


for top_tag, results_csv, save_dir in SUBFOLDERS:
    sub_dir = os.path.join(DATA_DIR, top_tag)
    if not os.path.isdir(sub_dir):
        print(f" No existe: {sub_dir} (skip)")
        continue

    emb_files = get_embedding_files(sub_dir)
    if not emb_files:
        print(f" No hay CSVs en: {sub_dir} (skip)")
        continue

    os.makedirs(save_dir, exist_ok=True)

    id_counter = get_max_id([results_csv])

    print(f"\n==============================")
    print(f"📂 Recorte: {top_tag} | embeddings={len(emb_files)} | presets={len(presets)}")
    print(f"📝 Resultados -> {results_csv}")
    print(f"💾 Modelos -> {save_dir}")
    print(f"🆔 ID inicial -> {id_counter + 1}")
    print(f"==============================\n")

    for preset_idx, row in presets.iterrows():
        print(f"\n=== [{top_tag}] PRESET {preset_idx + 1}/{len(presets)} ===")
        all_results = []

        layer_sizes = parse_list(row.get("layer_sizes"))
        dropout_layers = parse_list(row.get("dropout_layers"))
        norm_layers = parse_list(row.get("norm_layers"))

        num_layers = safe_int(row.get("num_layers"), default=(len(layer_sizes) if layer_sizes else 1))
        if layer_sizes and num_layers != len(layer_sizes):
            num_layers = len(layer_sizes)

        bidirectional = parse_bool(row.get("bidirectional", False))
        norm_mode = str(row.get("norm", "L2")).strip()
        dropout = safe_float(row.get("dropout", 0.0), default=0.0)

        pooling = str(row.get("pooling", "last")).strip().lower()
        fc_hidden = safe_int(row.get("fc_hidden"), default=None)

        activation = row.get("activation", None)
        if fc_hidden is None:
            activation = None
        else:
            activation = "relu" if (activation is None or pd.isna(activation)) else str(activation).strip().lower()

        norm_post_lstm = parse_bool(row.get("norm_post_lstm", False))
        native_mode = parse_bool(row.get("native_mode", False))

        batch_size = safe_int(row.get("batch_size"), default=64)
        optimizer_name = str(row.get("optimizer", "AdamW")).strip()
        lr = safe_float(row.get("learning_rate", row.get("lr", 1e-4)), default=1e-4)
        weight_decay = safe_float(row.get("weight_decay", 0.0), default=0.0)
        momentum_sgd = safe_float(row.get("momentum_sgd", 0.0), default=0.0) if optimizer_name.upper() == "SGD" else None

        preset_source = row.get("source_params", f"Preset_{preset_idx + 1}")
        preset_source = str(preset_source).strip()

        for fname in emb_files:
            print(f"  -> [{top_tag}] embedding: {fname}")

            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_lstm_data(
                df,
                norm=norm_mode,
                test_size=TEST_SIZE,
                seed=SEED,
            )

            tr_loader = DataLoader(
                train_list,
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_sequences,
            )
            te_loader = DataLoader(
                test_list,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_sequences,
            )

            input_dim = int(df.filter(like="feat_").shape[1])

            model = FlexibleLSTM(
                input_dim=input_dim,
                hidden_dim=None,
                num_layers=num_layers,
                layer_sizes=(layer_sizes if layer_sizes else None),
                bidirectional=bidirectional,
                dropout=dropout,
                dropout_layers=dropout_layers,
                norm_layers=norm_layers,
                fc_hidden=fc_hidden,
                activation=(activation if fc_hidden is not None else "relu"),
                pooling=pooling,
                norm_post_lstm=norm_post_lstm,
                native_mode=native_mode,
                num_classes=3,
            ).to(DEVICE)

            id_counter += 1
            run_id = id_counter

            history, best_state = run_training_lstm(
                model,
                tr_loader,
                te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM,
            )

            config = {
                "input_dim": input_dim,
                "num_layers": num_layers,
                "layer_sizes": layer_sizes,
                "bidirectional": bidirectional,
                "norm": norm_mode,
                "dropout": dropout,
                "dropout_layers": dropout_layers,
                "norm_layers": norm_layers,
                "pooling": pooling,
                "fc_hidden": fc_hidden,
                "activation": activation,
                "norm_post_lstm": norm_post_lstm,
                "native_mode": native_mode,
                "batch_size": batch_size,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
            }

            cw_to_store = str(CLASS_WEIGHTS) if (USE_CLASS_WEIGHTS and CLASS_WEIGHTS is not None) else history.get("class_weights", None)

            all_results.append({
                "id": run_id,
                "timestamp": timestamp,
                "target_embedding": fname,
                "top_variant": top_tag,
                "source_params": preset_source,
                "model": "LSTM",
                "seed": SEED,
                **config,
                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                model_path = os.path.join(save_dir, f"lstm{run_id}_v5_{top_tag}_{fname.replace('.csv','')}.pth")
                save_model(model, model_path, config, run_id)

        df_results = pd.DataFrame(all_results)
        if os.path.exists(results_csv):
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)
        else:
            df_results.to_csv(results_csv, index=False)

print("\n=== ENTRENAMIENTOS LSTM V5 COMPLETADOS (Top-K Steps) ===")


## V5: Prueba rápida con embeddings Low-K Steps (LSTM)

Esta celda ejecuta una prueba de control usando los embeddings **Low-K** (peores steps). Se reutilizan los mismos presets que en Top-K para entrenar y comparar de forma directa, esperando que el rendimiento sea ligeramente peor. Los resultados se guardan en CSV por variante (low30/low50/low70) para poder contrastarlos con los de Top-K.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.utils.get_max_id import get_max_id
from src.lstm.prepare_data import prepare_lstm_data, collate_sequences
from src.lstm.model import FlexibleLSTM
from src.lstm.train import run_training_lstm
from src.utils.find_best import topK_from_each_topx_csv

topK_from_each_topx_csv(
    input_dir="results/LSTM/",
    output_dir="presets_configs/LSTM/preset_top_K",
    model_name="LSTM",
    top_k=5,
    dedup=True
)


DATA_DIR = "Gait_Embeddings_LowSteps"

RESULTS_DIR = "results/LSTM"
os.makedirs(RESULTS_DIR, exist_ok=True)

VARIANTS = [
    ("30%", "presets_configs/LSTM/preset_top_K/lstm_results_v5_top30%_top5.csv", os.path.join(RESULTS_DIR, "lstm_results_v5_low30%.csv")),
    ("50%", "presets_configs/LSTM/preset_top_K/lstm_results_v5_top50%_top5.csv", os.path.join(RESULTS_DIR, "lstm_results_v5_low50%.csv")),
    ("70%", "presets_configs/LSTM/preset_top_K/lstm_results_v5_top70%_top5.csv", os.path.join(RESULTS_DIR, "lstm_results_v5_low70%.csv")),
]

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")


def parse_list(x):
    if pd.isna(x):
        return []
    try:
        out = ast.literal_eval(str(x))
        return out if isinstance(out, list) else []
    except Exception:
        return []


def parse_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in ["true", "1", "yes"]


def safe_int(x, default=None):
    if x is None or pd.isna(x):
        return default
    try:
        return int(float(x))
    except Exception:
        return default


def safe_float(x, default=None):
    if x is None or pd.isna(x):
        return default
    try:
        return float(x)
    except Exception:
        return default


for sub_name, presets_csv, results_csv in VARIANTS:
    sub_dir = os.path.join(DATA_DIR, sub_name)

    if not os.path.isdir(sub_dir):
        print(f" No existe: {sub_dir} (skip)")
        continue
    if not os.path.exists(presets_csv):
        print(f" No existe: {presets_csv} (skip)")
        continue

    presets = pd.read_csv(presets_csv, low_memory=False)
    emb_files = sorted([f for f in os.listdir(sub_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f" No hay embeddings en: {sub_dir} (skip)")
        continue

    id_counter = get_max_id([results_csv])

    print(f"\n=== LOW {sub_name} | presets={len(presets)} | embeddings={len(emb_files)} ===")
    print(f"-> results: {results_csv}\n")

    for preset_idx, row in presets.iterrows():
        print(f"--- {sub_name} preset {preset_idx+1}/{len(presets)} (best_f1={row.get('best_f1', float('nan'))}) ---")
        all_results = []

        layer_sizes = parse_list(row.get("layer_sizes"))
        dropout_layers = parse_list(row.get("dropout_layers"))
        norm_layers = parse_list(row.get("norm_layers"))

        num_layers = safe_int(row.get("num_layers"), default=(len(layer_sizes) if layer_sizes else 1))
        if layer_sizes and num_layers != len(layer_sizes):
            num_layers = len(layer_sizes)

        bidirectional = parse_bool(row.get("bidirectional", False))
        norm_mode = str(row.get("norm", "L2")).strip()
        dropout = safe_float(row.get("dropout", 0.0), default=0.0)

        pooling = str(row.get("pooling", "last")).strip().lower()
        fc_hidden = safe_int(row.get("fc_hidden"), default=None)

        activation = row.get("activation", None)
        if fc_hidden is None:
            activation = None
        else:
            activation = "relu" if (activation is None or pd.isna(activation)) else str(activation).strip().lower()

        norm_post_lstm = parse_bool(row.get("norm_post_lstm", False))
        native_mode = parse_bool(row.get("native_mode", False))

        batch_size = safe_int(row.get("batch_size"), default=64)
        optimizer_name = str(row.get("optimizer", "AdamW")).strip()
        lr = safe_float(row.get("learning_rate", row.get("lr", 1e-4)), default=1e-4)
        weight_decay = safe_float(row.get("weight_decay", 0.0), default=0.0)
        momentum_sgd = safe_float(row.get("momentum_sgd", 0.0), default=0.0) if optimizer_name.upper() == "SGD" else None

        preset_source = row.get("source_params", f"Preset_{preset_idx + 1}")
        preset_source = str(preset_source).strip()

        for fname in emb_files:
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_lstm_data(
                df,
                norm=norm_mode,
                test_size=TEST_SIZE,
                seed=SEED
            )

            tr_loader = DataLoader(
                train_list,
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_sequences
            )
            te_loader = DataLoader(
                test_list,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_sequences
            )

            input_dim = int(df.filter(like="feat_").shape[1])

            model = FlexibleLSTM(
                input_dim=input_dim,
                hidden_dim=None,
                num_layers=num_layers,
                layer_sizes=(layer_sizes if layer_sizes else None),
                bidirectional=bidirectional,
                dropout=dropout,
                dropout_layers=dropout_layers,
                norm_layers=norm_layers,
                pooling=pooling,
                fc_hidden=fc_hidden,
                activation=(activation if fc_hidden is not None else "relu"),
                norm_post_lstm=norm_post_lstm,
                native_mode=native_mode,
                num_classes=3
            ).to(DEVICE)

            id_counter += 1
            run_id = id_counter

            history, _ = run_training_lstm(
                model,
                tr_loader,
                te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM
            )

            config = {
                "input_dim": input_dim,
                "num_layers": num_layers,
                "layer_sizes": layer_sizes,
                "bidirectional": bidirectional,
                "norm": norm_mode,
                "dropout": dropout,
                "dropout_layers": dropout_layers,
                "norm_layers": norm_layers,
                "pooling": pooling,
                "fc_hidden": fc_hidden,
                "activation": activation,
                "norm_post_lstm": norm_post_lstm,
                "native_mode": native_mode,
                "batch_size": batch_size,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
            }

            all_results.append({
                "id": run_id,
                "timestamp": timestamp,
                "target_embedding": fname,
                "top_variant": sub_name,
                "source_params": preset_source,
                "model": "LSTM",
                "seed": SEED,
                **config,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

        df_results = pd.DataFrame(all_results)
        if os.path.exists(results_csv):
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)
        else:
            df_results.to_csv(results_csv, index=False)

print("\n=== DONE: LSTM v5 low 30/50/70 ===")


## Chequeo de configuración de LSTM

In [ ]:
from src.utils.check_model import check_model_config

model_path = "saved_models/LSTM/v2/lstm233_v2_gln_phase1.pth"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
check_model_config(model_path, device)

## Matriz de Confusión - Curva ROC - Curva PR (Precision Recall)

En el siguiente código se carga un checkpoint de un modelo LSTM/BiLSTM previamente entrenado junto con su configuración guardada, y se evalúa sobre el conjunto de prueba. Primero se recupera el checkpoint (config y state_dict) con torch.load (map_location al DEVICE) y, de forma defensiva, se infiere la arquitectura (layer_sizes, num_layers, bidirectional, dropout, pooling, capas de norm/dropout, native/modular, etc.) a partir del campo config o del propio checkpoint. A continuación se cargan los datos de embeddings por video_ID y se preparan las secuencias exactamente como en entrenamiento (prepare_lstm_data): split estratificado por video, normalización (MinMax o L2) ajustada solo sobre train, conversión a tensores y creación de listas (tensor_seq, label). El DataLoader usa collate_sequences para producir (padded, lengths, labels) y permitir pack_padded_sequence en la LSTM; el modelo FlexibleLSTM se instancia con los parámetros deducidos y se le cargan los pesos con model.load_state_dict(state_dict). Finalmente, el script evalúa el rendimiento usando funciones específicas para secuencias (evaluate_confusion_matrix_lstm y evaluate_multiclass_roc_lstm) que manejan padding/máscaras y las variantes de pooling (last/mean/max/attention), mostrando la matriz de confusión, curvas ROC one‑vs‑rest y métricas como accuracy y F1‑macro; también imprime aciertos (diagonal) y errores (fuera de diagonal) para un análisis de errores detallado.

In [ ]:
import os
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import ast 

from src.lstm.model import FlexibleLSTM
from src.lstm.prepare_data import prepare_lstm_data, collate_sequences
from src.utils.evaluate_model import evaluate_model

# 1. Configuración básica
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = "Gait_Embeddings_good"
MODEL_PATH = "saved_models/LSTM/v2/lstm233_v2_gln_phase1.pth"
TEST_SIZE = 0.1
SEED = 42

# 2. Cargar checkpoint y extraer config / pesos
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
config = checkpoint.get('config', {}) or {}
state_dict = checkpoint.get('model_state', checkpoint.get('model_state_dict', {}) or checkpoint)

# 3. Determinar nombre del CSV de embedding
try:
    EMBEDDING_FILE = os.path.basename(MODEL_PATH).split("_", 2)[2].replace(".pth", ".csv")
    print(f"Embedding file: {EMBEDDING_FILE}")
except Exception:
    print("No se pudo determinar el nombre del CSV de embedding.")
    raise

# 4. Construir parámetros de la LSTM desde config (simplificado)
try:
    layer_sizes_str = config['layer_sizes']
    layer_sizes = ast.literal_eval(layer_sizes_str) if isinstance(layer_sizes_str, str) else layer_sizes_str
except (KeyError, SyntaxError, ValueError) as e:
    raise ValueError(f"No se pudo encontrar o parsear 'layer_sizes' en la configuración del modelo: {e}")

num_layers = int(config.get('num_layers', len(layer_sizes)))
bidirectional = bool(config.get('bidirectional', config.get('bidir', False)))
dropout = float(config.get('dropout', config.get('dropout_rate', 0.0)))

def parse_layer_list(raw_list):
    if isinstance(raw_list, str):
        try: return ast.literal_eval(raw_list)
        except (ValueError, SyntaxError): return []
    return list(raw_list) if isinstance(raw_list, (list, tuple)) else []

dropout_layers = parse_layer_list(config.get('dropout_layers', []))
norm_layers = parse_layer_list(config.get('norm_layers', []))

fc_hidden = config.get('fc_hidden')
pooling = config.get('pooling', 'last')
norm_post_lstm = bool(config.get('norm_post_lstm', False))
activation = config.get('activation', 'relu')
native_mode = bool(config.get('native_mode', False))

# 5. Cargar los datos y preparar secuencias
df = pd.read_csv(os.path.join(DATA_DIR, EMBEDDING_FILE))
_, test_list, _, _ = prepare_lstm_data(df, norm=config.get('norm', 'minmax'), test_size=TEST_SIZE, seed=SEED)
input_dim = df.filter(like='feat_').shape[1]

# 6. Crear DataLoader de test
batch_size = int(config.get('batch_size', 64))
te_loader = DataLoader(test_list, batch_size=batch_size, shuffle=False, collate_fn=collate_sequences)

# 7. Instanciar el modelo LSTM
model = FlexibleLSTM(
    input_dim=input_dim,
    layer_sizes=layer_sizes,
    num_layers=num_layers,
    bidirectional=bidirectional,
    dropout=dropout,
    dropout_layers=dropout_layers,
    norm_layers=norm_layers,
    fc_hidden=fc_hidden,
    num_classes=3,
    activation=activation if activation else 'relu',
    pooling=pooling,
    norm_post_lstm=norm_post_lstm,
    native_mode=native_mode
).to(DEVICE)

# 8. Cargar pesos y pasar a eval
model.load_state_dict(state_dict)
model.eval()

# 9. Evaluación Universal
class_names = ['derecha', 'centro', 'izquierda']
evaluate_model(model, te_loader, DEVICE, class_names, model_type='lstm')


-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
# Modelo TCN - Temporal Convolutional Network

## Preparación de datos, split y bucle de entrenamiento (TCN)

En TCN se mantiene el mismo esquema de preparación que en LSTM: primero se reconstruyen las **secuencias por `video_ID`** (matriz `T×D` con frames y features), y el **split train/test** se realiza de forma **estratificada a nivel de vídeo** para evitar fugas de información entre frames del mismo clip. La normalización (MinMax o L2) se ajusta **solo con los frames del train** y luego se aplica a train y test. Como las secuencias tienen longitudes variables, el `collate_sequences` aplica **padding** para formar tensores batch y conserva las longitudes originales. El bucle `run_training_tcn` entrena con AMP, permite activar `class_weights`, `label_smoothing` y `clip_grad_norm`, y usa el mismo criterio de **checkpoint + early stopping híbrido** para guardar el mejor estado según macro-F1 una vez la loss ha bajado de un umbral, evitando entrenamientos inestables o estancados.


## Optimización de hiperparámetros con Optuna (TCN)


El Optuna de la TCN sigue el mismo patrón que en LSTM: para cada embedding se optimiza una configuración de hiperparámetros maximizando la **macro-F1 media en validación** mediante **Stratified K-Fold por `video_ID`** (evitando fugas entre frames del mismo vídeo) y con normalización ajustada solo en train de cada fold. El espacio de búsqueda incluye la arquitectura temporal (nº de bloques, `channels`, `dilations`, `kernel_size`, `convs_per_block`), y añade dos mecanismos por bloque: **LayerNorm al final del bloque** (`ln_block_end`) y **dropout por bloque** (flags + probabilidad global), además de pooling y cabeza FC. Para acelerar y hacer más eficiente la búsqueda se usa **TPE + Hyperband** (pruning de trials poco prometedores) y un **early stopping** interno por fold; al final, se exporta un CSV “limpio” reconstruyendo listas reales (channels/dilations/LN/dropout) y un resumen global con los mejores parámetros por embedding.


In [ ]:
from src.tcn.optuna import optimize_embeddings

best_df = optimize_embeddings()
best_df.head()

## V1: Entrenamiento con las mejores configuraciones de Optuna (TCN)

En esta fase (V1) se reutilizan las **mejores configuraciones obtenidas con Optuna** para la TCN y se aplican sistemáticamente a **todos los embeddings objetivo**. Para cada configuración óptima (una por embedding fuente), se reconstruyen las listas estructurales del modelo (canales, dilataciones, LayerNorm y dropout por bloque), se realiza el **split estratificado por `video_ID` (90/10)** con la misma preparación que en LSTM/MLP y se entrena la TCN guardando en un CSV las métricas finales y las mejores métricas alcanzadas. Opcionalmente, se guardan los IDs del split para reproducibilidad y, si el F1 supera un umbral, se almacena el checkpoint del modelo.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.tcn.model import FlexibleTCN
from src.tcn.prepare_data import prepare_tcn_data, collate_sequences
from src.tcn.train import run_training_tcn
from src.utils.get_max_id import get_max_id
from src.utils.save_methods import save_model, save_train_test_ids


DATA_DIR = "Gait_Embeddings_good"
BEST_PARAMS_CSV = "results/TCN/Optuna/best_params_TCN.csv"
RESULTS_CSV = "results/TCN/tcn_results_v1_DEFINITIVO.csv"
SAVE_DIR = "saved_models/TCN"
SPLITS_DIR = "results/TCN/splits"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 5000
F1_THRESHOLD = 0.55
TEST_SIZE = 0.1
SEED = 42
SAVE_IDS = False

os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

best_params_df = pd.read_csv(BEST_PARAMS_CSV, low_memory=False)
embedding_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])

def to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in ["none", "nan"]:
            return []
        return ast.literal_eval(s)
    return list(x)

current_id = get_max_id([RESULTS_CSV])

for _, row in best_params_df.iterrows():
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    results = []

    source_embedding = str(row["embedding"]).strip()
    print(f"\n=== Aplicando config óptima de {source_embedding} ===")

    channels = to_list(row.get("channels"))
    dilations = to_list(row.get("dilations"))
    ln_block_end = to_list(row.get("ln_block_end"))
    dropout_list = to_list(row.get("dropout"))

    if not channels:
        raise ValueError(f"Config inválida: 'channels' vacío en {source_embedding}")

    num_blocks = int(row.get("num_blocks")) if pd.notna(row.get("num_blocks")) else len(channels)
    if num_blocks != len(channels):
        num_blocks = len(channels)

    kernel_size = int(row["kernel_size"])
    convs_per_block = int(row["convs_per_block"])
    activation = str(row["activation"]).strip()

    use_weight_norm = bool(row.get("use_weight_norm", False))
    wn_on_skip = bool(row.get("wn_on_skip", False))
    pooling = str(row["pooling"]).strip()

    fc_hidden_val = row.get("fc_hidden")
    fc_hidden = None if pd.isna(fc_hidden_val) else int(fc_hidden_val)

    batch_size = int(row["batch_size"])
    optimizer_nm = str(row["optimizer"]).strip()
    lr = float(row["lr"])
    weight_decay = float(row.get("weight_decay", 0.0))
    norm = str(row.get("norm", "L2")).strip()

    momentum_sgd = 0.0
    if optimizer_nm.lower() == "sgd":
        momentum_sgd = float(row.get("momentum_sgd", 0.0)) if pd.notna(row.get("momentum_sgd", 0.0)) else 0.0

    for fname in embedding_files:
        print(f"\n--- Entrenando TCN v1 para target = {fname} ---")

        df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
        train_list, test_list, train_ids, test_ids = prepare_tcn_data(
            df, norm=norm, test_size=TEST_SIZE, seed=SEED
        )

        if SAVE_IDS:
            save_train_test_ids(
                fname,
                train_ids,
                test_ids,
                SEED,
                output_dir=os.path.join(SPLITS_DIR, f"seed{SEED}")
            )

        train_loader = DataLoader(
            train_list, batch_size=batch_size, shuffle=True, collate_fn=collate_sequences
        )
        test_loader = DataLoader(
            test_list, batch_size=batch_size, shuffle=False, collate_fn=collate_sequences
        )

        input_dim = int(df.filter(like="feat_").shape[1])

        model = FlexibleTCN(
            input_dim=input_dim,
            channels=channels,
            kernel_size=kernel_size,
            dilations=dilations,
            convs_per_block=convs_per_block,
            dropout=dropout_list,
            activation=activation,
            use_weight_norm=use_weight_norm,
            wn_on_skip=wn_on_skip,
            ln_block_end=ln_block_end,
            pooling=pooling,
            fc_hidden=fc_hidden,
            num_classes=3,
        ).to(DEVICE)

        current_id += 1

        history, best_state = run_training_tcn(
            model=model,
            train_loader=train_loader,
            test_loader=test_loader,
            epochs=EPOCHS,
            lr=lr,
            weight_decay=weight_decay,
            optimizer_name=optimizer_nm,
            momentum_sgd=momentum_sgd,
        )

        cfg = {
            "channels": channels,
            "dilations": dilations,
            "ln_block_end": ln_block_end,
            "dropout": dropout_list,
            "num_blocks": num_blocks,
            "kernel_size": kernel_size,
            "convs_per_block": convs_per_block,
            "activation": activation,
            "use_weight_norm": use_weight_norm,
            "wn_on_skip": wn_on_skip,
            "pooling": pooling,
            "fc_hidden": fc_hidden,
            "batch_size": batch_size,
            "optimizer": optimizer_nm,
            "momentum_sgd": momentum_sgd if optimizer_nm.lower() == "sgd" else None,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "norm": norm,
            "input_dim": input_dim,
        }

        results.append({
            "id": current_id,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": source_embedding,
            "model": "TCN",
            "seed": SEED,
            **cfg,
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": int(history["best_epoch"]),
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "epochs_trained": int(history["epochs_trained"]),
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        })

        if history["best_f1"] >= F1_THRESHOLD:
            print(f"  -> Guardando modelo (best F1={history['best_f1']:.4f})")
            model.load_state_dict(best_state)
            ckpt_name = f"tcn{current_id}_v1_{fname.replace('.csv','')}.pth"
            ckpt_path = os.path.join(SAVE_DIR, ckpt_name)
            save_model(model, ckpt_path, cfg, current_id)

    df_results = pd.DataFrame(results)
    if os.path.exists(RESULTS_CSV):
        with open(RESULTS_CSV, "a", newline="") as f:
            f.write("\n")
        df_results.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_results.to_csv(RESULTS_CSV, index=False)

    print(f"\nResultados para {source_embedding} guardados en {RESULTS_CSV}")

print("\nEntrenamientos TCN v1 finalizados 🟩")


## V2: Entrenamientos a partir de configuraciones personalizadas (TCN)
En V2 al igual que en la MLP y la LSTM, se entrena la TCN usando un conjunto de **presets personalizados** (arquitectura + optimización) definidos en un CSV. Para cada preset se reconstruyen las listas por bloque (canales, dilataciones, `ln_block_end` y `dropout`), se aplica el **mismo split estratificado por `video_ID` (90/10)** y se entrena el modelo contra todos los embeddings objetivo, registrando métricas y configuración en `tcn_results_v2_*.csv`. Si el mejor F1 del entrenamiento supera un umbral, se guarda también el checkpoint del modelo (y opcionalmente los IDs del split para reproducibilidad).


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.tcn.model import FlexibleTCN
from src.tcn.prepare_data import prepare_tcn_data, collate_sequences
from src.tcn.train import run_training_tcn
from src.utils.get_max_id import get_max_id
from src.utils.save_methods import save_model, save_train_test_ids


DATA_DIR = "Gait_Embeddings_good"
PRESETS_CSV = "presets_configs/TCN/presets_tcn.csv"
RESULTS_CSV = "results/TCN/tcn_results_v2.csv"
SAVE_DIR = "saved_models/TCN/v2"

TEST_SIZE = 0.1
SEED = 42
EPOCHS = 5000
F1_THRESHOLD = 0.55

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_IDS = False
SPLITS_DIR = os.path.join("results/TCN/splits", f"seed{SEED}")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

presets = pd.read_csv(PRESETS_CSV, low_memory=False)
emb_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]

id_counter = get_max_id(["results/TCN/tcn_results_v1_DEFINITIVO.csv", RESULTS_CSV])
is_first_run = not os.path.exists(RESULTS_CSV)

RESULT_COLUMNS = [
    "id", "timestamp", "target_embedding", "source_params", "model", "seed",
    "channels", "dilations", "ln_block_end", "dropout",
    "num_blocks", "kernel_size", "convs_per_block", "activation",
    "use_weight_norm", "wn_on_skip", "pooling", "fc_hidden",
    "batch_size", "optimizer", "momentum_sgd", "learning_rate", "weight_decay", "norm",
    "input_dim",
    "train_loss", "test_loss", "accuracy", "f1_macro", "f1_per_class",
    "best_accuracy", "best_f1", "best_f1_per_class", "best_epoch",
    "train_loss_best_state", "test_loss_best_state",
    "epochs_trained", "training", "duration_sec",
]

def to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in ["none", "nan"]:
            return []
        try:
            return ast.literal_eval(s)
        except Exception:
            return []
    try:
        return list(x)
    except Exception:
        return []

def to_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in ["true", "1", "yes"]

for i, r in presets.iterrows():
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    rows = []

    channels = to_list(r.get("channels"))
    dilations = to_list(r.get("dilations"))
    dropout_list = to_list(r.get("dropout"))
    ln_block_end = [to_bool(v) for v in to_list(r.get("ln_block_end"))]

    if not channels:
        print(f" Preset {i+1}: channels vacío -> se salta")
        continue

    num_blocks = len(channels)

    if len(dilations) != num_blocks:
        if len(dilations) == 0:
            dilations = [1] * num_blocks
        else:
            dilations = (dilations + [dilations[-1]] * num_blocks)[:num_blocks]

    if len(ln_block_end) != num_blocks:
        if len(ln_block_end) == 0:
            ln_block_end = [False] * num_blocks
        else:
            ln_block_end = (ln_block_end + [ln_block_end[-1]] * num_blocks)[:num_blocks]

    if len(dropout_list) != num_blocks:
        if len(dropout_list) == 0:
            dropout_list = [0.0] * num_blocks
        else:
            dropout_list = (dropout_list + [dropout_list[-1]] * num_blocks)[:num_blocks]
    dropout_list = [float(p) if p is not None else 0.0 for p in dropout_list]

    kernel_size = int(r["kernel_size"])
    convs_per_blk = int(r["convs_per_block"])
    activation = str(r["activation"]).strip()

    use_wn = to_bool(r.get("use_weight_norm", False))
    wn_on_skip = to_bool(r.get("wn_on_skip", False))
    pooling = str(r["pooling"]).strip()

    fc_val = r.get("fc_hidden", None)
    fc_hidden = None if pd.isna(fc_val) or str(fc_val).strip() == "" else int(float(fc_val))

    batch_size = int(r["batch_size"])
    optimizer_nm = str(r["optimizer"]).strip()
    lr = float(r["lr"]) if "lr" in presets.columns else float(r["learning_rate"])
    weight_decay = float(r.get("weight_decay", 0.0))
    norm = str(r.get("norm", "L2")).strip()

    momentum_for_train = 0.0
    momentum_for_cfg = None
    if optimizer_nm.upper() == "SGD":
        momentum_for_train = float(r.get("momentum_sgd", 0.0)) if pd.notna(r.get("momentum_sgd", 0.0)) else 0.0
        momentum_for_cfg = momentum_for_train

    print(
        f"\n=== PRESET {i+1}/{len(presets)} | blocks={num_blocks} | "
        f"k={kernel_size}x{convs_per_blk} | pool={pooling} | opt={optimizer_nm} | lr={lr} | wd={weight_decay} ==="
    )

    for fname in emb_files:
        print(f"\n--- Entrenando TCN v2 para target_embedding = {fname} ---")

        df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)

        train_list, test_list, train_ids, test_ids = prepare_tcn_data(
            df, norm=norm, test_size=TEST_SIZE, seed=SEED
        )

        if SAVE_IDS:
            save_train_test_ids(fname, train_ids, test_ids, SEED, output_dir=SPLITS_DIR)

        train_loader = DataLoader(
            train_list, batch_size=batch_size, shuffle=True, collate_fn=collate_sequences
        )
        test_loader = DataLoader(
            test_list, batch_size=batch_size, shuffle=False, collate_fn=collate_sequences
        )

        input_dim = int(df.filter(like="feat_").shape[1])

        config = {
            "input_dim": input_dim,
            "num_blocks": num_blocks,
            "channels": channels,
            "dilations": dilations,
            "kernel_size": kernel_size,
            "convs_per_block": convs_per_blk,
            "activation": activation,
            "use_weight_norm": use_wn,
            "wn_on_skip": wn_on_skip,
            "dropout": dropout_list,
            "ln_block_end": ln_block_end,
            "pooling": pooling,
            "fc_hidden": fc_hidden,
            "batch_size": batch_size,
            "optimizer": optimizer_nm,
            "momentum_sgd": momentum_for_cfg,
            "learining_rate": lr,
            "weight_decay": weight_decay,
            "norm": norm,
        }

        model = FlexibleTCN(
            input_dim=input_dim,
            channels=channels,
            kernel_size=kernel_size,
            dilations=dilations,
            convs_per_block=convs_per_blk,
            dropout=dropout_list,
            activation=activation,
            use_weight_norm=use_wn,
            wn_on_skip=wn_on_skip,
            ln_block_end=ln_block_end,
            pooling=pooling,
            fc_hidden=fc_hidden,
            num_classes=3,
        ).to(DEVICE)

        id_counter += 1

        history, best_state = run_training_tcn(
            model=model,
            train_loader=train_loader,
            test_loader=test_loader,
            epochs=EPOCHS,
            lr=lr,
            weight_decay=weight_decay,
            optimizer_name=optimizer_nm,
            momentum_sgd=momentum_for_train,
        )

        rows.append({
            "id": id_counter,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": f"Preset_v2_{i+1}",
            "model": "TCN",
            "seed": SEED,
            **config,
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
            "best_accuracy": round(history["best_acc"], 5) if history.get("best_acc", None) is not None else None,
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": int(history["best_epoch"]),
            "train_loss_best_state": round(history["train_loss_best_state"], 5) if history.get("train_loss_best_state", None) is not None else None,
            "test_loss_best_state": round(history["test_loss_best_state"], 5) if history.get("test_loss_best_state", None) is not None else None,
            "epochs_trained": int(history["epochs_trained"]),
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        })

        if history["best_f1"] >= F1_THRESHOLD:
            print(f"  -> Guardando modelo (best F1={history['best_f1']:.4f})")
            model.load_state_dict(best_state)
            ckpt_name = f"tcn{id_counter}_v2_{fname.replace('.csv','')}.pth"
            ckpt_path = os.path.join(SAVE_DIR, ckpt_name)
            save_model(model, ckpt_path, config, id_counter)

    if not rows:
        continue

    df_results = pd.DataFrame(rows)

    for c in RESULT_COLUMNS:
        if c not in df_results.columns:
            df_results[c] = None
    df_results = df_results[RESULT_COLUMNS]

    if is_first_run:
        df_results.to_csv(RESULTS_CSV, header=True, index=False)
        is_first_run = False
    else:
        with open(RESULTS_CSV, "a", newline="") as f:
            f.write("\n")
        df_results.to_csv(RESULTS_CSV, mode="a", header=False, index=False)

    print(f"\nResultados para PRESET {i+1} guardados en {RESULTS_CSV}")

print("\nEntrenamientos TCN v2 con presets completados ")


## V3: Re-entrenamiento robusto (TCN)

Siguiendo el mismo pipeline que en las anteriores arquitecturas, en V3 se reutilizan las **mejores configuraciones históricas** (V1–V2) y se vuelven a entrenar aplicando una “receta” de entrenamiento más robusta (class weights, label smoothing y clipping de gradiente). Para mantener trazabilidad y evitar duplicados, cada configuración se **canoniza** (listas por bloque normalizadas y strings en minúsculas) y se transforma en una **firma** que permite: (1) recuperar el `id` original si esa misma config ya existía para un embedding, o (2) asignar un `id` nuevo si es un caso no visto. Luego, para cada preset y cada embedding, se prepara el split estratificado por `video_ID`, se entrena con el `run_training_tcn` actualizado, se registran métricas y parámetros en `tcn_results_v3_*.csv`, y se guarda el checkpoint solo si el mejor F1 supera el umbral.


In [ ]:

import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.tcn.model import FlexibleTCN
from src.tcn.prepare_data import prepare_tcn_data, collate_sequences
from src.tcn.train import run_training_tcn
from src.utils.save_methods import save_model, save_train_test_ids
from src.utils.find_best import find_best_configs 
from src.utils.get_max_id import get_max_id


# ================== Paths / setup ==================
DATA_DIR    = "Gait_Embeddings_good"
RESULTS_CSV = "results/TCN/tcn_results_v3 DEFINITIVO.csv"
SAVE_DIR    = "saved_models/TCN/v3 DEFINITIVO"
HISTORICAL  = [
    "results/TCN/tcn_results_v1_DEFINITIVO.csv",
    "results/TCN/tcn_results_v2 DEFINITIVO.csv"
]

TEST_SIZE    = 0.1
SEED         = 42
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 5000
F1_THRESHOLD = 0.6

SAVE_IDS = False
IDS_DIR  = os.path.join("results/TCN/splits", f"seed{SEED}")

# V3 “recipe”
LABEL_SMOOTH  = 0.03
CLIP_NORM     = 1.0
CLASS_WEIGHTS = [1.0, 3.1864, 1.3333]

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
os.makedirs(IDS_DIR, exist_ok=True)

# ================== Firma de configuración ==================
HP_COLS = [
    "channels","dilations","kernel_size","convs_per_block","activation",
    "use_weight_norm","wn_on_skip","pooling","fc_hidden","num_blocks",
    "batch_size","optimizer","momentum_sgd","lr",
    "weight_decay","norm","dropout","ln_block_end"
]

def _s(x):
    return "" if pd.isna(x) else str(x).strip()

def _to_bool_str(v):
    vv = _s(v).lower()
    return "true" if vv in ("true","1","1.0","yes") else "false"

def _canonical_list_str(v):
    """
    Convierte listas guardadas como string a una representación estable sin espacios.
    Ej: "[64, 128]" -> "[64,128]"
    """
    vv = _s(v)
    if not vv:
        return "[]"
    try:
        return str(ast.literal_eval(vv)).replace(" ", "")
    except Exception:
        return vv.replace(" ", "")

def canonicalize_row(row):
    out = {}
    for k in HP_COLS:
        v = row.get(k, "")
        if k in ("channels","dilations","dropout","ln_block_end"):
            out[k] = _canonical_list_str(v)
        elif k in ("activation","pooling","optimizer","norm"):
            out[k] = _s(v).lower()
        elif k in ("use_weight_norm","wn_on_skip"):
            out[k] = _to_bool_str(v)
        else:
            out[k] = _s(v)
    return out

def signature_from_row(row):
    # tupla ordenada: hash estable sin json
    return tuple(sorted(canonicalize_row(row).items()))

# ================== Cargar histórico y crear LOOKUP ==================
hist_files = [p for p in HISTORICAL if os.path.exists(p)]
if hist_files:
    df_hist = pd.concat([pd.read_csv(p, low_memory=False) for p in hist_files], ignore_index=True)
else:
    df_hist = pd.DataFrame(columns=["id","target_embedding",*HP_COLS])

if "id" not in df_hist.columns:
    df_hist["id"] = pd.Series(dtype=int)

# firma para cada fila histórica
df_hist["__sig__"] = df_hist.apply(signature_from_row, axis=1)

LOOKUP = {
    (r["target_embedding"], r["__sig__"]): int(r["id"])
    for _, r in df_hist.iterrows()
    if "target_embedding" in df_hist.columns and not pd.isna(r.get("id", None))
}

# si ya existe CSV V3, seguimos ids a partir de ahí; si no, a partir del máximo histórico
next_new_id = get_max_id([RESULTS_CSV] + hist_files) + 1

# ================== Cargar BEST CONFIGS (para V3) ==================
BEST_CFG_PATH = "results/TCN/best_configs_tcn_v3.csv"
if os.path.exists(BEST_CFG_PATH):
    presets_df = pd.read_csv(BEST_CFG_PATH, low_memory=False)
else:
    presets_df = None

if presets_df is None or presets_df.empty:
    presets_df = find_best_configs(
        csv_files=hist_files,
        f1_threshold=F1_THRESHOLD,
        model_name="TCN",
        output_file=BEST_CFG_PATH
    )

if presets_df is None or presets_df.empty:
    raise ValueError(" No se encontraron configuraciones para entrenar en V3.")

print(f"🔎 Se entrenarán {len(presets_df)} configuraciones únicas (V3).")

# ================== LOOP PRINCIPAL ==================
emb_list = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])

def to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if s == "" or s.lower() in ["none","nan"]:
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

def to_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in ("true","1","1.0","yes")

for i, p in presets_df.iterrows():
    all_rows = []

    channels      = to_list(p.get("channels"))
    dilations     = to_list(p.get("dilations"))
    dropout_list  = to_list(p.get("dropout"))
    ln_list       = to_list(p.get("ln_block_end"))

    # seguridad: si num_blocks no cuadra, lo forzamos a len(channels)
    num_blocks = int(p.get("num_blocks", len(channels))) if len(channels) > 0 else int(p.get("num_blocks", 0))
    if len(channels) > 0:
        num_blocks = len(channels)

    # normalizar longitudes (por si el CSV está “sucio”)
    if len(dilations) != num_blocks:
        dilations = (dilations + [dilations[-1]] * num_blocks)[:num_blocks] if dilations else [1] * num_blocks
    if len(dropout_list) != num_blocks:
        dropout_list = (dropout_list + [dropout_list[-1]] * num_blocks)[:num_blocks] if dropout_list else [0.0] * num_blocks
    if len(ln_list) != num_blocks:
        ln_list = (ln_list + [ln_list[-1]] * num_blocks)[:num_blocks] if ln_list else [False] * num_blocks

    # tu modelo suele aceptar lista de float (0.0 = sin dropout). Si el tuyo usa None, cambia aquí.
    dropout_for_model = [0.0 if (x is None or float(x) <= 0.0) else float(x) for x in dropout_list]
    ln_block_end = [bool(x) for x in ln_list]

    kernel_size   = int(p["kernel_size"])
    convs_per_blk = int(p["convs_per_block"])
    activation    = _s(p["activation"]).lower()
    use_wn        = to_bool(p.get("use_weight_norm", False))
    wn_on_skip    = to_bool(p.get("wn_on_skip", False))
    pooling       = _s(p.get("pooling", "mean")).lower()

    fc_hidden_raw = p.get("fc_hidden", None)
    fc_hidden     = None if pd.isna(fc_hidden_raw) or str(fc_hidden_raw).strip()=="" else int(float(fc_hidden_raw))

    batch_size    = int(float(p.get("batch_size", 64)))
    optimizer_nm  = _s(p.get("optimizer", "AdamW"))
    lr            = float(p.get("lr", p.get("learning_rate", 1e-4)))
    weight_decay  = float(p.get("weight_decay", 0.0)) if not pd.isna(p.get("weight_decay", 0.0)) else 0.0
    norm          = _s(p.get("norm", "L2"))

    momentum_sgd  = 0.0
    if optimizer_nm.upper() == "SGD":
        m = p.get("momentum_sgd", 0.0)
        momentum_sgd = 0.0 if pd.isna(m) else float(m)

    # firma del preset (para mapear ids)
    p_sig = signature_from_row(p)

    print(f"\n==============================")
    print(f"PRESET V3 {i+1}/{len(presets_df)} | blocks={num_blocks} | act={activation} | pool={pooling}")
    print(f"==============================")

    for fname in emb_list:

        # mantener id si ya existía esa (config, embedding)
        run_id = LOOKUP.get((fname, p_sig))
        if run_id is None:
            run_id = next_new_id
            next_new_id += 1

        df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
        input_dim = int(df.filter(like="feat_").shape[1])

        train_list, test_list, train_ids, test_ids = prepare_tcn_data(
            df, norm=norm, test_size=TEST_SIZE, seed=SEED
        )

        if SAVE_IDS:
            save_train_test_ids(fname, train_ids, test_ids, SEED, output_dir=IDS_DIR)

        tr_loader = DataLoader(train_list, batch_size=batch_size, shuffle=True,  collate_fn=collate_sequences)
        te_loader = DataLoader(test_list,  batch_size=batch_size, shuffle=False, collate_fn=collate_sequences)

        model = FlexibleTCN(
            input_dim=input_dim,
            channels=channels,
            kernel_size=kernel_size,
            dilations=dilations,
            convs_per_block=convs_per_blk,
            dropout=dropout_for_model,
            activation=activation,
            use_weight_norm=use_wn,
            wn_on_skip=wn_on_skip,
            ln_block_end=ln_block_end,
            pooling=pooling,
            fc_hidden=fc_hidden,
            num_classes=3,
        ).to(DEVICE)

        config = {
            "input_dim": input_dim,
            "num_blocks": num_blocks,
            "channels": channels,
            "dilations": dilations,
            "kernel_size": kernel_size,
            "convs_per_block": convs_per_blk,
            "activation": activation,
            "use_weight_norm": use_wn,
            "wn_on_skip": wn_on_skip,
            "dropout": dropout_for_model,      # lo que realmente usa el modelo
            "ln_block_end": ln_block_end,
            "pooling": pooling,
            "fc_hidden": fc_hidden,
            "batch_size": batch_size,
            "optimizer": optimizer_nm,
            "momentum_sgd": (momentum_sgd if optimizer_nm.upper()=="SGD" else None),
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "norm": norm,
        }

        history, best_state = run_training_tcn(
            model=model,
            train_loader=tr_loader,
            test_loader=te_loader,
            epochs=EPOCHS,
            lr=lr,
            weight_decay=weight_decay,
            optimizer_name=optimizer_nm,
            momentum_sgd=momentum_sgd,
            use_class_weights=True,
            class_weights=CLASS_WEIGHTS,
            label_smoothing=LABEL_SMOOTH,
            clip_grad_norm=CLIP_NORM,
        )

        row = {
            "id": run_id,
            "timestamp": pd.Timestamp.now().strftime("%Y%m%d_%H%M%S"),
            "target_embedding": fname,
            "source_params": "best_configs_v3",
            "model": "TCN",
            "seed": SEED,
            **config,

            "label_smoothing": LABEL_SMOOTH,
            "clip_grad_norm": CLIP_NORM,
            "class_weights": str(CLASS_WEIGHTS),

            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],

            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": int(history["best_epoch"]),
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "epochs_trained": int(history["epochs_trained"]),
            "training": "STOPPED" if bool(history["early_stopped"]) else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        }

        all_rows.append(row)

        if history["best_f1"] >= F1_THRESHOLD:
            model.load_state_dict(best_state)
            ckpt_path = os.path.join(SAVE_DIR, f"tcn{run_id}_v3_{fname.replace('.csv','')}.pth")
            save_model(model, ckpt_path, config, run_id)

    # ================== Guardar filas de este preset ==================
    if all_rows:
        df_out = pd.DataFrame(all_rows)

        if not os.path.exists(RESULTS_CSV):
            df_out.to_csv(RESULTS_CSV, index=False)
        else:
            with open(RESULTS_CSV, "a", newline="") as f:
                f.write("\n")
            df_out.to_csv(RESULTS_CSV, mode="a", header=False, index=False)

        # Ordenar por id para tener un CSV consistente
        df_final = pd.read_csv(RESULTS_CSV, low_memory=False).sort_values(by="id", ascending=True)
        df_final.to_csv(RESULTS_CSV, index=False)

        print(f"Resultados añadidos y CSV ordenado por ID → {RESULTS_CSV}")
    else:
        print("No se entrenó ningún modelo para este preset.")


## Comparación de resultados de entrenamiento V1, V2 y V3 (TCN)


*   **`mean_f1` (y `mean_diff`)**: Esta es la métrica más importante para evaluar la mejora general. `mean_f1_v3` es el F1-score *promedio* de todos los entrenamientos de la `v3` para un embedding, mientras que `mean_f1_v1_v2` es el promedio de los históricos. La columna **`mean_diff`** te dice si, en conjunto, tus nuevos entrenamientos son mejores. **Un valor positivo y alto aquí es tu principal objetivo**, ya que indica que el rendimiento promedio ha mejorado de forma consistente.

*   **`max_f1` (y `max_diff`)**: Esta métrica se centra en el rendimiento pico. Compara el mejor F1-score individual que se consiguió en `v3` (`max_f1_v3`) con el mejor récord histórico (`max_f1_v1_v2`). La columna **`max_diff`** te muestra si has batido un nuevo "récord" de rendimiento. Es útil para ver el potencial máximo de una configuración, aunque una mejora aquí podría ser un golpe de suerte si la media no ha subido.

*   **`std_f1` (Desviación Estándar)**: Esta métrica mide la **estabilidad y consistencia** de tus resultados. Un valor bajo es bueno, ya que significa que la mayoría de los entrenamientos para ese embedding obtuvieron un F1-score muy similar. Si **`std_f1_v3` es menor que `std_f1_v1_v2`**, es una excelente noticia, porque indica que tus nuevos entrenamientos no solo son mejores, sino también más fiables y predecibles.

In [ ]:
from src.utils.comparison_results import compare_model_results

MODEL_TYPE = "TCN"
HISTORICAL_FILES = [
    "results/TCN/tcn_results_v1.csv",
    "results/TCN/tcn_results_v2.csv"
]
FINAL_FILE = "results/TCN/tcn_results_v3.csv"
OUTPUT_DIR = "results/Comparisons_V1_V2_V3"

compare_model_results(
    model_type=MODEL_TYPE,
    historical_files=HISTORICAL_FILES,
    final_file_path=FINAL_FILE,
    output_dir=OUTPUT_DIR
)

## V4: Entrenamientos con embeddings combinados (TCN)

En **V4 (TCN)** se repite el mismo esquema que en MLP/LSTM pero sobre los **embeddings combinados** (subcarpetas `concat/mean/reduced`). La idea es aplicar **presets específicos por variante** (para no mezclar “recetas” entre tipos de combinación) y entrenar cada preset contra todos los embeddings de su subcarpeta, guardando (1) un CSV de resultados por variante y (2) checkpoints solo si superan un umbral de F1. En esta fase se fija la “receta operativa” que ya viste que desbloquea el entrenamiento: **class weights** para el desbalance, **clip_grad_norm** para estabilidad/convergencia (gradient clipping es un estándar clásico), y **label_smoothing** como regularizador opcional para reducir sobreconfianza y estabilizar la test loss.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.tcn.model import FlexibleTCN
from src.tcn.prepare_data import prepare_tcn_data, collate_sequences
from src.tcn.train import run_training_tcn
from src.utils.save_methods import save_model
from src.utils.get_max_id import get_max_id

DATA_DIR = "Gait_Embeddings_Combined"
RESULTS_DIR = "results/TCN"
DEFAULT_PRESETS_CSV = "presets_configs/TCN/best_configs_tcn.csv"

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000
THRESHOLD = 0.6

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0

os.makedirs(RESULTS_DIR, exist_ok=True)

VARIANTS = [
    ("concat",  "presets_configs/TCN/presets_tcn_concat.csv",  os.path.join(RESULTS_DIR, "tcn_results_v4_concat.csv"),  "saved_models/TCN/v4_concat"),
    ("mean",    "presets_configs/TCN/presets_tcn_mean.csv",    os.path.join(RESULTS_DIR, "tcn_results_v4_mean.csv"),    "saved_models/TCN/v4_mean"),
    ("reduced", "presets_configs/TCN/presets_tcn_reduced.csv", os.path.join(RESULTS_DIR, "tcn_results_v4_reduced.csv"), "saved_models/TCN/v4_reduced"),
]

for _, _, _, sd in VARIANTS:
    os.makedirs(sd, exist_ok=True)

def parse_list(v, default=None):
    if default is None:
        default = []
    if pd.isna(v):
        return list(default)
    if isinstance(v, list):
        return v
    s = str(v).strip()
    if s == "" or s.lower() in ["none", "nan"]:
        return list(default)
    try:
        out = ast.literal_eval(s)
        return out if isinstance(out, list) else list(default)
    except Exception:
        return list(default)

def parse_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in ["true", "1", "1.0", "yes"]

def ensure_len(x, L, fill):
    x = list(x)
    if len(x) == L:
        return x
    if len(x) == 0:
        return [fill] * L
    if len(x) < L:
        return x + [x[-1]] * (L - len(x))
    return x[:L]

for sub_name, presets_csv, results_csv, save_dir in VARIANTS:
    sub_dir = os.path.join(DATA_DIR, sub_name)
    if not os.path.isdir(sub_dir):
        print(f" No existe: {sub_dir} (skip)")
        continue

    if not os.path.exists(presets_csv):
        if os.path.exists(DEFAULT_PRESETS_CSV):
            presets_csv = DEFAULT_PRESETS_CSV
            print(f" Presets específicos no encontrados. Uso fallback: {DEFAULT_PRESETS_CSV}")
        else:
            print(f" No existe presets: {presets_csv} ni fallback: {DEFAULT_PRESETS_CSV} (skip)")
            continue

    emb_files = sorted([f for f in os.listdir(sub_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f" No hay embeddings en: {sub_dir} (skip)")
        continue

    presets = pd.read_csv(presets_csv, low_memory=False)
    if presets.empty:
        print(f" Presets vacíos: {presets_csv} (skip)")
        continue

    id_counter = get_max_id([results_csv])
    is_first = not os.path.exists(results_csv)

    print(f"\n==============================")
    print(f"📂 V4 / {sub_name} | presets={len(presets)} | embeddings={len(emb_files)}")
    print(f"📝 results -> {results_csv}")
    print(f"💾 models  -> {save_dir}")
    print(f"==============================\n")

    for preset_idx, row in presets.iterrows():
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        all_results = []

        channels = parse_list(row.get("channels", "[]"))
        dilations = parse_list(row.get("dilations", "[]"))
        ln_block_end = parse_list(row.get("ln_block_end", "[]"))
        dropout_list = parse_list(row.get("dropout", "[]"))

        if len(channels) == 0:
            print(f" Preset {preset_idx+1}: channels vacío (skip)")
            continue

        num_blocks = len(channels)
        dilations = ensure_len(dilations, num_blocks, 1)
        ln_block_end = [bool(x) for x in ensure_len(ln_block_end, num_blocks, False)]

        if len(dropout_list) == 0:
            dropout_list = [0.0] * num_blocks
        else:
            dropout_list = ensure_len(dropout_list, num_blocks, 0.0)
        dropout_for_model = [0.0 if (x is None or float(x) <= 0.0) else float(x) for x in dropout_list]

        kernel_size = int(row.get("kernel_size", 3))
        convs_per_block = int(row.get("convs_per_block", 2))
        activation = str(row.get("activation", "relu")).strip().lower()

        use_weight_norm = parse_bool(row.get("use_weight_norm", False))
        wn_on_skip = parse_bool(row.get("wn_on_skip", False))

        pooling = str(row.get("pooling", "mean")).strip().lower()

        fc_hidden_raw = row.get("fc_hidden", None)
        fc_hidden = None if pd.isna(fc_hidden_raw) or str(fc_hidden_raw).strip() == "" else int(float(fc_hidden_raw))

        norm = str(row.get("norm", "L2")).strip()

        batch_size = int(float(row.get("batch_size", 64)))
        optimizer_name = str(row.get("optimizer", "AdamW")).strip()
        lr = float(row.get("learning_rate", row.get("lr", 1e-4)))
        weight_decay = float(row.get("weight_decay", 0.0)) if not pd.isna(row.get("weight_decay", 0.0)) else 0.0

        momentum_sgd = None
        if optimizer_name.upper() == "SGD":
            m = row.get("momentum_sgd", 0.0)
            momentum_sgd = 0.0 if pd.isna(m) else float(m)

        print(f"\n=== [{sub_name}] PRESET {preset_idx+1}/{len(presets)} ===")

        for fname in emb_files:
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_tcn_data(
                df, norm=norm, test_size=TEST_SIZE, seed=SEED
            )

            tr_loader = DataLoader(train_list, batch_size=batch_size, shuffle=True,  collate_fn=collate_sequences)
            te_loader = DataLoader(test_list,  batch_size=batch_size, shuffle=False, collate_fn=collate_sequences)

            input_dim = int(df.filter(like="feat_").shape[1])

            model = FlexibleTCN(
                input_dim=input_dim,
                channels=channels,
                kernel_size=kernel_size,
                dilations=dilations,
                convs_per_block=convs_per_block,
                dropout=dropout_for_model,
                activation=activation,
                use_weight_norm=use_weight_norm,
                wn_on_skip=wn_on_skip,
                ln_block_end=ln_block_end,
                pooling=pooling,
                fc_hidden=fc_hidden,
                num_classes=3,
            ).to(DEVICE)

            id_counter += 1

            history, best_state = run_training_tcn(
                model=model,
                train_loader=tr_loader,
                test_loader=te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=(momentum_sgd if momentum_sgd is not None else 0.0),
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=(CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None),
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM
            )

            config = {
                "input_dim": input_dim,
                "num_blocks": num_blocks,
                "channels": channels,
                "dilations": dilations,
                "dropout": dropout_for_model,
                "ln_block_end": ln_block_end,
                "kernel_size": kernel_size,
                "convs_per_block": convs_per_block,
                "activation": activation,
                "use_weight_norm": use_weight_norm,
                "wn_on_skip": wn_on_skip,
                "pooling": pooling,
                "fc_hidden": fc_hidden,
                "batch_size": batch_size,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
                "norm": norm,
            }

            cw_to_store = history.get("class_weights", None) if (USE_CLASS_WEIGHTS and CLASS_WEIGHTS is None) else (str(CLASS_WEIGHTS) if USE_CLASS_WEIGHTS else None)

            all_results.append({
                "id": id_counter,
                "timestamp": timestamp,
                "target_embedding": fname,
                "combine_variant": sub_name,
                "source_params": f"Preset_{preset_idx+1}",
                "model": "TCN",
                "seed": SEED,
                **config,
                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if bool(history["early_stopped"]) else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                model_path = os.path.join(save_dir, f"tcn{id_counter}_v4_{sub_name}_{fname.replace('.csv','')}.pth")
                save_model(model, model_path, config, id_counter)

        df_block = pd.DataFrame(all_results)
        if is_first:
            df_block.to_csv(results_csv, index=False)
            is_first = False
        else:
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_block.to_csv(results_csv, mode="a", header=False, index=False)

print("\n=== ENTRENAMIENTOS TCN V4 COMPLETADOS (concat/mean/reduced) ===")


## V5: Embeddings con embeddings Top-K Steps (TCN)

Al igual que en los anteriores modelos (MLP y LSTM), en **V5 (TCN)** se evalúa el efecto de recortar la secuencia temporal a los **Top-K steps** (30/50/70) manteniendo el resto del pipeline idéntico: para cada variante (`30%`, `50%`, `70%`) se cargan sus embeddings recortados, se aplican **presets** (arquitectura/optimizador) y se entrena sobre el split estratificado por `video_ID`. Los resultados se guardan en **un CSV por variante** y los modelos se almacenan solo si superan un umbral de **F1 macro**, dejando trazabilidad con el campo `top_variant`.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.tcn.model import FlexibleTCN
from src.tcn.prepare_data import prepare_tcn_data, collate_sequences
from src.tcn.train import run_training_tcn
from src.utils.save_methods import save_model
from src.utils.get_max_id import get_max_id

DATA_DIR = "Gait_Embeddings_TopSteps"
PRESETS_CSV = "presets_configs/TCN/best_configs_tcn.csv"

RESULTS_DIR = "results/TCN"
os.makedirs(RESULTS_DIR, exist_ok=True)

VARIANTS = [
    ("30%", PRESETS_CSV, os.path.join(RESULTS_DIR, "tcn_results_v5_top30%.csv"), "saved_models/TCN/v5_top30%"),
    ("50%", PRESETS_CSV, os.path.join(RESULTS_DIR, "tcn_results_v5_top50%.csv"), "saved_models/TCN/v5_top50%"),
    ("70%", PRESETS_CSV, os.path.join(RESULTS_DIR, "tcn_results_v5_top70%.csv"), "saved_models/TCN/v5_top70%"),
]

for _, _, _, sd in VARIANTS:
    os.makedirs(sd, exist_ok=True)

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000
THRESHOLD = 0.6

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0

def parse_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in ["true", "1", "1.0", "yes"]

def parse_list(v, default=None):
    if default is None:
        default = []
    if pd.isna(v):
        return list(default)
    if isinstance(v, list):
        return v
    s = str(v).strip()
    if s == "" or s.lower() in ["none", "nan"]:
        return list(default)
    try:
        out = ast.literal_eval(s)
        return out if isinstance(out, list) else list(default)
    except Exception:
        return list(default)

def ensure_len(x, L, fill):
    x = list(x)
    if len(x) == L:
        return x
    if len(x) == 0:
        return [fill] * L
    if len(x) < L:
        return x + [x[-1]] * (L - len(x))
    return x[:L]

for top_tag, presets_csv, results_csv, save_dir in VARIANTS:
    sub_dir = os.path.join(DATA_DIR, top_tag)
    if not os.path.isdir(sub_dir):
        print(f" No existe: {sub_dir} (skip)")
        continue
    if not os.path.exists(presets_csv):
        print(f" No existe presets_csv: {presets_csv} (skip)")
        continue

    presets = pd.read_csv(presets_csv, low_memory=False)
    if presets.empty:
        print(f"Presets vacíos: {presets_csv} (skip)")
        continue

    emb_files = sorted([f for f in os.listdir(sub_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f"No hay embeddings en: {sub_dir} (skip)")
        continue

    id_counter = get_max_id([results_csv])
    is_first = not os.path.exists(results_csv)

    print(f"\n==============================")
    print(f"📂 TOP {top_tag} | presets={len(presets)} | embeddings={len(emb_files)}")
    print(f"📝 results -> {results_csv}")
    print(f"💾 models  -> {save_dir}")
    print(f"==============================\n")

    for preset_idx, row in presets.iterrows():
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        all_results = []

        channels = parse_list(row.get("channels", "[]"))
        dilations = parse_list(row.get("dilations", "[]"))
        ln_block_end = parse_list(row.get("ln_block_end", "[]"))
        dropout_list = parse_list(row.get("dropout", "[]"))

        if len(channels) == 0:
            print(f" Preset {preset_idx+1}: channels vacío (skip)")
            continue

        num_blocks = len(channels)
        dilations = ensure_len(dilations, num_blocks, 1)
        ln_block_end = [bool(x) for x in ensure_len(ln_block_end, num_blocks, False)]

        if len(dropout_list) == 0:
            dropout_list = [0.0] * num_blocks
        else:
            dropout_list = ensure_len(dropout_list, num_blocks, 0.0)
        dropout_for_model = [0.0 if (x is None or float(x) <= 0.0) else float(x) for x in dropout_list]

        kernel_size = int(row.get("kernel_size", 3))
        convs_per_block = int(row.get("convs_per_block", 2))
        activation = str(row.get("activation", "relu")).strip().lower()

        use_weight_norm = parse_bool(row.get("use_weight_norm", False))
        wn_on_skip = parse_bool(row.get("wn_on_skip", False))

        pooling = str(row.get("pooling", "mean")).strip().lower()

        fc_hidden_raw = row.get("fc_hidden", None)
        fc_hidden = None if pd.isna(fc_hidden_raw) or str(fc_hidden_raw).strip() == "" else int(float(fc_hidden_raw))

        norm = str(row.get("norm", "L2")).strip()

        batch_size = int(float(row.get("batch_size", 64)))
        optimizer_name = str(row.get("optimizer", "AdamW")).strip()
        lr = float(row.get("learning_rate", row.get("lr", 1e-4)))
        weight_decay = float(row.get("weight_decay", 0.0)) if not pd.isna(row.get("weight_decay", 0.0)) else 0.0

        momentum_sgd = None
        if optimizer_name.upper() == "SGD":
            m = row.get("momentum_sgd", 0.0)
            momentum_sgd = 0.0 if pd.isna(m) else float(m)

        print(f"\n=== [{top_tag}] PRESET {preset_idx+1}/{len(presets)} ===")

        for fname in emb_files:
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_tcn_data(
                df, norm=norm, test_size=TEST_SIZE, seed=SEED
            )

            tr_loader = DataLoader(train_list, batch_size=batch_size, shuffle=True,  collate_fn=collate_sequences)
            te_loader = DataLoader(test_list,  batch_size=batch_size, shuffle=False, collate_fn=collate_sequences)

            input_dim = int(df.filter(like="feat_").shape[1])

            model = FlexibleTCN(
                input_dim=input_dim,
                channels=channels,
                kernel_size=kernel_size,
                dilations=dilations,
                convs_per_block=convs_per_block,
                dropout=dropout_for_model,
                activation=activation,
                use_weight_norm=use_weight_norm,
                wn_on_skip=wn_on_skip,
                ln_block_end=ln_block_end,
                pooling=pooling,
                fc_hidden=fc_hidden,
                num_classes=3
            ).to(DEVICE)

            id_counter += 1

            history, best_state = run_training_tcn(
                model=model,
                train_loader=tr_loader,
                test_loader=te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=(momentum_sgd if momentum_sgd is not None else 0.0),
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=(CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None),
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM
            )

            config = {
                "input_dim": input_dim,
                "num_blocks": num_blocks,
                "channels": channels,
                "dilations": dilations,
                "dropout": dropout_for_model,
                "ln_block_end": ln_block_end,
                "kernel_size": kernel_size,
                "convs_per_block": convs_per_block,
                "activation": activation,
                "use_weight_norm": use_weight_norm,
                "wn_on_skip": wn_on_skip,
                "pooling": pooling,
                "fc_hidden": fc_hidden,
                "norm": norm,
                "batch_size": batch_size,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
            }

            cw_to_store = history.get("class_weights", None) if (USE_CLASS_WEIGHTS and CLASS_WEIGHTS is None) else (str(CLASS_WEIGHTS) if USE_CLASS_WEIGHTS else None)

            all_results.append({
                "id": id_counter,
                "timestamp": timestamp,
                "target_embedding": fname,
                "top_variant": top_tag,
                "source_params": f"Preset_{preset_idx+1}",
                "model": "TCN",
                "seed": SEED,
                **config,
                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if bool(history["early_stopped"]) else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                model_path = os.path.join(save_dir, f"tcn{id_counter}_v5_{top_tag}_{fname.replace('.csv','')}.pth")
                save_model(model, model_path, config, id_counter)

        df_block = pd.DataFrame(all_results)
        if is_first:
            df_block.to_csv(results_csv, index=False)
            is_first = False
        else:
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_block.to_csv(results_csv, mode="a", header=False, index=False)

print("\n=== ENTRENAMIENTOS TCN V5 COMPLETADOS (TOP 30/50/70) ===")


## V5: Prueba rápida con embeddings Low-K Steps (TCN)

En **V5 (TCN) Low-Steps** se repite exactamente el mismo esquema que en Top-K, pero usando embeddings generados con los **steps menos relevantes** (Low 30/50/70). Para cada variante se cargan sus presets (derivados de los Top-K correspondientes), se entrena sobre los embeddings de esa subcarpeta y se guardan métricas/modelos en ficheros separados, manteniendo `top_variant` para trazabilidad (aunque conceptualmente sea “low”).


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.tcn.model import FlexibleTCN
from src.tcn.prepare_data import prepare_tcn_data, collate_sequences
from src.tcn.train import run_training_tcn
from src.utils.save_methods import save_model
from src.utils.get_max_id import get_max_id
from src.utils.find_best import topK_from_each_topx_csv

topK_from_each_topx_csv(
    input_dir="results/TCN/",
    output_dir="presets_configs/TCN/preset_top_K",
    model_name="TCN",
    top_k=5,
    dedup=True
)



DATA_DIR = "Gait_Embeddings_LowSteps"

RESULTS_DIR = "results/TCN"
os.makedirs(RESULTS_DIR, exist_ok=True)

VARIANTS = [
    ("30%", "presets_configs/TCN/preset_top_K/tcn_results_v5_top30%_top5.csv", os.path.join(RESULTS_DIR, "tcn_results_v5_low30%.csv"), "saved_models/TCN/v5_low30%"),
    ("50%", "presets_configs/TCN/preset_top_K/tcn_results_v5_top50%_top5.csv", os.path.join(RESULTS_DIR, "tcn_results_v5_low50%.csv"), "saved_models/TCN/v5_low50%"),
    ("70%", "presets_configs/TCN/preset_top_K/tcn_results_v5_top70%_top5.csv", os.path.join(RESULTS_DIR, "tcn_results_v5_low70%.csv"), "saved_models/TCN/v5_low70%"),
]

for _, _, _, sd in VARIANTS:
    os.makedirs(sd, exist_ok=True)

TEST_SIZE = 0.1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5000
THRESHOLD = 0.60

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH = 0.02
CLIP_NORM = 1.0

def parse_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in ["true", "1", "1.0", "yes"]

def parse_list(v, default=None):
    if default is None:
        default = []
    if pd.isna(v):
        return list(default)
    if isinstance(v, list):
        return v
    s = str(v).strip()
    if s == "" or s.lower() in ["none", "nan"]:
        return list(default)
    try:
        out = ast.literal_eval(s)
        return out if isinstance(out, list) else list(default)
    except Exception:
        return list(default)

def ensure_len(x, L, fill):
    x = list(x)
    if len(x) == L:
        return x
    if len(x) == 0:
        return [fill] * L
    if len(x) < L:
        return x + [x[-1]] * (L - len(x))
    return x[:L]

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

for top_tag, presets_csv, results_csv, save_dir in VARIANTS:
    sub_dir = os.path.join(DATA_DIR, top_tag)

    if not os.path.isdir(sub_dir):
        print(f" Subcarpeta no encontrada: {sub_dir} (skip)")
        continue

    if not os.path.exists(presets_csv):
        print(f" Presets no encontrado: {presets_csv} (skip)")
        continue

    presets = pd.read_csv(presets_csv, low_memory=False)
    if presets.empty:
        print(f" Presets vacíos: {presets_csv} (skip)")
        continue

    emb_files = sorted([f for f in os.listdir(sub_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f" No hay CSVs en: {sub_dir} (skip)")
        continue

    id_counter = get_max_id([results_csv])
    is_first = not os.path.exists(results_csv)

    print(f"\n==============================")
    print(f"📂 LOW {top_tag} | embeddings={len(emb_files)} | presets={len(presets)}")
    print(f"📝 Resultados -> {results_csv}")
    print(f"💾 Modelos -> {save_dir}")
    print(f"==============================\n")

    for preset_idx, row in presets.iterrows():
        print(f"\n=== [LOW {top_tag}] PRESET {preset_idx+1}/{len(presets)} ===")
        all_results = []

        channels = parse_list(row.get("channels", "[]"))
        dilations = parse_list(row.get("dilations", "[]"))
        ln_block_end = parse_list(row.get("ln_block_end", "[]"))
        dropout_list = parse_list(row.get("dropout", "[]"))

        if len(channels) == 0:
            print(f" Preset {preset_idx+1}: channels vacío (skip)")
            continue

        num_blocks = len(channels)
        dilations = ensure_len(dilations, num_blocks, 1)
        ln_block_end = [bool(x) for x in ensure_len(ln_block_end, num_blocks, False)]

        if len(dropout_list) == 0:
            dropout_list = [0.0] * num_blocks
        else:
            dropout_list = ensure_len(dropout_list, num_blocks, 0.0)
        dropout_for_model = [0.0 if (x is None or float(x) <= 0.0) else float(x) for x in dropout_list]

        kernel_size = int(row.get("kernel_size", 3))
        convs_per_block = int(row.get("convs_per_block", 2))
        activation = str(row.get("activation", "relu")).strip().lower()

        use_weight_norm = parse_bool(row.get("use_weight_norm", False))
        wn_on_skip = parse_bool(row.get("wn_on_skip", False))

        pooling = str(row.get("pooling", "last")).strip().lower()

        fc_raw = row.get("fc_hidden", None)
        fc_hidden = None if pd.isna(fc_raw) or str(fc_raw).strip() == "" else int(float(fc_raw))

        norm = str(row.get("norm", "L2")).strip()

        batch_size = int(float(row.get("batch_size", 64)))
        optimizer_name = str(row.get("optimizer", "AdamW")).strip()
        lr = float(row.get("learning_rate", row.get("lr", 1e-4)))
        weight_decay = float(row.get("weight_decay", 0.0)) if not pd.isna(row.get("weight_decay", 0.0)) else 0.0

        momentum_sgd = None
        if optimizer_name.upper() == "SGD":
            m = row.get("momentum_sgd", 0.0)
            momentum_sgd = 0.0 if pd.isna(m) else float(m)

        preset_source = str(row.get("source_params", f"Preset_{preset_idx+1}")).strip()

        for fname in emb_files:
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_tcn_data(
                df, norm=norm, test_size=TEST_SIZE, seed=SEED
            )

            tr_loader = DataLoader(train_list, batch_size=batch_size, shuffle=True,  collate_fn=collate_sequences)
            te_loader = DataLoader(test_list,  batch_size=batch_size, shuffle=False, collate_fn=collate_sequences)

            input_dim = int(df.filter(like="feat_").shape[1])

            model = FlexibleTCN(
                input_dim=input_dim,
                channels=channels,
                kernel_size=kernel_size,
                dilations=dilations,
                convs_per_block=convs_per_block,
                dropout=dropout_for_model,
                activation=activation,
                use_weight_norm=use_weight_norm,
                wn_on_skip=wn_on_skip,
                ln_block_end=ln_block_end,
                pooling=pooling,
                fc_hidden=fc_hidden,
                num_classes=3
            ).to(DEVICE)

            id_counter += 1

            history, best_state = run_training_tcn(
                model=model,
                train_loader=tr_loader,
                test_loader=te_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=(momentum_sgd if momentum_sgd is not None else 0.0),
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=(CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None),
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM
            )

            config = {
                "input_dim": input_dim,
                "num_blocks": num_blocks,
                "channels": channels,
                "dilations": dilations,
                "dropout": dropout_for_model,
                "ln_block_end": ln_block_end,
                "kernel_size": kernel_size,
                "convs_per_block": convs_per_block,
                "activation": activation,
                "use_weight_norm": use_weight_norm,
                "wn_on_skip": wn_on_skip,
                "pooling": pooling,
                "fc_hidden": fc_hidden,
                "norm": norm,
                "batch_size": batch_size,
                "optimizer": optimizer_name,
                "momentum_sgd": momentum_sgd,
                "learning_rate": lr,
                "weight_decay": weight_decay,
            }

            cw_to_store = history.get("class_weights", None) if (USE_CLASS_WEIGHTS and CLASS_WEIGHTS is None) else (str(CLASS_WEIGHTS) if USE_CLASS_WEIGHTS else None)

            all_results.append({
                "id": id_counter,
                "timestamp": timestamp,
                "target_embedding": fname,
                "top_variant": top_tag,
                "source_params": preset_source,
                "model": "TCN",
                "seed": SEED,
                **config,
                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,
                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),
                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),
                "training": "STOPPED" if bool(history["early_stopped"]) else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                model_path = os.path.join(save_dir, f"tcn{id_counter}_v5_low{top_tag}_{fname.replace('.csv','')}.pth")
                save_model(model, model_path, config, id_counter)

        df_block = pd.DataFrame(all_results)
        if is_first:
            df_block.to_csv(results_csv, index=False)
            is_first = False
        else:
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_block.to_csv(results_csv, mode="a", header=False, index=False)

print("\n=== DONE: TCN v5 LOW 30/50/70 ===")


## Chequeo de configuración de TCN

In [ ]:
from src.utils.check_model import check_model_config

model_path = "saved_models/TCN/v5_top30%/tcn21_v4_30%_gaitpart_OUMVLP_TOP30.pth"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
check_model_config(model_path, device)

## Matriz de Confusión - Curva ROC - Curva PR (Precision Recall)

In [ ]:
import os
import ast
import pandas as pd
import torch
import numpy as np
from torch.utils.data import DataLoader

from src.tcn.model import FlexibleTCN
from src.tcn.prepare_data import prepare_tcn_data, collate_sequences
from src.utils.evaluate_model import evaluate_model 


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "Gait_Embeddings_good"

# --- CAMBIA ESTA RUTA ---
MODEL_PATH = "saved_models/TCN/v3/tcn753_v3_gln_phase1.pth"

TEST_SIZE = 0.1
SEED = 42
CLASS_NAMES = ["derecha", "centro", "izquierda"]  # orden real de tus labels (0/1/2)


def _parse_list(v, default=None):
    if default is None:
        default = []
    if v is None:
        return list(default)
    if isinstance(v, list):
        return v
    if isinstance(v, (tuple, np.ndarray)):
        return list(v)
    if isinstance(v, str):
        s = v.strip()
        if s == "" or s.lower() in ["none", "nan"]:
            return list(default)
        try:
            out = ast.literal_eval(s)
            return out if isinstance(out, list) else list(default)
        except Exception:
            return list(default)
    return list(default)

def _infer_embedding_csv_from_ckpt(model_path: str) -> str:
    """
    Intenta inferir el CSV del embedding desde el nombre del checkpoint:
      tcn{id}_v{N}_{embedding_name}.pth  ->  {embedding_name}.csv
    Si no se puede, lanza error para que lo pongas a mano.
    """
    base = os.path.basename(model_path).replace(".pth", "")
    parts = base.split("_")
    # esperable: ["tcn6", "v3", "gaitgl", "OUMVLP"]  -> embedding = "gaitgl_OUMVLP.csv"
    if len(parts) < 3:
        raise ValueError(f"No puedo inferir el embedding desde: {base}")
    emb = "_".join(parts[2:]) + ".csv"
    return emb

def _get_state_dict(ckpt: dict) -> dict:
    for k in ["model_state", "model_state_dict", "state_dict"]:
        if k in ckpt and isinstance(ckpt[k], dict) and len(ckpt[k]) > 0:
            return ckpt[k]
    raise ValueError("Checkpoint sin pesos: no encuentro 'model_state'/'model_state_dict'/'state_dict'.")


if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"No se encontró el modelo en: {MODEL_PATH}")

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
config = checkpoint.get("config", {})
state_dict = _get_state_dict(checkpoint)

if not isinstance(config, dict) or len(config) == 0:
    raise ValueError("El checkpoint no trae 'config' válida.")


EMBEDDING_FILE = _infer_embedding_csv_from_ckpt(MODEL_PATH)
emb_path = os.path.join(DATA_DIR, EMBEDDING_FILE)
if not os.path.exists(emb_path):
    raise FileNotFoundError(
        f" No existe el embedding inferido: {emb_path}\n"
        f"Si tu nombre de checkpoint no sigue el patrón, define EMBEDDING_FILE manualmente."
    )

print(f" Embedding usado para test: {EMBEDDING_FILE}")



df = pd.read_csv(emb_path, low_memory=False)

_, test_list, _, _ = prepare_tcn_data(
    df,
    norm=config.get("norm", "L2"),
    test_size=TEST_SIZE,
    seed=SEED
)

test_loader = DataLoader(
    test_list,
    batch_size=int(config.get("batch_size", 64)),
    shuffle=False,
    collate_fn=collate_sequences
)


channels     = _parse_list(config.get("channels", []))
dilations    = _parse_list(config.get("dilations", []))
dropout      = _parse_list(config.get("dropout", []))
ln_block_end = _parse_list(config.get("ln_block_end", []))

# Seguridad: si guardaste dropout/ln como escalares, conviértelos a listas
if not isinstance(dropout, list):
    dropout = [float(dropout)]
if not isinstance(ln_block_end, list):
    ln_block_end = [bool(ln_block_end)]

model = FlexibleTCN(
    input_dim=int(config["input_dim"]),
    channels=channels,
    kernel_size=int(config["kernel_size"]),
    dilations=dilations,
    convs_per_block=int(config["convs_per_block"]),
    dropout=dropout,
    activation=str(config.get("activation", "relu")),
    use_weight_norm=bool(config.get("use_weight_norm", False)),
    wn_on_skip=bool(config.get("wn_on_skip", False)),
    ln_block_end=ln_block_end,
    pooling=str(config.get("pooling", "last")),
    fc_hidden=(None if config.get("fc_hidden", None) is None else int(config["fc_hidden"])),
    num_classes=3
).to(DEVICE)

model.load_state_dict(state_dict)
model.eval()

evaluate_model(
    model=model,
    dataloader=test_loader,
    device=DEVICE,
    class_names=CLASS_NAMES,
    model_type="tcn"
)


---
---
# Modelo Transformer

El modelo Transformer empleado se basa en la arquitectura encoder propuesta por Vaswani et al. (2017) en Attention Is All You Need.
Cada capa está compuesta por dos subbloques: un mecanismo de autoatención multi-cabeza (Multi-Head Self-Attention) y una red feed-forward intermedia (Position-Wise Feed Forward Network), ambos conectados mediante residual connections y normalizados con dos capas LayerNorm.

En esta implementación se utiliza la variante Pre-Norm, donde la normalización se aplica antes de cada subbloque, siguiendo la recomendación de Xiong et al. (2020) por su mayor estabilidad en arquitecturas profundas.
Además, se incluye la opción de atención local (basada en Image Transformer, Parmar et al. 2018) que restringe el campo de atención a ventanas adyacentes, y la posibilidad de utilizar codificaciones posicionales sinusoidales o aprendibles, como en el Vision Transformer (ViT) de Dosovitskiy et al. (2021).

El token [CLS] se emplea como vector de agregación global, actuando como representación de la secuencia completa. Por ello, no se aplica un mecanismo adicional de atención global o pooling de atención, ya que el propio [CLS] y la autoatención interna realizan esta función implícitamente.

La implementación es completamente compatible con PyTorch (v2.3) y mantiene la estructura estándar de TransformerEncoderLayer, pero permite configurar el número de capas, cabezas de atención, activación, tipo de normalización, dimensión del feed-forward, codificación posicional y tipo de atención, ofreciendo así una versión FlexibleTransformer adaptada al contexto de embeddings de penalti utilizados en este trabajo.

La principal extensión introducida en esta implementación es su flexibilidad estructural, que permite definir una lista arbitraria de dimensiones de representación (model_dim) para cada capa, haciendo posible construir variantes uniformes, crecientes o decrecientes en profundidad. Este diseño convierte al modelo en una versión adaptable del Transformer clásico, capaz de aproximarse a configuraciones como Vision Transformer (ViT) cuando se usa codificación aprendible y pooling tipo [CLS], o a Time-Series Transformer (TST) y Temporal Transformer Encoders cuando se aplican secuencias con codificación sinusoidal y pooling promedio. De este modo, la arquitectura conserva la potencia representacional del Transformer original, pero con la capacidad de ajustar su jerarquía de abstracción interna según la naturaleza de los datos.


## Preparación de datos, split y bucle de entrenamiento: Transformer

Este bloque define el **pipeline estándar del Transformer** para trabajar con **secuencias de longitud variable**: `collate_transformer` aplica *padding* y construye una **máscara booleana** que marca qué pasos temporales son válidos, evitando que el modelo atienda a posiciones de padding. `prepare_transformer_data` agrupa por `video_ID`, aplica **normalización** (*MinMax* o *L2*) y realiza un **split estratificado por vídeo** para prevenir fugas entre train y test. Finalmente, `run_training_transformer` implementa el bucle de entrenamiento con **AMP**, selección de optimizador, opción de **class weights / label smoothing / gradient clipping**, y un **early stopping híbrido** que combina estancamiento en *loss* y *F1*, guardando el mejor *checkpoint* cuando la *loss* ya está por debajo de un umbral y el *F1* mejora.



#### Guardar la atencion del Transformer

En el Transformer, cada capa encoder calcula una matriz de atención que refleja cómo cada posición de la secuencia (en este caso, cada franja horizontal del vídeo) se relaciona con las demás.
A medida que las capas se apilan, las representaciones se vuelven más ricas: las primeras capas captan dependencias locales, mientras que las últimas capas integran información contextual de toda la secuencia.
Por ello, la atención de la última capa puede interpretarse como la atención final refinada del modelo, que indica qué regiones del cuerpo o del movimiento influyen más en la predicción final.
Visualizar esta matriz permite entender qué partes del embedding son más relevantes para la clasificación, proporcionando una interpretación semántica del comportamiento interno del Transformer.

In [ ]:
# import os
# import torch
# import pandas as pd


# def save_attention_maps(model, x_sample, mask_sample, save_dir: str, model_name: str = "transformer"):
#     """
#     Guarda mapas de atención tras un forward ENMASCARADO (key_padding_mask activo).
#     - x_sample: (B, T, D)
#     - mask_sample: (B, T) True=token válido, False=padding
#     """
#     os.makedirs(save_dir, exist_ok=True)

#     model.eval()
#     with torch.no_grad():
#         _ = model(x_sample, mask_sample)   #  CLAVE: pasar mask

#     attn_data = {}
#     for i, layer in enumerate(model.layers):
#         attn = getattr(layer, "last_attn_map", None)
#         if attn is not None:
#             # attn esperado: (B, num_heads, T, T)
#             attn_data[f"layer_{i}"] = attn.clone()

#     if not attn_data:
#         print(" No se encontraron mapas de atención en el modelo.")
#         return


#     # (Opcional pero recomendable) guardar también la máscara usada (alineada con attn)
#     if getattr(model, "use_cls_token", False):
#         B = mask_sample.size(0)
#         cls_col = torch.ones(B, 1, dtype=torch.bool, device=mask_sample.device)
#         attn_data["__mask__"] = torch.cat([cls_col, mask_sample], dim=1).detach().cpu()
#     else:
#         attn_data["__mask__"] = mask_sample.detach().cpu()


#     save_path = os.path.join(save_dir, f"{model_name}_attn.pt")
#     torch.save(attn_data, save_path)
#     print(f"🧠 Mapas de atención guardados en {save_path}")



# import matplotlib.pyplot as plt
# import seaborn as sns
# import torch

# def plot_attention_heatmap(attn_layer, mask=None, layer_idx=0, head_idx=None, batch_idx=0, save_path=None):
#     """
#     attn_layer: Tensor (B, H, T, T) como lo guarda tu Transformer
#     mask: Tensor (B, T) (sin CLS). Si tu modelo usa CLS, dentro del modelo se añade.
#           Aquí asumimos que el attn_layer YA incluye CLS si se usó.
#     """
#     # elegir una muestra del batch
#     attn = attn_layer[batch_idx]  # (H, T, T)

#     if head_idx is None:
#         attn = attn.mean(dim=0)   # (T, T)
#         title = f"Layer {layer_idx} — Mean over heads"
#     else:
#         attn = attn[head_idx]     # (T, T)
#         title = f"Layer {layer_idx} — Head {head_idx}"

#     # Si hay máscara, recortamos a longitud real (incluyendo CLS si existe)
#     if mask is not None:
#         m = mask[batch_idx].bool().cpu()
#         T_real = int(m.sum().item())
#         attn = attn[:T_real, :T_real]

#     plt.figure(figsize=(7, 6))
#     sns.heatmap(attn.cpu().numpy(), cmap="magma", square=True)
#     plt.title(title)
#     plt.xlabel("Key / Context step")
#     plt.ylabel("Query / Focus step")
#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, dpi=300)
#         print(f" Guardado en {save_path}")
#     else:
#         plt.show()


In [ ]:
# import torch

# # Cargar los datos de atención
# try:
#     attn_data = torch.load("saved_models/Transformer/v1/attn_maps/transformer99_v1_gaitpart_OUMVLP_attn.pt")
# except FileNotFoundError:
#     print("Error: El archivo Asegúrate de que la ruta sea correcta.")
#     exit()

# # Obtener las claves de las capas y ordenarlas para asegurar un orden consistente
# layer_keys = sorted(attn_data.keys())
# num_layers = len(layer_keys)

# print(f"Capas con mapas de atención: {layer_keys}")
# print(f"Número de capas detectadas: {num_layers}")

# print("\n Generando mapas de atención globales (media de todas las cabezas) para cada capa:")
# for i, layer_key in enumerate(layer_keys):
#     attn_layer = attn_data[layer_key]
    
#     # Calcular la media de todas las cabezas para esta capa
#     # Asumiendo que attn_layer tiene la forma (num_heads, seq_len, seq_len)
#     attn_mean = attn_layer.mean(dim=0)
#     plot_attention_heatmap(attn_mean, layer_idx=f"{i} (media)")

# # (Opcional) Una cabeza concreta para ver un patrón individual
# # Mantenemos este ejemplo para una cabeza específica de la última capa,
# # pero ahora de forma más robusta.
# print("\n (Opcional) Graficando una cabeza concreta para ver un patrón individual:")
# if layer_keys: # Asegurarse de que haya al menos una capa
#     last_layer_key = layer_keys[-1]
#     attn_last = attn_data[last_layer_key]
    
#     # Asegurarse de que la capa tenga al menos una cabeza de atención
#     if attn_last.shape[0] > 0:
#         # Graficar la primera cabeza (índice 0) de la última capa
#         plot_attention_heatmap(attn_last[0], layer_idx=f"Última capa ({last_layer_key})", head_idx=1)
#     else:
#         print(f"No hay cabezas de atención en la última capa ('{last_layer_key}') para graficar una cabeza individual.")
# else:
#     print("No hay capas de atención cargadas para graficar una cabeza individual.")


## Optimización de hiperparámetros con Optuna (Transformer)

Este bloque ejecuta **Optuna + Hyperband** para encontrar hiperparámetros robustos del Transformer por cada embedding. Para cada archivo, primero se hace un **split estratificado por `video_ID`** (manteniendo el test fuera) y, sobre el conjunto de entrenamiento, se optimiza el **F1 macro medio en validación** mediante **StratifiedKFold** también por vídeo, evitando fugas entre secuencias del mismo clip. El espacio de búsqueda cubre la arquitectura (número de capas, **patrones de `model_dims`** por capa con restricción de divisibilidad respecto al número de cabezas, `dim_feedforward`, dropout, activación, pre/post-norm, codificación posicional y pooling con opción de token CLS) y la optimización (lr, weight decay, batch size y optimizador con momentum si es SGD). Durante cada trial se usa **AMP** para acelerar el entrenamiento y se aplican **early stopping** y **pruning**: `trial.report()` alimenta al pruner para cortar pruebas malas temprano, y el early stopping por fold evita gastar épocas cuando el F1 se estanca. Al finalizar, los trials se exportan a CSV en un formato “limpio” reconstruyendo `model_dims` como lista y se guarda un resumen `best_params_Transformer.csv` con la mejor configuración por embedding.


In [ ]:
from src.transformer.optuna import optimize_embeddings

best_df = optimize_embeddings()
best_df.head()

## V1: Entrenamiento con las mejores configuraciones de Optuna (Transformers)

En **V1** se reutilizan las configuraciones óptimas obtenidas con **Optuna** y se aplican a todos los embeddings objetivo, manteniendo el mismo *split* estratificado por vídeo, entrenamiento y registro de métricas en CSV. Los modelos que superan el umbral de **F1** se guardan como *checkpoints*, y se desactiva el guardado de **mapas de atención** para evitar errores de memoria e inestabilidad durante la post-extracción.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.transformer.prepare_data import prepare_transformer_data, collate_transformer
from src.transformer.train import run_training_transformer
from src.transformer.model import FlexibleTransformer
from src.utils.save_methods import save_model, save_train_test_ids 


DATA_DIR        = "Gait_Embeddings_good"
BEST_PARAMS_CSV = "results/Transformer/Optuna/best_params_Transformer.csv"
RESULTS_CSV     = "results/Transformer/transformer_results_v1_Optuna.csv"
SAVE_DIR        = "saved_models/Transformer/v1"
SPLITS_DIR      = "results/Transformer/splits"

DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS       = 5000
F1_THRESHOLD = 0.55
TEST_SIZE    = 0.1
SEED         = 42
SAVE_IDS     = True

os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

# ID inicial (si ya hay resultados)
try:
    prev = pd.read_csv(RESULTS_CSV, low_memory=False)
    current_id = int(prev["id"].max()) if not prev.empty else 0
except (FileNotFoundError, pd.errors.EmptyDataError):
    current_id = 0

is_first_run = (not os.path.exists(RESULTS_CSV)) or (current_id == 0)


# cargar best params y lista de embeddings
best_params_df = pd.read_csv(BEST_PARAMS_CSV, low_memory=False)
embedding_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])


def _parse_list(x, default=None):
    if default is None:
        default = []
    if pd.isna(x):
        return default
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if s == "" or s.lower() in ("none", "nan"):
        return default
    try:
        return ast.literal_eval(s)
    except:
        return default


def _parse_float(x, default=0.0):
    if pd.isna(x):
        return default
    return float(x)


def _parse_int(x, default=0):
    if pd.isna(x):
        return default
    return int(float(x))


def _parse_bool(x, default=False):
    if pd.isna(x):
        return default
    return str(x).strip().lower() in ("true", "1", "yes")



# LOOP SOBRE CONFIGS ÓPTIMAS 
for _, row in best_params_df.iterrows():
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    results = []

    source_embedding = str(row.get("embedding", "unknown"))
    print(f"\n{'='*15} Config óptima obtenida de {source_embedding} {'='*15}")

    model_dims = _parse_list(row.get("model_dims"), default=128)  # puede ser int o lista según tu modelo

    optimizer_name = str(row.get("optimizer", "AdamW")).strip()
    optimizer_lower = optimizer_name.lower()

    momentum_sgd = 0.0
    if optimizer_lower == "sgd":
        momentum_sgd = _parse_float(row.get("momentum_sgd", 0.0), default=0.0)

    config_base = {
        "model_dims": model_dims,
        "num_heads": _parse_int(row.get("num_heads", 4), default=4),
        "num_layers": _parse_int(row.get("num_layers", 2), default=2),
        "dim_feedforward": _parse_int(row.get("dim_feedforward", 256), default=256),
        "dropout": _parse_float(row.get("dropout", 0.1), default=0.1),
        "activation": str(row.get("activation", "relu")).strip(),
        "norm_type": str(row.get("norm_type", "layernorm")).strip(),
        "pos_encoding": str(row.get("pos_encoding", "sin")).strip(),
        "pooling": str(row.get("pooling", "mean")).strip(),
        "use_cls_token": (str(row.get("pooling", "mean")).strip().lower() == "cls"),
        "lr": _parse_float(row.get("lr", 1e-4), default=1e-4),
        "weight_decay": _parse_float(row.get("weight_decay", 0.0), default=0.0),
        "batch_size": _parse_int(row.get("batch_size", 64), default=64),
        "optimizer": optimizer_name,
        "momentum_sgd": momentum_sgd,
        "norm": str(row.get("norm", "L2")).strip(),
    }


    # aplicar config a cada embedding target
    for fname in embedding_files:
        print(f"\n--- Entrenando Transformer v1 para target_embedding = {fname} ---")

        df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)

        train_list, test_list, train_ids, test_ids = prepare_transformer_data(
            df,
            norm=config_base["norm"],
            test_size=TEST_SIZE,
            seed=SEED,
        )

        if SAVE_IDS:
            save_train_test_ids(
                fname,
                train_ids,
                test_ids,
                SEED,
                output_dir=os.path.join(SPLITS_DIR, f"seed{SEED}")
            )

        train_loader = DataLoader(
            train_list,
            batch_size=config_base["batch_size"],
            shuffle=True,
            collate_fn=collate_transformer,
        )
        test_loader = DataLoader(
            test_list,
            batch_size=config_base["batch_size"],
            shuffle=False,
            collate_fn=collate_transformer,
        )

        input_dim = int(df.filter(like="feat_").shape[1])
        max_seq_len = max(seq.shape[0] for (seq, _) in train_list)

        model = FlexibleTransformer(
            input_dim=input_dim,
            model_dim=config_base["model_dims"],
            num_heads=config_base["num_heads"],
            num_layers=config_base["num_layers"],
            dim_feedforward=config_base["dim_feedforward"],
            dropout=config_base["dropout"],
            activation=config_base["activation"],
            norm_type=config_base["norm_type"],
            pos_encoding=config_base["pos_encoding"],
            use_cls_token=config_base["use_cls_token"],
            pooling=config_base["pooling"],
            num_classes=3,
            max_seq_len=max_seq_len,
        ).to(DEVICE)

        current_id += 1

        history, best_state = run_training_transformer(
            model=model,
            train_loader=train_loader,
            test_loader=test_loader,
            epochs=EPOCHS,
            lr=config_base["lr"],
            weight_decay=config_base["weight_decay"],
            optimizer_name=config_base["optimizer"],
            momentum_sgd=config_base["momentum_sgd"],
        )

        cfg = dict(config_base)
        cfg["input_dim"] = input_dim

        res_row = {
            "id": current_id,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": source_embedding,
            "model": "Transformer",
            "seed": SEED,
            **cfg,

            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],

            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": history["best_epoch"],
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "epochs_trained": history["epochs_trained"],
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        }

        results.append(res_row)

        if history["best_f1"] >= F1_THRESHOLD:
            print(f"  -> Guardando modelo (best F1={history['best_f1']:.4f})")
            model.load_state_dict(best_state)
            ckpt_name = f"transformer{current_id}_v1_{fname.replace('.csv','')}.pth"
            ckpt_path = os.path.join(SAVE_DIR, ckpt_name)
            save_model(model, ckpt_path, cfg, current_id)

            # -------------------------------
            # Guardado de atención DESACTIVADO
            # -------------------------------
            # (se comenta para evitar OOM/crash al guardar (B,H,T,T) por capa)
            #
            # model.eval()
            # try:
            #     x_sample, mask_sample, _ = next(iter(test_loader))
            #     x_sample, mask_sample = x_sample.to(DEVICE), mask_sample.to(DEVICE)
            #     attn_dir = os.path.join(SAVE_DIR, "attn_maps")
            #     os.makedirs(attn_dir, exist_ok=True)
            #     save_attention_maps(
            #         model=model,
            #         x_sample=x_sample,
            #         mask_sample=mask_sample,
            #         save_dir=attn_dir,
            #         model_name=f"transformer{current_id}_v1_{fname.replace('.csv','')}"
            #     )
            # except StopIteration:
            #     pass

    df_results = pd.DataFrame(results)
    if is_first_run:
        df_results.to_csv(RESULTS_CSV, header=True, index=False)
        is_first_run = False
    else:
        with open(RESULTS_CSV, "a", newline="") as f:
            f.write("\n")
        df_results.to_csv(RESULTS_CSV, mode="a", header=False, index=False)

    print(f"\nResultados para config origen {source_embedding} guardados en {RESULTS_CSV}")

print("\nEntrenamientos Transformer v1 con best_params completados")


## V2: Entrenamientos a partir de configuraciones personalizadas (Transformer)

En **V2** se evalúan **configuraciones personalizadas (presets)** del Transformer, siguiendo el mismo protocolo que en V1 pero sustituyendo las configuraciones óptimas de Optuna por un conjunto controlado de arquitecturas/hyperparámetros definidos manualmente. Para cada preset, el script recorre todos los embeddings disponibles, aplica `prepare_transformer_data` para agrupar por **video_ID**, normalizar (MinMax o L2) y realizar un **split estratificado por vídeo** (evitando fugas), y construye `DataLoader` con `collate_transformer`, que añade padding y genera la **máscara booleana** necesaria para que la autoatención ignore los pasos de padding. 

Durante el entrenamiento, `run_training_transformer` gestiona AMP, selección de optimizador y el **early stopping híbrido**, guardando el mejor estado cuando la loss ya está por debajo del umbral operativo y el F1 macro mejora. Al finalizar cada entrenamiento se registra una fila en el CSV con la configuración exacta, métricas de la última época y métricas del mejor checkpoint (best\_f1, best\_epoch, etc.). Si el modelo supera el umbral de F1, se guarda el checkpoint en disco con su configuración para reproducibilidad. En esta versión, el guardado de mapas de atención puede desactivarse para evitar problemas de memoria/compatibilidad en post-procesado.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader


from src.transformer.model import FlexibleTransformer
from src.transformer.prepare_data import prepare_transformer_data, collate_transformer
from src.transformer.train import run_training_transformer
from src.utils.save_methods import save_model
from src.utils.get_max_id import get_max_id



DATA_DIR    = "Gait_Embeddings_good"
PRESETS_CSV = "presets_configs/Transformer/presets_transformer.csv"
RESULTS_CSV = "results/Transformer/transformer_results_v2.csv"
SAVE_DIR    = "saved_models/Transformer/v2"

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 5000
THRESHOLD    = 0.55
TEST_SIZE    = 0.1
SEED         = 42

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)


presets = pd.read_csv(PRESETS_CSV, low_memory=False)
embedding_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])

hist_files = ["results/Transformer/transformer_results_v1_Optuna.csv",RESULTS_CSV]
id_counter = get_max_id(hist_files)

def _to_list(x):
    """Convierte strings tipo '[...]' a lista."""
    if isinstance(x, str):
        x = x.strip()
        if x.startswith("["):
            return ast.literal_eval(x)
    return list(x)

def _safe_float(x, default=0.0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def _safe_int(x, default=0):
    try:
        if pd.isna(x):
            return default
        return int(x)
    except Exception:
        return default


# BUCLE PRINCIPAL SOBRE PRESETS

for i, row in presets.iterrows():
    print(f"\n{'='*15} PRESET {i+1}/{len(presets)} {'='*15}")
    rows_out = []
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

    # ---------------- Reconstrucción limpia del preset ----------------
    model_dims = _to_list(row["model_dims"])
    num_layers = _safe_int(row["num_layers"])
    if len(model_dims) != num_layers:
        raise ValueError(f"Inconsistencia: len(model_dims)={len(model_dims)} != num_layers={num_layers}")

    optimizer_name = str(row["optimizer"]).strip()
    momentum_sgd = _safe_float(row.get("momentum_sgd", 0.0), default=0.0) if optimizer_name.lower() == "sgd" else 0.0

    config_base = {
        "model_dims": model_dims,
        "num_heads": _safe_int(row["num_heads"]),
        "num_layers": num_layers,
        "dim_feedforward": _safe_int(row["dim_feedforward"]),
        "dropout": _safe_float(row["dropout"]),
        "activation": str(row["activation"]).strip(),
        "norm_type": str(row["norm_type"]).strip(),
        "pos_encoding": str(row["pos_encoding"]).strip(),
        "pooling": str(row["pooling"]).strip(),
        "use_cls_token": (str(row["pooling"]).strip().lower() == "cls"),
        "lr": _safe_float(row["lr"]),
        "weight_decay": _safe_float(row["weight_decay"]),
        "batch_size": _safe_int(row["batch_size"], default=64),
        "optimizer": optimizer_name,
        "momentum_sgd": momentum_sgd,
        "norm": str(row["norm"]).strip(),
    }


    # ENTRENAMIENTO EN TODOS LOS EMBEDDINGS
    for fname in embedding_files:
        print(f"\n--- Entrenando Transformer v2 para target_embedding = {fname} ---")
        df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)

        # ------ Preparar datos ------
        train_list, test_list, train_ids, test_ids = prepare_transformer_data(
            df=df,
            norm=config_base["norm"],
            test_size=TEST_SIZE,
            seed=SEED,
        )

        train_loader = DataLoader(
            train_list,
            batch_size=config_base["batch_size"],
            shuffle=True,
            collate_fn=collate_transformer,
        )
        test_loader = DataLoader(
            test_list,
            batch_size=config_base["batch_size"],
            shuffle=False,
            collate_fn=collate_transformer,
        )

        input_dim   = int(df.filter(like="feat_").shape[1])
        max_seq_len = max(seq.shape[0] for (seq, _) in train_list)

        # ------ Instanciar modelo ------
        model = FlexibleTransformer(
            input_dim=input_dim,
            model_dim=config_base["model_dims"],
            num_heads=config_base["num_heads"],
            num_layers=config_base["num_layers"],
            dim_feedforward=config_base["dim_feedforward"],
            dropout=config_base["dropout"],
            activation=config_base["activation"],
            norm_type=config_base["norm_type"],
            pos_encoding=config_base["pos_encoding"],
            use_cls_token=config_base["use_cls_token"],
            pooling=config_base["pooling"],
            num_classes=3,
            max_seq_len=max_seq_len,
        ).to(DEVICE)

        id_counter += 1

        # ------ Entrenar ------
        history, best_state = run_training_transformer(
            model=model,
            train_loader=train_loader,
            test_loader=test_loader,
            epochs=EPOCHS,
            lr=config_base["lr"],
            weight_decay=config_base["weight_decay"],
            optimizer_name=config_base["optimizer"],
            momentum_sgd=config_base["momentum_sgd"],
        )

        # ------ Fila de resultados ------
        config = dict(config_base)
        config["input_dim"] = input_dim

        row_out = {
            "id": id_counter,
            "timestamp": timestamp,
            "target_embedding": fname,
            "source_params": f"Preset_v2_{i+1}",
            "model": "Transformer",
            "seed": SEED,
            **config,

            # últimas métricas
            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],

            # mejores métricas (checkpoint)
            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": int(history["best_epoch"]),
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "epochs_trained": int(history["epochs_trained"]),
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 3),
        }
        rows_out.append(row_out)

        # ------ Guardar modelo si supera umbral ------
        if history["best_f1"] >= THRESHOLD:
            print(f"  -> Guardando checkpoint (best F1={history['best_f1']:.4f})")
            model.load_state_dict(best_state)

            ckpt_name = f"transformer{id_counter}_v2_{fname.replace('.csv','')}.pth"
            ckpt_path = os.path.join(SAVE_DIR, ckpt_name)
            save_model(model, ckpt_path, config, id_counter)

            # ---------------------------------------------------------
            # Mapas de atención (DESACTIVADO / COMENTADO)
            # ---------------------------------------------------------
            # model.eval()
            # try:
            #     x_sample, mask_sample, _ = next(iter(test_loader))
            #     x_sample, mask_sample = x_sample.to(DEVICE), mask_sample.to(DEVICE)
            #
            #     attn_dir = os.path.join(SAVE_DIR, "attn_maps")
            #     os.makedirs(attn_dir, exist_ok=True)
            #
            #     save_attention_maps(
            #         model=model,
            #         x_sample=x_sample,
            #         mask_sample=mask_sample,
            #         save_dir=attn_dir,
            #         model_name=f"transformer{id_counter}_v2_{fname.replace('.csv','')}",
            #     )
            # except StopIteration:
            #     print("  [Aviso] test_loader vacío; no se generan mapas de atención.")


    df_results = pd.DataFrame(rows_out)
    if os.path.exists(RESULTS_CSV):
        with open(RESULTS_CSV, "a", newline="") as f:
            f.write("\n")
        df_results.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_results.to_csv(RESULTS_CSV, index=False)

    print(f"\n Finalizado preset {i+1}/{len(presets)} → {RESULTS_CSV}")

print("\n Entrenamientos Transformer V2 (presets) completados.")


## V3: Re-entrenamiento robusto (Transformer)

En **V3** el Transformer entra en una fase de **refinamiento**: en vez de explorar de cero, se reutiliza el histórico de **V1 y V2** para recuperar únicamente las configuraciones que ya demostraron rendimiento suficiente (F1 ≥ umbral). La idea no es “probar más”, sino **re-entrenar de forma controlada** esas combinaciones bajo un *recipe* unificado (class weights + label smoothing + gradient clipping) para estabilizar la convergencia y reducir comportamientos frágiles en validación.

A nivel práctico, el bloque construye una **firma estable** de cada configuración (normalizando listas, booleanos y strings) para poder **mapear exactamente** cada par *(embedding, configuración)* al **mismo ID** usado en el histórico. Con esto se mantiene trazabilidad y se evita duplicar modelos. Después, se carga el listado de “mejores configs” (`best_configs_transformer_v3.csv`) y se entrena sobre todos los embeddings con el mismo split estratificado por `video_ID`. Cada ejecución guarda métricas completas (última época y mejor checkpoint) y, si supera el umbral, persiste el modelo con su configuración asociada.


In [ ]:
import os
import json
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.transformer.model import FlexibleTransformer
from src.transformer.prepare_data import prepare_transformer_data, collate_transformer
from src.transformer.train import run_training_transformer
from src.utils.save_methods import save_model
from src.utils.find_best import find_best_configs 


DATA_DIR    = "Gait_Embeddings_good"
RESULTS_CSV = "results/Transformer/transformer_results_v3.csv"
SAVE_DIR    = "saved_models/Transformer/v3"

HISTORICAL  = [
    "results/Transformer/transformer_results_v1_Optuna.csv",
    "results/Transformer/transformer_results_v2.csv",
]

TEST_SIZE   = 0.1
SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS      = 5000
THRESHOLD   = 0.6

LABEL_SMOOTH  = 0.03
CLIP_NORM     = 1.0
CLASS_WEIGHTS = [1.0, 3.1864, 1.3333]

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)

MATCH_COLS = [
    "model_dims","num_heads","num_layers","dim_feedforward","dropout",
    "activation","norm_type","pos_encoding","pooling","use_cls_token",
    "batch_size","optimizer","momentum_sgd","lr","weight_decay","norm"
]

def normalize_value(v):
    if pd.isna(v):
        return None

    if isinstance(v, str):
        vv = v.strip()

        try:
            parsed = ast.literal_eval(vv)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        if vv.lower() in ["true", "false"]:
            return vv.lower() == "true"

        return vv.lower()

    return v

def normalize_config_row(row):
    return {col: normalize_value(row.get(col)) for col in MATCH_COLS}

def config_signature(row):
    clean = normalize_config_row(row)
    return json.dumps(clean, sort_keys=True)

def _model_dims_from_row(prow):
    raw_md = normalize_value(prow.get("model_dims"))
    n_layers = int(prow.get("num_layers"))
    if isinstance(raw_md, list):
        return raw_md
    return [int(raw_md)] * n_layers


# CARGAR HISTÓRICO Y CREAR LOOKUP (embedding, sig) → ID

hist_files = [p for p in HISTORICAL if os.path.exists(p)]
if not hist_files:
    raise FileNotFoundError("No hay histórico Transformer (V1/V2).")

df_hist = pd.concat([pd.read_csv(f, low_memory=False) for f in hist_files], ignore_index=True)
df_hist["__sig__"] = df_hist.apply(config_signature, axis=1)

LOOKUP = {
    (r["target_embedding"], r["__sig__"]): int(r["id"])
    for _, r in df_hist.iterrows()
    if not pd.isna(r.get("id"))
}

print(f"LOOKUP creado con {len(LOOKUP)} combinaciones (embedding,config).")


best_configs_path = "results/Transformer/best_configs_transformer_v3.csv"
if not os.path.exists(best_configs_path):
    presets_df = find_best_configs(
        csv_files=hist_files,
        output_file=best_configs_path,
        model_name="Transformer",
        f1_threshold=THRESHOLD
    )
else:
    presets_df = pd.read_csv(best_configs_path, low_memory=False)

if presets_df is None or presets_df.empty:
    raise ValueError("No hay presets válidos para V3.")

print(f"Configs seleccionadas para V3: {len(presets_df)}")


# LOOP PRINCIPAL TRANSFORMER V3

emb_list = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])

for idx, prow in presets_df.iterrows():
    all_results = []

    print("\n------------------------------")
    print(f"PRESET {idx+1}/{len(presets_df)}")
    print("------------------------------")

    sig = config_signature(prow)

    for fname in emb_list:
        run_id = LOOKUP.get((fname, sig))
        if run_id is None:
            continue

        df_emb = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)

        model_dims = _model_dims_from_row(prow)
        optimizer_nm = str(prow.get("optimizer")).strip()
        momentum_sgd = float(prow.get("momentum_sgd")) if optimizer_nm.upper() == "SGD" else 0.0

        config = {
            "model_dims": model_dims,
            "num_heads": int(prow.get("num_heads")),
            "num_layers": int(prow.get("num_layers")),
            "dim_feedforward": int(prow.get("dim_feedforward")),
            "dropout": float(prow.get("dropout")),
            "activation": str(prow.get("activation")).strip(),
            "norm_type": str(prow.get("norm_type")).strip(),
            "pos_encoding": str(prow.get("pos_encoding")).strip(),
            "pooling": str(prow.get("pooling")).strip(),
            "use_cls_token": bool(normalize_value(prow.get("use_cls_token"))),
            "batch_size": int(prow.get("batch_size")),
            "optimizer": optimizer_nm,
            "momentum_sgd": momentum_sgd if optimizer_nm.upper() == "SGD" else None,
            "lr": float(prow.get("lr")),
            "weight_decay": float(prow.get("weight_decay")),
            "norm": str(prow.get("norm")).strip() if pd.notna(prow.get("norm")) else "",
        }

        train_list, test_list, _, _ = prepare_transformer_data(
            df=df_emb,
            norm=config["norm"],
            test_size=TEST_SIZE,
            seed=SEED
        )

        train_loader = DataLoader(
            train_list,
            batch_size=config["batch_size"],
            shuffle=True,
            collate_fn=collate_transformer
        )
        test_loader = DataLoader(
            test_list,
            batch_size=config["batch_size"],
            shuffle=False,
            collate_fn=collate_transformer
        )

        input_dim = int(df_emb.filter(like="feat_").shape[1])
        config["input_dim"] = input_dim
        max_seq_len = max(seq.shape[0] for (seq, _) in train_list)

        model = FlexibleTransformer(
            input_dim=input_dim,
            model_dim=config["model_dims"],
            num_heads=config["num_heads"],
            num_layers=config["num_layers"],
            dim_feedforward=config["dim_feedforward"],
            dropout=config["dropout"],
            activation=config["activation"],
            norm_type=config["norm_type"],
            pos_encoding=config["pos_encoding"],
            pooling=config["pooling"],
            use_cls_token=config["use_cls_token"],
            num_classes=3,
            max_seq_len=max_seq_len,
        ).to(DEVICE)

        history, best_state = run_training_transformer(
            model=model,
            train_loader=train_loader,
            test_loader=test_loader,
            epochs=EPOCHS,
            lr=config["lr"],
            weight_decay=config["weight_decay"],
            optimizer_name=config["optimizer"],
            momentum_sgd=(momentum_sgd if optimizer_nm.upper() == "SGD" else 0.0),
            use_class_weights=True,
            class_weights=CLASS_WEIGHTS,
            label_smoothing=LABEL_SMOOTH,
            clip_grad_norm=CLIP_NORM
        )

        row_out = {
            "id": run_id,
            "timestamp": pd.Timestamp.now().strftime("%Y%m%d_%H%M%S"),
            "target_embedding": fname,
            "source_params": "best_configs_v3",
            "model": "Transformer",
            "seed": SEED,
            **config,

            "label_smoothing": LABEL_SMOOTH,
            "clip_grad_norm": CLIP_NORM,
            "class_weights": str(CLASS_WEIGHTS),

            "train_loss": round(history["train_loss"][-1], 5),
            "test_loss": round(history["test_loss"][-1], 5),
            "accuracy": round(history["test_acc"][-1], 5),
            "f1_macro": round(history["test_f1"][-1], 5),
            "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],

            "best_accuracy": round(history["best_acc"], 5),
            "best_f1": round(history["best_f1"], 5),
            "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
            "best_epoch": int(history["best_epoch"]),
            "train_loss_best_state": round(history["train_loss_best_state"], 5),
            "test_loss_best_state": round(history["test_loss_best_state"], 5),
            "epochs_trained": int(history["epochs_trained"]),
            "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
            "duration_sec": round(history["duration_sec"], 5),
        }

        all_results.append(row_out)

        if history["best_f1"] >= THRESHOLD:
            model.load_state_dict(best_state)
            ckpt_path = os.path.join(SAVE_DIR, f"transformer{run_id}_v3_{fname.replace('.csv','')}.pth")
            save_model(model, ckpt_path, config, run_id)

    if all_results:
        df_out = pd.DataFrame(all_results)

        if os.path.exists(RESULTS_CSV):
            with open(RESULTS_CSV, "a", newline="") as f:
                f.write("\n")
            df_out.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
        else:
            df_out.to_csv(RESULTS_CSV, index=False)

        df_final = pd.read_csv(RESULTS_CSV, low_memory=False).sort_values(by="id", ascending=True)
        df_final.to_csv(RESULTS_CSV, index=False)

        print(f"Resultados añadidos y CSV ordenado por ID → {RESULTS_CSV}")
    else:
        print("No se entrenó ningún modelo para este preset.")


## Comparación de resultados de entrenamiento V1, V2 y V3 (Transformer)


*   **`mean_f1` (y `mean_diff`)**: Esta es la métrica más importante para evaluar la mejora general. `mean_f1_v3` es el F1-score *promedio* de todos los entrenamientos de la `v3` para un embedding, mientras que `mean_f1_v1_v2` es el promedio de los históricos. La columna **`mean_diff`** te dice si, en conjunto, tus nuevos entrenamientos son mejores. **Un valor positivo y alto aquí es tu principal objetivo**, ya que indica que el rendimiento promedio ha mejorado de forma consistente.

*   **`max_f1` (y `max_diff`)**: Esta métrica se centra en el rendimiento pico. Compara el mejor F1-score individual que se consiguió en `v3` (`max_f1_v3`) con el mejor récord histórico (`max_f1_v1_v2`). La columna **`max_diff`** te muestra si has batido un nuevo "récord" de rendimiento. Es útil para ver el potencial máximo de una configuración, aunque una mejora aquí podría ser un golpe de suerte si la media no ha subido.

*   **`std_f1` (Desviación Estándar)**: Esta métrica mide la **estabilidad y consistencia** de tus resultados. Un valor bajo es bueno, ya que significa que la mayoría de los entrenamientos para ese embedding obtuvieron un F1-score muy similar. Si **`std_f1_v3` es menor que `std_f1_v1_v2`**, es una excelente noticia, porque indica que tus nuevos entrenamientos no solo son mejores, sino también más fiables y predecibles.

In [ ]:
from src.utils.comparison_results import compare_model_results

MODEL_TYPE = "Transformer"
HISTORICAL_FILES = [
    "results/Transformer/transformer_results_v1_Optuna.csv",
    "results/Transformer/transformer_results_v2.csv"
]
FINAL_FILE = "results/Transformer/transformer_results_v3.csv"
OUTPUT_DIR = "results/Comparisons_V1_V2_V3"


compare_model_results(
    model_type=MODEL_TYPE,
    historical_files=HISTORICAL_FILES,
    final_file_path=FINAL_FILE,
    output_dir=OUTPUT_DIR
)

## V4: Entrenamientos con embeddings combinados (Transformer)

En **V4** se entrena el **Transformer** sobre embeddings **combinados** en tres variantes: **concat**, **mean** y **reduced**. Los resultados previos mostraron que, sin *gradient clipping*, la optimización tiende a quedarse estancada (pérdida “plana” y F1 sin mejora real), por lo que en esta fase se fija **clip_grad_norm** como requisito operativo. Además, debido al coste del Transformer, se mantiene **AMP** (mixed precision) dentro del bucle de entrenamiento para que el volumen de experimentación sea asumible.

También se mantiene **class weights** (por desbalance) y se incorpora **label smoothing** como regularizador práctico: en estas ejecuciones suele reducir de forma clara la **test loss** (mitigando sobreconfianza) sin penalizar sistemáticamente el **F1 macro**, estabilizando la validación.

El código siguiente unifica la ejecución de las tres variantes (**concat/mean/reduced**) en un único bloque: para cada subcarpeta carga su **CSV de presets específico**, entrena todas las combinaciones *(preset × embedding)*, guarda métricas en su CSV de resultados y persiste checkpoints cuando se supera el umbral de F1.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.transformer.model import FlexibleTransformer
from src.transformer.prepare_data import prepare_transformer_data, collate_transformer
from src.transformer.train import run_training_transformer
from src.utils.get_max_id import get_max_id
from src.utils.save_methods import save_model



DATA_DIR = "Gait_Embeddings_Combined"
RESULTS_DIR = "results/Transformer"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS    = 5000
THRESHOLD = 0.6
TEST_SIZE = 0.1
SEED      = 42

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None  # o fija una lista tipo [..] de pesos
LABEL_SMOOTH  = 0.02
CLIP_NORM     = 1.0

SUBFOLDERS = [
    # (name, presets_csv, results_csv, save_dir)
    ("concat","presets_configs/Transformer/presets_transformer_concat.csv",
     os.path.join(RESULTS_DIR, "transformer_results_v4_concat.csv"),
     "saved_models/Transformer/v4_concat"),

    ("mean", "presets_configs/Transformer/presets_transformer_mean.csv",
     os.path.join(RESULTS_DIR, "transformer_results_v4_mean.csv"),
     "saved_models/Transformer/v4_mean"),

    ("reduced", "presets_configs/Transformer/presets_transformer_reduced.csv",
     os.path.join(RESULTS_DIR, "transformer_results_v4_reduced.csv"),
     "saved_models/Transformer/v4_reduced"),
]


def _safe_list(x, default=None):
    """Convierte strings tipo '[..]' a lista; si está vacío, devuelve default."""
    if default is None:
        default = []
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return default
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if not s:
        return default
    try:
        out = ast.literal_eval(s)
        return out if isinstance(out, list) else default
    except Exception:
        return default

def _to_str(x, default=""):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return default
    return str(x).strip()

def _to_float(x, default=0.0):
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return float(default)
        return float(x)
    except Exception:
        return float(default)

def _to_int(x, default=0):
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return int(default)
        return int(x)
    except Exception:
        return int(default)

for sub_name, presets_csv, results_csv, save_dir in SUBFOLDERS:
    sub_dir = os.path.join(DATA_DIR, sub_name)

    # ---- checks ----
    if not os.path.isdir(sub_dir):
        print(f" Subcarpeta no encontrada: {sub_dir} (skip)")
        continue
    if not os.path.exists(presets_csv):
        print(f" Presets no encontrado: {presets_csv} (skip)")
        continue

    os.makedirs(save_dir, exist_ok=True)

    emb_files = sorted([f for f in os.listdir(sub_dir) if f.endswith(".csv")])
    if not emb_files:
        print(f" No hay embeddings en: {sub_dir} (skip)")
        continue

    presets = pd.read_csv(presets_csv, low_memory=False)
    if presets.empty:
        print(f" Presets vacío: {presets_csv} (skip)")
        continue

    id_counter = get_max_id([results_csv])
    print("\n" + "=" * 70)
    print(f"📂 Variante: {sub_name} | embeddings={len(emb_files)} | presets={len(presets)}")
    print(f"📝 Resultados -> {results_csv}")
    print(f"💾 Modelos -> {save_dir}")
    print("=" * 70)

    for preset_idx, row in presets.iterrows():
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        all_results = []

        # -------------------------
        # Leer hiperparámetros Transformer
        # -------------------------
        model_dims = _safe_list(row.get("model_dims"))
        num_layers = _to_int(row.get("num_layers"), default=len(model_dims) if model_dims else 1)

        if model_dims and len(model_dims) != num_layers:
            raise ValueError(
                f"Inconsistencia en model_dims y num_layers "
                f"(preset {preset_idx+1} en {presets_csv})."
            )

        optimizer_name = _to_str(row.get("optimizer", "AdamW"))
        momentum_sgd = _to_float(row.get("momentum_sgd", 0.0)) if optimizer_name.lower() == "sgd" else 0.0

        num_heads       = _to_int(row.get("num_heads", 4))
        dim_feedforward = _to_int(row.get("dim_feedforward", 256))
        dropout         = _to_float(row.get("dropout", 0.1))
        activation      = _to_str(row.get("activation", "gelu"))
        norm_type       = _to_str(row.get("norm_type", "prenorm"))
        pos_encoding    = _to_str(row.get("pos_encoding", "learned"))
        pooling         = _to_str(row.get("pooling", "cls"))
        use_cls_token   = (pooling.lower() == "cls")

        lr           = _to_float(row.get("lr", row.get("learning_rate", 1e-4)))
        weight_decay = _to_float(row.get("weight_decay", 0.0))
        batch_size   = _to_int(row.get("batch_size", 64))
        norm         = _to_str(row.get("norm", "L2"))

        config_base = {
            "model_dims": model_dims,
            "num_heads": num_heads,
            "num_layers": num_layers,
            "dim_feedforward": dim_feedforward,
            "dropout": dropout,
            "activation": activation,
            "norm_type": norm_type,
            "pos_encoding": pos_encoding,
            "pooling": pooling,
            "use_cls_token": use_cls_token,
            "lr": lr,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "optimizer": optimizer_name,
            "momentum_sgd": (momentum_sgd if optimizer_name.lower() == "sgd" else None),
            "norm": norm,
        }

        print(f"\n=== [{sub_name}] PRESET {preset_idx+1}/{len(presets)} ===")

        # ======================================================
        # ENTRENAR CONTRA CADA EMBEDDING DE ESA SUBCARPETA
        # ======================================================
        for fname in emb_files:
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_transformer_data(
                df=df,
                norm=norm,
                test_size=TEST_SIZE,
                seed=SEED
            )

            train_loader = DataLoader(
                train_list,
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_transformer
            )
            test_loader = DataLoader(
                test_list,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_transformer
            )

            input_dim = int(df.filter(like="feat_").shape[1])
            max_seq_len = max(seq.shape[0] for (seq, _) in train_list) if len(train_list) > 0 else 128

            model = FlexibleTransformer(
                input_dim=input_dim,
                model_dim=model_dims,
                num_heads=num_heads,
                num_layers=num_layers,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                activation=activation,
                norm_type=norm_type,
                pos_encoding=pos_encoding,
                use_cls_token=use_cls_token,
                pooling=pooling,
                num_classes=3,
                max_seq_len=max_seq_len
            ).to(DEVICE)

            id_counter += 1

            history, best_state = run_training_transformer(
                model=model,
                train_loader=train_loader,
                test_loader=test_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=(CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None),
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM
            )

            # lo que guardamos en config + resultados
            config = dict(config_base)
            config["input_dim"] = input_dim
            config["max_seq_len"] = max_seq_len  # útil para reproducibilidad

            cw_to_store = str(CLASS_WEIGHTS) if (USE_CLASS_WEIGHTS and CLASS_WEIGHTS is not None) else history.get("class_weights", None)

            all_results.append({
                "id": id_counter,
                "timestamp": timestamp,
                "target_embedding": fname,
                "variant": sub_name,                  # concat/mean/reduced
                "source_params": f"Preset_{preset_idx+1}",
                "model": "Transformer",
                "seed": SEED,

                **config,

                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,

                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),

                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),

                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                ckpt_name = f"transformer{id_counter}_v4_{sub_name}_{fname.replace('.csv','')}.pth"
                ckpt_path = os.path.join(save_dir, ckpt_name)
                save_model(model, ckpt_path, config, id_counter)

        # ---- append resultados de este preset ----
        df_results = pd.DataFrame(all_results)
        if os.path.exists(results_csv):
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)
        else:
            df_results.to_csv(results_csv, index=False)

print("\n=== ENTRENAMIENTOS TRANSFORMER V4 COMPLETADOS (concat/mean/reduced) ===")


## V5: Entrenamientos con embeddings combinados por Top-K Steps (Transformer)

En **V5** se evalúa el Transformer con embeddings recortados mediante **Top-K Steps**, donde cada variante (30/50/70%) conserva únicamente las franjas temporales consideradas más informativas. El objetivo es comprobar si, al reducir pasos “menos relevantes”, el modelo aprende representaciones más discriminativas y mejora su generalización.

El código unifica el entrenamiento por recortes: para cada subcarpeta **(30%, 50%, 70%)** carga los embeddings correspondientes, aplica la misma receta establecida en V4 (**AMP + class weights + gradient clipping + label smoothing**), registra métricas en un CSV específico por variante y guarda checkpoints cuando el **F1 macro** supera el umbral.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.transformer.model import FlexibleTransformer
from src.transformer.prepare_data import prepare_transformer_data, collate_transformer
from src.transformer.train import run_training_transformer
from src.utils.save_methods import save_model
from src.utils.get_max_id import get_max_id


DATA_DIR = "Gait_Embeddings_TopSteps"
PRESETS_CSV = "presets_configs/Transformer/best_configs_Transformer.csv"

RESULTS_DIR = "results/Transformer"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS    = 5000
THRESHOLD = 0.6
TEST_SIZE = 0.1
SEED      = 42

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH  = 0.02
CLIP_NORM     = 1.0


VARIANTS = [
    ("30%", os.path.join(RESULTS_DIR, "transformer_results_v5_top30%.csv"), "saved_models/Transformer/v5_top30%"),
    ("50%", os.path.join(RESULTS_DIR, "transformer_results_v5_top50%.csv"), "saved_models/Transformer/v5_top50%"),
    ("70%", os.path.join(RESULTS_DIR, "transformer_results_v5_top70%.csv"), "saved_models/Transformer/v5_top70%"),
]

for _, _, sd in VARIANTS:
    os.makedirs(sd, exist_ok=True)

presets = pd.read_csv(PRESETS_CSV, low_memory=False)
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

def _safe_list(x, default=None):
    if default is None:
        default = []
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return default
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if not s:
        return default
    try:
        out = ast.literal_eval(s)
        return out if isinstance(out, list) else default
    except Exception:
        return default

def _to_str(x, default=""):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return default
    return str(x).strip()

def _to_float(x, default=0.0):
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return float(default)
        return float(x)
    except Exception:
        return float(default)

def _to_int(x, default=0):
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return int(default)
        return int(x)
    except Exception:
        return int(default)

def get_embedding_files(folder_path: str):
    if not os.path.isdir(folder_path):
        return []
    return sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])

for top_tag, results_csv, save_dir in VARIANTS:
    sub_dir = os.path.join(DATA_DIR, top_tag)

    if not os.path.isdir(sub_dir):
        print(f" Subcarpeta no encontrada: {sub_dir} (skip)")
        continue

    emb_files = get_embedding_files(sub_dir)
    if not emb_files:
        print(f" No hay CSVs en: {sub_dir} (skip)")
        continue

    id_counter = get_max_id([results_csv])

    print("\n" + "=" * 70)
    print(f"📂 Top-K: {top_tag} | embeddings={len(emb_files)} | presets={len(presets)}")
    print(f"📝 Resultados -> {results_csv}")
    print(f"💾 Modelos -> {save_dir}")
    print("=" * 70)

    for preset_idx, row in presets.iterrows():
        print(f"\n=== [TOP {top_tag}] PRESET {preset_idx+1}/{len(presets)} ===")
        all_results = []

        model_dims = _safe_list(row.get("model_dims"))
        num_layers = _to_int(row.get("num_layers"), default=len(model_dims) if model_dims else 1)

        if model_dims and len(model_dims) != num_layers:
            raise ValueError(f"Inconsistencia model_dims/num_layers en preset {preset_idx+1}.")

        optimizer_name = _to_str(row.get("optimizer", "AdamW"))
        momentum_sgd = _to_float(row.get("momentum_sgd", 0.0)) if optimizer_name.lower() == "sgd" else 0.0

        num_heads       = _to_int(row.get("num_heads", 4))
        dim_feedforward = _to_int(row.get("dim_feedforward", 256))
        dropout         = _to_float(row.get("dropout", 0.1))
        activation      = _to_str(row.get("activation", "gelu"))
        norm_type       = _to_str(row.get("norm_type", "prenorm"))
        pos_encoding    = _to_str(row.get("pos_encoding", "learned"))
        pooling         = _to_str(row.get("pooling", "cls"))
        use_cls_token   = (pooling.lower() == "cls")

        lr           = _to_float(row.get("lr", row.get("learning_rate", 1e-4)))
        weight_decay = _to_float(row.get("weight_decay", 0.0))
        batch_size   = _to_int(row.get("batch_size", 64))
        norm         = _to_str(row.get("norm", "L2"))

        config_base = {
            "model_dims": model_dims,
            "num_heads": num_heads,
            "num_layers": num_layers,
            "dim_feedforward": dim_feedforward,
            "dropout": dropout,
            "activation": activation,
            "norm_type": norm_type,
            "pos_encoding": pos_encoding,
            "pooling": pooling,
            "use_cls_token": use_cls_token,
            "lr": lr,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "optimizer": optimizer_name,
            "momentum_sgd": (momentum_sgd if optimizer_name.lower() == "sgd" else None),
            "norm": norm,
        }

        for fname in emb_files:
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_transformer_data(
                df=df,
                norm=norm,
                test_size=TEST_SIZE,
                seed=SEED
            )

            train_loader = DataLoader(
                train_list,
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_transformer
            )
            test_loader = DataLoader(
                test_list,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_transformer
            )

            input_dim = int(df.filter(like="feat_").shape[1])
            max_seq_len = max(seq.shape[0] for (seq, _) in train_list) if len(train_list) > 0 else 128

            model = FlexibleTransformer(
                input_dim=input_dim,
                model_dim=model_dims,
                num_heads=num_heads,
                num_layers=num_layers,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                activation=activation,
                norm_type=norm_type,
                pos_encoding=pos_encoding,
                use_cls_token=use_cls_token,
                pooling=pooling,
                num_classes=3,
                max_seq_len=max_seq_len
            ).to(DEVICE)

            id_counter += 1

            history, best_state = run_training_transformer(
                model=model,
                train_loader=train_loader,
                test_loader=test_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=(CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None),
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM
            )

            config = dict(config_base)
            config["input_dim"] = input_dim
            config["max_seq_len"] = max_seq_len

            cw_to_store = str(CLASS_WEIGHTS) if (USE_CLASS_WEIGHTS and CLASS_WEIGHTS is not None) else history.get("class_weights", None)

            all_results.append({
                "id": id_counter,
                "timestamp": timestamp,
                "target_embedding": fname,
                "top_variant": top_tag,
                "source_params": f"Preset_{preset_idx+1}",
                "model": "Transformer",
                "seed": SEED,

                **config,

                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,

                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),

                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),

                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                ckpt_name = f"transformer{id_counter}_v5_top{top_tag}_{fname.replace('.csv','')}.pth"
                ckpt_path = os.path.join(save_dir, ckpt_name)
                save_model(model, ckpt_path, config, id_counter)

        df_results = pd.DataFrame(all_results)
        if os.path.exists(results_csv):
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)
        else:
            df_results.to_csv(results_csv, index=False)

print("\n=== DONE: Transformer V5 (TopSteps 30/50/70) ===")


## V5: Prueba rápida con embeddings Low-k Steps (Transformer)

Esta celda ejecuta una **prueba rápida** del Transformer con embeddings recortados por **Low-K Steps** (los pasos menos informativos). Para cada variante (30/50/70%) se cargan presets específicos y se entrena sobre todos los embeddings de la subcarpeta correspondiente, manteniendo la misma receta de entrenamiento establecida en V4/V5 (AMP, class weights, label smoothing y gradient clipping) para que la comparación sea justa. Los resultados se guardan en un CSV por variante y se almacenan checkpoints cuando el F1 macro supera el umbral definido.


In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.transformer.model import FlexibleTransformer
from src.transformer.prepare_data import prepare_transformer_data, collate_transformer
from src.transformer.train import run_training_transformer
from src.utils.save_methods import save_model
from src.utils.get_max_id import get_max_id

from src.utils.find_best import topK_from_each_topx_csv

topK_from_each_topx_csv(
    input_dir="results/Transformer/",
    output_dir="presets_configs/Transformer/preset_top_K",
    model_name="Transformer",
    top_k=5,
    dedup=True
)



DATA_DIR = "Gait_Embeddings_LowSteps"  # 30%/50%/70% dentro

RESULTS_DIR = "results/Transformer"
os.makedirs(RESULTS_DIR, exist_ok=True)

VARIANTS = [
    #(subfolder, presets_csv, results_csv, save_dir)
    ("30%", "presets_configs/Transformer/preset_top_K/transformer_results_v5_top30%_top5.csv",
     os.path.join(RESULTS_DIR, "transformer_results_v5_low30%.csv"),
     "saved_models/Transformer/v5_low30%"),
    ("50%", "presets_configs/Transformer/presets_top50%.csv",
     os.path.join(RESULTS_DIR, "transformer_results_v5_low50%.csv"),
     "saved_models/Transformer/v5_low50%"),
    ("70%", "presets_configs/Transformer/presets_top70%.csv",
     os.path.join(RESULTS_DIR, "transformer_results_v5_low70%.csv"),
     "saved_models/Transformer/v5_low70%"),
]

for _, _, _, sd in VARIANTS:
    os.makedirs(sd, exist_ok=True)

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS    = 5000
THRESHOLD = 0.6
TEST_SIZE = 0.1
SEED      = 42

# ===== Recipe (igual que V4/V5 Top) =====
USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = None
LABEL_SMOOTH  = 0.02
CLIP_NORM     = 1.0

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

def _safe_list(x, default=None):
    if default is None:
        default = []
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return default
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if not s:
        return default
    try:
        out = ast.literal_eval(s)
        return out if isinstance(out, list) else default
    except Exception:
        return default

def _to_str(x, default=""):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return default
    return str(x).strip()

def _to_float(x, default=0.0):
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return float(default)
        return float(x)
    except Exception:
        return float(default)

def _to_int(x, default=0):
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return int(default)
        return int(x)
    except Exception:
        return int(default)

def get_embedding_files(folder_path: str):
    if not os.path.isdir(folder_path):
        return []
    return sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])

# =========================================================
# ENTRENAMIENTO
# =========================================================
for low_tag, presets_csv, results_csv, save_dir in VARIANTS:
    sub_dir = os.path.join(DATA_DIR, low_tag)

    if not os.path.isdir(sub_dir):
        print(f" Subcarpeta no encontrada: {sub_dir} (skip)")
        continue

    if not os.path.exists(presets_csv):
        print(f" Presets no encontrado: {presets_csv} (skip)")
        continue

    presets = pd.read_csv(presets_csv, low_memory=False)

    emb_files = get_embedding_files(sub_dir)
    if not emb_files:
        print(f" No hay CSVs en: {sub_dir} (skip)")
        continue

    id_counter = get_max_id([results_csv])

    print("\n" + "=" * 70)
    print(f"📂 LOW-K: {low_tag} | embeddings={len(emb_files)} | presets={len(presets)}")
    print(f"📝 Resultados -> {results_csv}")
    print(f"💾 Modelos -> {save_dir}")
    print("=" * 70)

    for preset_idx, row in presets.iterrows():
        print(f"\n=== [LOW {low_tag}] PRESET {preset_idx+1}/{len(presets)} ===")
        all_results = []

        # -------------------------
        # Leer hiperparámetros Transformer
        # -------------------------
        model_dims = _safe_list(row.get("model_dims"))
        num_layers = _to_int(row.get("num_layers"), default=len(model_dims) if model_dims else 1)

        if model_dims and len(model_dims) != num_layers:
            raise ValueError(f"Inconsistencia model_dims/num_layers en preset {preset_idx+1}.")

        optimizer_name = _to_str(row.get("optimizer", "AdamW"))
        momentum_sgd_train = _to_float(row.get("momentum_sgd", 0.0)) if optimizer_name.lower() == "sgd" else 0.0
        momentum_sgd_store = momentum_sgd_train if optimizer_name.lower() == "sgd" else None

        num_heads       = _to_int(row.get("num_heads", 4))
        dim_feedforward = _to_int(row.get("dim_feedforward", 256))
        dropout         = _to_float(row.get("dropout", 0.1))
        activation      = _to_str(row.get("activation", "gelu"))
        norm_type       = _to_str(row.get("norm_type", "prenorm"))
        pos_encoding    = _to_str(row.get("pos_encoding", "learned"))
        pooling         = _to_str(row.get("pooling", "cls"))
        use_cls_token   = (pooling.lower() == "cls")

        lr           = _to_float(row.get("lr", row.get("learning_rate", 1e-4)))
        weight_decay = _to_float(row.get("weight_decay", 0.0))
        batch_size   = _to_int(row.get("batch_size", 64))
        norm         = _to_str(row.get("norm", "L2"))

        preset_source = _to_str(row.get("source_params", f"Preset_{preset_idx+1}"))

        config_base = {
            "model_dims": model_dims,
            "num_heads": num_heads,
            "num_layers": num_layers,
            "dim_feedforward": dim_feedforward,
            "dropout": dropout,
            "activation": activation,
            "norm_type": norm_type,
            "pos_encoding": pos_encoding,
            "pooling": pooling,
            "use_cls_token": use_cls_token,
            "lr": lr,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "optimizer": optimizer_name,
            "momentum_sgd": momentum_sgd_store,
            "norm": norm,
        }

        # ======================================================
        # ENTRENAR CONTRA CADA EMBEDDING DE ESA SUBCARPETA
        # ======================================================
        for fname in emb_files:
            print(f"  -> [LOW {low_tag}] embedding: {fname}")
            df = pd.read_csv(os.path.join(sub_dir, fname), low_memory=False)

            train_list, test_list, _, _ = prepare_transformer_data(
                df=df,
                norm=norm,
                test_size=TEST_SIZE,
                seed=SEED
            )

            train_loader = DataLoader(
                train_list,
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_transformer
            )
            test_loader = DataLoader(
                test_list,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_transformer
            )

            input_dim = int(df.filter(like="feat_").shape[1])
            max_seq_len = max(seq.shape[0] for (seq, _) in train_list) if len(train_list) > 0 else 128

            model = FlexibleTransformer(
                input_dim=input_dim,
                model_dim=model_dims,
                num_heads=num_heads,
                num_layers=num_layers,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                activation=activation,
                norm_type=norm_type,
                pos_encoding=pos_encoding,
                use_cls_token=use_cls_token,
                pooling=pooling,
                num_classes=3,
                max_seq_len=max_seq_len
            ).to(DEVICE)

            id_counter += 1

            history, best_state = run_training_transformer(
                model=model,
                train_loader=train_loader,
                test_loader=test_loader,
                epochs=EPOCHS,
                lr=lr,
                weight_decay=weight_decay,
                optimizer_name=optimizer_name,
                momentum_sgd=momentum_sgd_train,
                use_class_weights=USE_CLASS_WEIGHTS,
                class_weights=(CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None),
                label_smoothing=LABEL_SMOOTH,
                clip_grad_norm=CLIP_NORM
            )

            config = dict(config_base)
            config["input_dim"] = input_dim
            config["max_seq_len"] = max_seq_len

            cw_to_store = history.get("class_weights", None)

            all_results.append({
                "id": id_counter,
                "timestamp": timestamp,
                "target_embedding": fname,
                "top_variant": low_tag,  # mantengo el nombre para consistencia con tus CSVs
                "source_params": preset_source,
                "model": "Transformer",
                "seed": SEED,

                **config,

                "class_weights": cw_to_store,
                "label_smoothing": LABEL_SMOOTH,
                "clip_grad_norm": CLIP_NORM,

                "train_loss": round(history["train_loss"][-1], 5),
                "test_loss": round(history["test_loss"][-1], 5),
                "accuracy": round(history["test_acc"][-1], 5),
                "f1_macro": round(history["test_f1"][-1], 5),
                "f1_per_class": [round(v, 5) for v in history["f1_per_class"][-1]],
                "epochs_trained": int(history["epochs_trained"]),

                "train_loss_best_state": round(history["train_loss_best_state"], 5),
                "test_loss_best_state": round(history["test_loss_best_state"], 5),
                "best_accuracy": round(history["best_acc"], 5),
                "best_f1": round(history["best_f1"], 5),
                "best_f1_per_class": [round(v, 5) for v in history["best_f1_per_class"]],
                "best_epoch": int(history["best_epoch"]),

                "training": "STOPPED" if history["early_stopped"] else "COMPLETED",
                "duration_sec": round(history.get("duration_sec", 0.0), 3),
            })

            if history["best_f1"] >= THRESHOLD:
                model.load_state_dict(best_state)
                ckpt_name = f"transformer{id_counter}_v5_low{low_tag}_{fname.replace('.csv','')}.pth"
                ckpt_path = os.path.join(save_dir, ckpt_name)
                save_model(model, ckpt_path, config, id_counter)

                # --- Atención (DESACTIVADO por estabilidad) ---
                # try:
                #     model.eval()
                #     x_sample, mask_sample, _ = next(iter(test_loader))
                #     x_sample, mask_sample = x_sample.to(DEVICE), mask_sample.to(DEVICE)
                #
                #     attn_dir = os.path.join(save_dir, "attn_maps")
                #     os.makedirs(attn_dir, exist_ok=True)
                #
                #     save_attention_maps(
                #         model=model,
                #         x_sample=x_sample,
                #         mask_sample=mask_sample,
                #         save_dir=attn_dir,
                #         model_name=f"transformer{id_counter}_v5_low{low_tag}_{fname.replace('.csv','')}",
                #     )
                # except Exception as e:
                #     print(f" No se pudieron guardar mapas de atención: {e}")

        # Guardar resultados del preset (append)
        df_results = pd.DataFrame(all_results)
        if os.path.exists(results_csv):
            with open(results_csv, "a", newline="") as f:
                f.write("\n")
            df_results.to_csv(results_csv, mode="a", header=False, index=False)
        else:
            df_results.to_csv(results_csv, index=False)

print("\n=== DONE: Transformer V5 LOW-K (30/50/70) ===")


####### Limpieza de modelos guardados

In [ ]:
import os
import re
import pandas as pd

def _extract_id_from_filename(fname: str):
    """
    Extrae el primer número que aparezca en el nombre (ej: mlp123_v4_xxx.pth -> 123).
    Si no encuentra, devuelve None.
    """
    m = re.search(r'(\d+)', fname)
    return int(m.group(1)) if m else None

def clean_saved_models(
    results_csv: str,
    save_dir: str,
    threshold: float = 0.61,
    metric_col: str = "best_f1",
    id_col: str = "id",
    exts=(".pth", ".pt"),
    dry_run: bool = True,
):
    if not os.path.isfile(results_csv):
        raise FileNotFoundError(f"No existe results_csv: {results_csv}")
    if not os.path.isdir(save_dir):
        raise FileNotFoundError(f"No existe save_dir: {save_dir}")

    df = pd.read_csv(results_csv)

    if id_col not in df.columns:
        raise ValueError(f"El CSV no tiene la columna '{id_col}'. Columnas: {list(df.columns)}")
    if metric_col not in df.columns:
        raise ValueError(f"El CSV no tiene la columna '{metric_col}'. Columnas: {list(df.columns)}")

    # Coerción robusta
    df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
    df[metric_col] = pd.to_numeric(df[metric_col], errors="coerce")

    keep_ids = set(
        df.loc[df[metric_col] >= threshold, id_col]
          .dropna()
          .astype(int)
          .unique()
          .tolist()
    )

    files = [f for f in os.listdir(save_dir) if f.lower().endswith(exts)]
    to_delete = []
    skipped = []

    for f in files:
        fid = _extract_id_from_filename(f)
        if fid is None:
            skipped.append((f, "no_id_found"))
            continue

        # Si el id no está en los que cumplen el threshold, se borra
        if fid not in keep_ids:
            to_delete.append((f, fid))

    # Report
    print(f"\n📄 CSV: {results_csv}")
    print(f"📁 DIR: {save_dir}")
    print(f" Umbral: {threshold} sobre '{metric_col}'")
    print(f" IDs a conservar: {len(keep_ids)}")
    print(f"🧹 Candidatos a borrar: {len(to_delete)}")
    print(f" Ficheros saltados: {len(skipped)}")

    # Acción
    deleted = 0
    for f, fid in to_delete:
        path = os.path.join(save_dir, f)
        if dry_run:
            print(f"[DRY] Borraría: {path} (id={fid})")
        else:
            os.remove(path)
            deleted += 1
            print(f"🗑️ Borrado: {path} (id={fid})")

    if not dry_run:
        print(f"\n Borrados totales: {deleted}")
    else:
        print("\nℹ️ Dry-run activo: no se ha borrado nada.")

    return {
        "keep_ids": keep_ids,
        "to_delete": to_delete,
        "skipped": skipped,
    }


# =========================
# EJEMPLO DE USO (edita esto)
# =========================

JOBS = [
    ("results/LSTM/lstm_results_v1_Optuna.csv", "saved_models/LSTM/v1"),
    ("results/LSTM/lstm_results_v2.csv", "saved_models/LSTM/v2"),
    ("results/LSTM/lstm_results_v3.csv", "saved_models/LSTM/v3"),
    ("results/TCN/tcn_results_v1_Optuna.csv", "saved_models/TCN/v1"),
    ("results/TCN/tcn_results_v2.csv", "saved_models/TCN/v2"),
    ("results/TCN/tcn_results_v3.csv", "saved_models/TCN/v3"),
    ("results/Transformer/transformer_results_v1_Optuna.csv", "saved_models/Transformer/v1"),
    ("results/Transformer/transformer_results_v2.csv", "saved_models/Transformer/v2"),
    ("results/Transformer/transformer_results_v3.csv", "saved_models/Transformer/v3"),
]

THRESH = 0.6
DRY_RUN = True  # <- ponlo a False cuando verifiques la salida

for results_csv, save_dir in JOBS:
    clean_saved_models(
        results_csv=results_csv,
        save_dir=save_dir,
        threshold=THRESH,
        metric_col="best_f1",
        id_col="id",
        dry_run=DRY_RUN
    )


In [ ]:
import re
from pathlib import Path
import pandas as pd

# Sirve para:
#  - transformer2_v2_xxx.pth
#  - transformer2_v2_xxx_attn.pt
#  - mlp123_v3_xxx.pth
MODEL_PREFIX_RE = re.compile(r'^(mlp|lstm|tcn|transformer)(\d+)_', re.IGNORECASE)

def extract_id_from_filename(fname: str):
    m = MODEL_PREFIX_RE.search(fname)
    return int(m.group(2)) if m else None

def load_best_f1_by_id(results_csvs, id_col="id", metric_col="best_f1"):
    if isinstance(results_csvs, (str, Path)):
        results_csvs = [results_csvs]

    frames = []
    for p in results_csvs:
        p = Path(p)
        if not p.is_file():
            raise FileNotFoundError(f"No existe CSV: {p}")
        df = pd.read_csv(p)
        if id_col not in df.columns:
            raise ValueError(f"CSV {p} no tiene columna '{id_col}'")
        if metric_col not in df.columns:
            raise ValueError(f"CSV {p} no tiene columna '{metric_col}'")

        df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
        df[metric_col] = pd.to_numeric(df[metric_col], errors="coerce")
        df = df.dropna(subset=[id_col, metric_col])
        df[id_col] = df[id_col].astype(int)

        frames.append(df[[id_col, metric_col]])

    all_df = pd.concat(frames, ignore_index=True)

    # Si un id aparece varias veces (distintos embeddings), nos quedamos con el max best_f1
    return all_df.groupby(id_col)[metric_col].max().to_dict()

def clean_saved_artifacts_by_threshold(
    results_csvs,
    base_dir,
    threshold=0.575,
    id_col="id",
    metric_col="best_f1",
    dry_run=True,
    extensions=(".pth", ".pt"),         # .pt para attn maps
    delete_if_id_missing=False,         # por seguridad: si no está en CSV, no borramos
):
    base_dir = Path(base_dir)
    if not base_dir.is_dir():
        raise FileNotFoundError(f"No existe carpeta: {base_dir}")

    best_f1_by_id = load_best_f1_by_id(results_csvs, id_col=id_col, metric_col=metric_col)

    files = []
    for ext in extensions:
        files.extend(base_dir.rglob(f"*{ext}"))

    to_delete = []
    skipped = []

    for f in files:
        fid = extract_id_from_filename(f.name)
        if fid is None:
            skipped.append((str(f), "no_match_prefix_id"))
            continue

        f1 = best_f1_by_id.get(fid, None)
        if f1 is None:
            if delete_if_id_missing:
                to_delete.append((f, fid, None))
            else:
                skipped.append((str(f), f"id_not_in_csv({fid})"))
            continue

        if float(f1) < threshold:
            to_delete.append((f, fid, float(f1)))

    print(f"\n📁 Base: {base_dir}")
    print(f"📄 CSVs: {results_csvs}")
    print(f" Umbral: {threshold} (col='{metric_col}')")
    print(f"🔎 Archivos encontrados ({extensions}): {len(files)}")
    print(f"🗑️ Candidatos a borrar: {len(to_delete)}")
    print(f" Saltados: {len(skipped)}")

    for f, fid, f1 in to_delete:
        if dry_run:
            print(f"[DRY] Borraría: {f} | id={fid} | best_f1={f1}")
        else:
            f.unlink()
            print(f"🗑️ Borrado: {f} | id={fid} | best_f1={f1}")

    if dry_run:
        print("\nℹ️ Dry-run activo: no se ha borrado nada.")
    else:
        print("\n Borrado completado.")

    return {"to_delete": to_delete, "skipped": skipped}


# =========================
# EJEMPLO (Transformer)
# =========================
# OJO: usa los CSV que correspondan con esa carpeta/version.
# Si en la carpeta hay modelos v1 y v2, pásale ambos CSVs.
# Por ejemplo:
results_csvs = [
  "results/Transformer/transformer_results_v1 DEFINITIVO.csv",
  "results/Transformer/transformer_results_v2 DEFINITIVO.csv",
]

clean_saved_artifacts_by_threshold(
    results_csvs=results_csvs,
    base_dir="saved_models/Transformer/v1 DEFINITIVO",  # o ".../v1 DEFINITIVO"
    threshold=0.6,
    dry_run=False,
)


## Chequeo de configuración de Transformer

In [ ]:
import torch
from src.utils.check_model import check_model_config

model_path = "saved_models/Transformer/v4_mean/transformer3_v4_mean_gaitpart_16steps_mean.pth"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
check_model_config(model_path, device)